# Phase 5 · Legal statute identification

Predict a set of statute IDs for each case. This notebook runs Model 3 full fine-tuning, Model 1 LoRMA, and Model 4 distillation. The Model 2 adapter comparison is in a separate notebook. Each run evaluates its best checkpoint and selects a decision threshold on development data.

## Required files and environment

- Attach `phase5_kaggle_assets.zip` or its extracted Phase 1 model folders and evaluator. Model 2 uses `encoder_decoder_500m_batch4_trial`.
- Attach the official `Phase_5_LSI/` data, or enable Internet and set `HF_TOKEN` to download it.
- Enable a GPU for training. Restore a previous run with the same settings and GPU count before resuming it.

## 1. Packages and imports


In [3]:
import subprocess
import sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'huggingface-hub>=0.34,<1', 'scikit-learn>=1.4,<2',
    'ijson>=3.3,<4', 'pandas>=2.2,<3', 'matplotlib>=3.8,<4', 'tqdm>=4.66,<5'], check=True)


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.26.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
transformers 5.16.1 requires huggingface-hub<2.0,>=1.5.0, but you have huggingface-hub 0.36.2 which is incompatible.


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'huggingface-hub>=0.34,<1', 'scikit-learn>=1.4,<2', 'ijson>=3.3,<4', 'pandas>=2.2,<3', 'matplotlib>=3.8,<4', 'tqdm>=4.66,<5'], returncode=0)

In [4]:
import copy
import gc
import hashlib
import heapq
import itertools
import json
import math
import os
import random
import re
import shutil
import subprocess
import sys
import time
import zipfile
from collections import Counter, defaultdict
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import HTML, display
import base64
import html
from sklearn.metrics import f1_score, precision_score, recall_score
from torch.utils.checkpoint import checkpoint
from torch.utils.data import DataLoader, Dataset

PAD, UNK, BOS, EOS = range(4)

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")
    temporary.replace(path)

def digest_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def normalize_id(value):
    value = str(value)
    return value[:-4] if value.endswith(".txt") else value


import contextlib
import csv
from datetime import datetime
from urllib.parse import quote
import importlib
import ijson
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from IPython.display import FileLink
from sklearn.model_selection import KFold

VERSION = 'phase5_lsi_v1'
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
for index in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(index)
    print(f'GPU {index}: {props.name}, {props.total_memory / 2**30:.2f} GiB')


PyTorch: 2.11.0+cu128 | CUDA: True
GPU 0: Tesla T4, 14.56 GiB
GPU 1: Tesla T4, 14.56 GiB


## 2. Settings

The classifier combines up to four overlapping 1,024-token windows using label-specific attention.

In [5]:
INPUT_ROOT = Path('/kaggle/input')
OUTPUT_ROOT = Path('/kaggle/working/phase5_lsi')
ASSET_DIR = OUTPUT_ROOT / 'assets'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
ASSET_DIR.mkdir(parents=True, exist_ok=True)
DATA_ROOT = None
PREPARED_CACHE = None
PREPARED_CACHES = []
SEARCH_ROOTS = []
REPO = 'Exploration-Lab/CS-781-Capstone'
DATA_REVISION = None
LABEL_FIELD = 'labels'
MODEL_NAMES = ['model1', 'model2', 'model3', 'model4']
MODEL_DIRS = dict.fromkeys(MODEL_NAMES)
RUN_NAMES = ['model3_full', 'model1_lorma', 'model2_adapters', 'model4_distilled']
TEACHER_RUN_NAME = None
EXPERIMENTS = {}
TOKEN_FOLDERS = {}
RUN_REPRESENTATION_ANALYSIS = False
RUN_MODEL2_SMALL_SAMPLE_CHECK = True

DEFAULT = dict(
    seed=42, epochs=20, min_epochs=0, patience=3, freeze_backbone=False,
    chunk_tokens=1024, overlap=128, max_chunks=4, representation='hierarchical',
    attention_mode='label', head_dim=128, dropout=0.2,
    effective_batch=8, micro_batch_per_gpu=1, activation_checkpointing=True,
    backbone_lr=2e-5, adapter_lr=1e-4, head_lr=2e-4,
    warmup_fraction=0.06, min_lr_ratio=0.1, weight_decay=0.01, clip=1.0,
    rank=8, alpha=16, adapter_dim=64, adapter_dropout=0.1,
    model2_pool='encoder_query', student_keep=None,
    loss='bce', positive_weighting=False, positive_weight_power=0.5, positive_weight_cap=10.0,
    asl_gamma_neg=4.0, asl_gamma_pos=0.0, asl_clip=0.05,
    kd_alpha=0.5, kd_temperature=2.0,
    threshold_grid=[0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8],
    log_every=20, save_every=100, max_hours=10.5,
    amp_initial_scale=1024.0, max_consecutive_amp_skips=12, capacity_headroom_gib=1.5,
)
print(json.dumps(DEFAULT, indent=2))


{
  "seed": 42,
  "epochs": 20,
  "min_epochs": 0,
  "patience": 3,
  "freeze_backbone": false,
  "chunk_tokens": 1024,
  "overlap": 128,
  "max_chunks": 4,
  "representation": "hierarchical",
  "attention_mode": "label",
  "head_dim": 128,
  "dropout": 0.2,
  "effective_batch": 8,
  "micro_batch_per_gpu": 1,
  "activation_checkpointing": true,
  "backbone_lr": 2e-05,
  "adapter_lr": 0.0001,
  "head_lr": 0.0002,
  "warmup_fraction": 0.06,
  "min_lr_ratio": 0.1,
  "weight_decay": 0.01,
  "clip": 1.0,
  "rank": 8,
  "alpha": 16,
  "adapter_dim": 64,
  "adapter_dropout": 0.1,
  "model2_pool": "encoder_query",
  "student_keep": null,
  "loss": "bce",
  "positive_weighting": false,
  "positive_weight_power": 0.5,
  "positive_weight_cap": 10.0,
  "asl_gamma_neg": 4.0,
  "asl_gamma_pos": 0.0,
  "asl_clip": 0.05,
  "kd_alpha": 0.5,
  "kd_temperature": 2.0,
  "threshold_grid": [
    0.05,
    0.1,
    0.2,
    0.3,
    0.4,
    0.5,
    0.6,
    0.7,
    0.8
  ],
  "log_every": 20,
  "save_ever

In [6]:
def classify_model_config(config):
    if 'encoder_layers' in config and 'decoder_layers' in config: return 'model2'
    if 'kernel_size' in config and 'dilation_cycle' in config: return 'model4'
    if 'window' in config and 'global_every' in config: return 'model3'
    if 'layers' in config and 'heads' in config and 'context' in config: return 'model1'
    return None


def unpack_uploads():
    restored = []
    for archive_path in sorted(INPUT_ROOT.rglob('*.zip')):
        if not archive_path.name.startswith(('phase5_kaggle_assets', 'phase5_shared_', *[name + '_' for name in RUN_NAMES])):
            continue
        destination = OUTPUT_ROOT / 'attached_backups' / archive_path.stem
        identity = dict(path=str(archive_path), bytes=archive_path.stat().st_size,
                        modified=archive_path.stat().st_mtime_ns)
        marker = destination / 'imported.json'
        if marker.is_file() and read_json(marker) == identity:
            restored.append(destination)
            continue
        with zipfile.ZipFile(archive_path) as archive:
            selected = []
            for item in archive.infolist():
                parts = Path(item.filename).parts
                if item.is_dir() or not parts or '..' in parts or Path(item.filename).is_absolute(): continue
                if parts[0] in ('phase5_model_assets', 'phase5_lsi', 'Phase_5_LSI'):
                    selected.append(item)
            needed = sum(item.file_size for item in selected)
            if shutil.disk_usage(OUTPUT_ROOT).free < needed + 256 * 2**20:
                raise OSError('Not enough disk space to unpack the attached Phase 5 ZIP.')
            for item in tqdm(selected, desc=f'Import {archive_path.stem}'):
                target = destination / item.filename
                if not target.resolve().is_relative_to(destination.resolve()):
                    raise ValueError('Unsafe ZIP member path.')
                target.parent.mkdir(parents=True, exist_ok=True)
                temporary = target.with_suffix(target.suffix + '.importing')
                with archive.open(item) as incoming, temporary.open('wb') as outgoing:
                    shutil.copyfileobj(incoming, outgoing, length=8 * 2**20)
                temporary.replace(target)
            write_json(marker, identity)
        restored.append(destination)
    return [INPUT_ROOT] + restored


def resolve_inputs():
    global SEARCH_ROOTS, PREPARED_CACHES
    SEARCH_ROOTS = unpack_uploads()
    candidates = defaultdict(list)
    for root in SEARCH_ROOTS:
        for path in root.rglob('run_config.json'):
            if all((path.parent / name).is_file() for name in ('best.pt', 'tokenizer.json')):
                kind = classify_model_config(read_json(path).get('model', {}))
                if kind: candidates[kind].append(path.parent)
    for model_name in MODEL_NAMES:
        if MODEL_DIRS[model_name] is None:
            options = sorted(set(candidates[model_name]))
            if len(options) != 1:
                raise ValueError(f'Set MODEL_DIRS[{model_name!r}]. Found: {options}')
            MODEL_DIRS[model_name] = options[0]
        MODEL_DIRS[model_name] = Path(MODEL_DIRS[model_name])
        for filename in ('best.pt', 'tokenizer.json', 'run_config.json'):
            if not (MODEL_DIRS[model_name] / filename).is_file():
                raise FileNotFoundError(MODEL_DIRS[model_name] / filename)
    def valid(folder):
        return all((folder / name).is_file() for name in ('Train/train.json', 'Train/dev.json', 'Test/test.json'))
    destination = ASSET_DIR / 'Phase_5_LSI'
    if DATA_ROOT is not None:
        source = Path(DATA_ROOT)
    elif valid(destination):
        source = destination
    else:
        options = sorted(set(path.parent.parent for root in SEARCH_ROOTS for path in root.rglob('train.json')
                             if path.parent.name == 'Train' and 'Phase_5_LSI' in path.parts and valid(path.parent.parent)))
        if len(options) > 1:
            identities = {tuple(digest_file(folder / name) for name in (
                'Train/train.json', 'Train/dev.json', 'Test/test.json')) for folder in options}
            if len(identities) != 1: raise ValueError('Multiple different Phase 5 datasets attached. Set DATA_ROOT.')
        if options:
            source = options[0]
        else:
            from huggingface_hub import HfApi, snapshot_download
            token = os.environ.get('HF_TOKEN')
            if not token:
                try:
                    from kaggle_secrets import UserSecretsClient
                    token = UserSecretsClient().get_secret('HF_TOKEN')
                except Exception:
                    token = None
            revision = DATA_REVISION
            try:
                if revision is None: revision = HfApi(token=token).dataset_info(REPO).sha
                snapshot = Path(snapshot_download(repo_id=REPO, repo_type='dataset', revision=revision,
                    allow_patterns=['Phase_5_LSI/**'], token=token))
            except Exception as error:
                raise RuntimeError('Phase 5 download failed. Enable Internet and grant this notebook access to your HF_TOKEN Kaggle secret for the official gated dataset.') from error
            finally:
                token = None
            source = snapshot / 'Phase_5_LSI'
            write_json(ASSET_DIR / 'download_provenance.json', dict(repo=REPO, revision=revision))
    if not valid(source): raise FileNotFoundError(f'Incomplete Phase_5_LSI folder: {source}')
    for name in ('Train/train.json', 'Train/dev.json', 'Test/test.json'):
        target = destination / name
        target.parent.mkdir(parents=True, exist_ok=True)
        if source.resolve() != destination.resolve():
            if not target.is_file() or digest_file(target) != digest_file(source / name):
                shutil.copy2(source / name, target)
    roots = [Path(PREPARED_CACHE)] if PREPARED_CACHE else []
    for root in SEARCH_ROOTS:
        roots.extend(path for path in root.rglob('token_cache') if path.is_dir())
    PREPARED_CACHES = sorted(set(roots))
    print('Data:', destination, flush=True)
    for name, folder in MODEL_DIRS.items(): print(name, folder, flush=True)
    return destination


def restore_saved_runs():
    for name in RUN_NAMES:
        options = []
        for root in SEARCH_ROOTS:
            for path in root.rglob('config.json'):
                if not (path.parent / 'latest.pt').is_file(): continue
                try: spec = read_json(path)
                except (ValueError, OSError): continue
                if spec.get('name') == name: options.append(path.parent)
        options = sorted(set(options))
        if not options: continue
        if len(options) > 1 and len({digest_file(folder / 'latest.pt') for folder in options}) != 1:
            raise ValueError(f'Multiple different saved runs named {name}. Attach a single version.')
        source = options[0]
        destination = run_folder(name)
        missing = [path for path in source.rglob('*') if path.is_file() and path.suffix != '.tmp'
                   and not (destination / path.relative_to(source)).exists()]
        if shutil.disk_usage(OUTPUT_ROOT).free < sum(path.stat().st_size for path in missing) + 256 * 2**20:
            raise OSError(f'Not enough working space to restore {name}.')
        for path in tqdm(missing, desc=f'Restore {name}'):
            target = destination / path.relative_to(source)
            target.parent.mkdir(parents=True, exist_ok=True)
            temporary = target.with_suffix(target.suffix + '.restoring')
            shutil.copy2(path, temporary)
            temporary.replace(target)
        print('Resume folder:', destination, flush=True)


In [7]:
DATA_ROOT = resolve_inputs()


Data: /kaggle/working/phase5_lsi/assets/Phase_5_LSI
model1 /kaggle/input/datasets/rishabhy26/phase5/phase5_model_assets/model1_decoder
model2 /kaggle/input/datasets/rishabhy26/phase5/phase5_model_assets/model2_encoder_decoder
model3 /kaggle/input/datasets/rishabhy26/phase5/phase5_model_assets/model3_local_global
model4 /kaggle/input/datasets/rishabhy26/phase5/phase5_model_assets/model4_gated_cnn


## 3. Audit the official data

Setup reads the label IDs from train and development data. It reports duplicates, conflicting annotations, label frequencies, and overlap between splits. Identical repeated IDs collapse to one case; conflicting duplicates stop for inspection. Training copies of held-out text are excluded. Test answer fields, if present, are counted without using their contents.

The evaluator derives its label dimension from the reference file. IDs are preserved exactly in submissions. Internal columns may be dense even when the original IDs have gaps; reported Macro-F1 includes those gaps to match the evaluator. A train label above the maximum development label is reported explicitly because the supplied evaluator cannot score such a prediction on that development reference.


In [8]:
def iter_records(path):
    with open(path, 'rb') as handle:
        first = handle.read(1024).lstrip(b'\xef\xbb\xbf \r\n\t')[:1]
    with open(path, 'rb') as handle:
        if first == b'[':
            yield from ijson.items(handle, 'item')
        elif first == b'{':
            for key, value in ijson.kvitems(handle, ''):
                if not isinstance(value, dict):
                    raise ValueError(f'Expected records in {path}, found {key!r}')
                record = dict(value)
                record.setdefault('id', key)
                yield record
        else:
            raise ValueError(f'Expected JSON records: {path}')


def text_value(record):
    text = record.get('text')
    if isinstance(text, list) and all(isinstance(sentence, str) for sentence in text):
        text = '\n'.join(text)
    if not isinstance(text, str) or not text.strip():
        raise ValueError(f'Empty/unsupported text for {record.get("id")}')
    return text


def labelled_record(record, split):
    identifier = record.get('id')
    if not isinstance(identifier, str) or not identifier.strip():
        raise ValueError('Case IDs must be nonempty strings; preserve their original values.')
    text = text_value(record)
    labels = None
    if split != 'test':
        raw = record.get(LABEL_FIELD)
        if not isinstance(raw, list) or any(type(label) is not int or label < 0 for label in raw):
            raise ValueError(f'Case {identifier}: expected a list of nonnegative integer labels in {LABEL_FIELD!r}')
        if len(set(raw)) != len(raw):
            raise ValueError(f'Duplicate statute label for {identifier}')
        labels = sorted(raw)
    return identifier, text, labels


def source_file(split):
    return DATA_ROOT / ('Test/test.json' if split == 'test' else f'Train/{split}.json')


def object_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False).encode()).hexdigest()


def audit_corpus():
    index, profiles, exclusions = {}, {}, []
    blocked = set()
    dev_text = set()
    dev_ids = set()
    all_labels = set()
    for split in ('dev', 'test', 'train'):
        rows, identities, duplicate_text = [], {}, {}
        words, cardinality = [], []
        duplicates = exposed = overlap_text = overlap_ids = 0
        label_counts = Counter()
        for record in tqdm(iter_records(source_file(split)), desc=f'Audit {split}'):
            identifier, text, labels = labelled_record(record, split)
            normalized = hashlib.sha256(' '.join(text.split()).encode()).hexdigest()
            identity = (hashlib.sha256(text.encode()).hexdigest(), None if labels is None else tuple(labels))
            if identifier in identities:
                if identities[identifier] != identity:
                    raise ValueError(f'Conflicting repeated ID in {split}: {identifier}')
                duplicates += 1
                continue
            identities[identifier] = identity
            if split == 'test':
                exposed += any(key in record for key in ('labels', 'label', 'gold', 'answer', 'answers'))
                overlap_text += normalized in dev_text
                overlap_ids += identifier in dev_ids
            if split == 'train':
                old = duplicate_text.get(normalized)
                if old is not None and old != tuple(labels):
                    raise ValueError(f'Conflicting labels for identical training text: {identifier}')
                repeated = normalized in duplicate_text
                duplicate_text[normalized] = tuple(labels)
                if normalized in blocked or repeated:
                    exclusions.append(dict(id=identifier, reason='heldout_text' if normalized in blocked else 'duplicate_train_text'))
                    continue
            else:
                blocked.add(normalized)
                if split == 'dev':
                    dev_text.add(normalized)
                    dev_ids.add(identifier)
            rows.append(dict(id=identifier, labels=labels, text_sha256=normalized, words=len(text.split())))
            words.append(len(text.split()))
            if labels is not None:
                all_labels.update(labels)
                label_counts.update(labels)
                cardinality.append(len(labels))
        if not rows:
            raise ValueError(f'Empty split after preprocessing: {split}')
        index[split] = rows
        profiles[split] = dict(examples=len(rows), identical_duplicate_ids=duplicates,
            word_percentiles=dict(zip(['50', '90', '95', '99', '100'], np.percentile(words, [50, 90, 95, 99, 100]).tolist())),
            label_counts={str(key): value for key, value in sorted(label_counts.items())},
            labels_per_case_mean=float(np.mean(cardinality)) if cardinality else None,
            labels_per_case_max=max(cardinality, default=None))
        if split == 'test':
            profiles[split].update(answer_fields_present=exposed,
                                   dev_text_matches=overlap_text, dev_id_matches=overlap_ids)
    if not all_labels:
        raise ValueError('No supervised statute labels found.')
    labels = sorted(all_labels)
    train_seen = set(label for row in index['train'] for label in row['labels'])
    dev_seen = set(label for row in index['dev'] for label in row['labels'])
    metadata = dict(ids=labels, dimension=len(labels), max_id=max(labels),
        evaluator_dimension=max(labels) + 1,
        dev_evaluator_dimension=max(dev_seen, default=-1) + 1,
        dev_labels_absent_from_training=sorted(dev_seen - train_seen),
        integer_gaps=[label for label in range(max(labels) + 1) if label not in all_labels])
    write_json(ASSET_DIR / 'case_index.json', index)
    write_json(ASSET_DIR / 'label_metadata.json', metadata)
    write_json(ASSET_DIR / 'corpus_audit.json', dict(splits=profiles, exclusions=len(exclusions)))
    write_json(ASSET_DIR / 'excluded_train.json', exclusions)
    write_json(ASSET_DIR / 'dev_truth.json', [dict(id=row['id'], gold=row['labels']) for row in index['dev']])
    counts = np.array([sum(label in row['labels'] for row in index['train']) for label in labels])
    with (ASSET_DIR / 'label_frequencies.csv').open('w', newline='', encoding='utf-8') as handle:
        writer = csv.writer(handle)
        writer.writerow(['statute_id', 'train_support', 'dev_support'])
        for label, count in zip(labels, counts):
            writer.writerow([label, int(count), sum(label in row['labels'] for row in index['dev'])])
    print(json.dumps(dict(splits=profiles, labels=metadata), indent=2), flush=True)
    if profiles['test']['answer_fields_present'] or profiles['test']['dev_text_matches']:
        print('Data warning: exposed test fields or dev/test overlap detected. Test answer fields are ignored; consult the organizer about evaluation validity.', flush=True)
    return index, metadata


def label_matrix(records):
    result = np.zeros((len(records), len(LABEL_IDS)), dtype=np.float32)
    for i, row in enumerate(records):
        for label in row['labels'] or []:
            result[i, LABEL_TO_INDEX[label]] = 1
    return result


def prepare_tokens(model_name, split):
    base = BASES[model_name]
    key = base['tokenizer_sha256'][:16]
    wanted = dict(version=VERSION, tokenizer=base['tokenizer_sha256'],
                  index=object_hash(CASE_INDEX[split]), source=digest_file(source_file(split)))
    destination = OUTPUT_ROOT / 'token_cache' / key / split
    candidates = [Path(root) / key / split for root in PREPARED_CACHES] + [destination]
    for folder in candidates:
        if not (folder / 'manifest.json').is_file():
            continue
        manifest = read_json(folder / 'manifest.json')
        if manifest['signature'] != wanted:
            if folder == destination:
                raise ValueError(f'Writable token cache mismatch: {folder}')
            continue
        if digest_file(folder / 'tokens.bin') != manifest['sha256']:
            raise ValueError(f'Corrupt token cache: {folder}')
        if [row['id'] for row in read_json(folder / 'index.json')] != [row['id'] for row in CASE_INDEX[split]]:
            raise ValueError('Token cache case order differs.')
        return folder
    destination.mkdir(parents=True, exist_ok=True)
    partial = destination / 'progress.json'
    state = read_json(partial) if partial.is_file() else dict(signature=wanted, rows=[], tokens=0)
    if state['signature'] != wanted:
        raise ValueError('Partial token cache changed.')
    kept = {row['id'] for row in CASE_INDEX[split]}
    seen = set()
    binary = destination / 'tokens.bin'
    if state['tokens'] and (not binary.is_file() or binary.stat().st_size < state['tokens'] * 2):
        raise ValueError('Partial tokens are missing or shorter than the progress record.')
    with open(binary, 'r+b' if binary.is_file() else 'w+b') as handle:
        handle.truncate(state['tokens'] * 2)
        handle.seek(0, 2)
        position = 0
        for record in tqdm(iter_records(source_file(split)), desc=f'Tokenize {split}'):
            identifier = record['id']
            if identifier not in kept or identifier in seen:
                continue
            seen.add(identifier)
            if position < len(state['rows']):
                if state['rows'][position]['id'] != identifier:
                    raise ValueError('Partial case order changed.')
                position += 1
                continue
            ids = base['tokenizer'].encode(text_value(record)).ids
            if not ids or min(ids) < 0 or max(ids) >= min(65536, base['config']['vocab_size']):
                raise ValueError(f'Invalid tokens: {identifier}')
            np.asarray(ids, dtype='<u2').tofile(handle)
            state['rows'].append(dict(id=identifier, start=state['tokens'], length=len(ids)))
            state['tokens'] += len(ids)
            position += 1
            if position % 100 == 0:
                handle.flush()
                os.fsync(handle.fileno())
                write_json(partial, state)
        handle.flush()
        os.fsync(handle.fileno())
    if [row['id'] for row in state['rows']] != [row['id'] for row in CASE_INDEX[split]]:
        raise ValueError('Prepared token order differs from the audited corpus.')
    write_json(destination / 'index.json', state['rows'])
    write_json(destination / 'manifest.json', dict(signature=wanted, tokens=state['tokens'],
        sha256=digest_file(binary), token_percentiles=np.percentile(
            [row['length'] for row in state['rows']], [50, 90, 95, 99, 100]).tolist()))
    return destination


def prepare_all_tokens():
    shared = {}
    for model_name in MODEL_NAMES:
        for split in ('train', 'dev', 'test'):
            key = (BASES[model_name]['tokenizer_sha256'], split)
            if key not in shared:
                shared[key] = prepare_tokens(model_name, split)
            TOKEN_FOLDERS[(model_name, split)] = shared[key]
    for split in ('train', 'dev', 'test'):
        print(split, read_json(TOKEN_FOLDERS[('model1', split)] / 'manifest.json')['token_percentiles'], flush=True)


def window_starts(length, cfg):
    size = cfg['chunk_tokens']
    if size < 2 or not 0 <= cfg['overlap'] < size:
        raise ValueError('Invalid chunk length or overlap.')
    last = max(0, length - size)
    if cfg['representation'] == 'head': return [0]
    if cfg['representation'] == 'tail': return [last]
    if cfg['representation'] == 'head_tail': return sorted({0, last})
    if cfg['representation'] != 'hierarchical' or cfg['max_chunks'] < 2:
        raise ValueError('Unknown representation or invalid max_chunks.')
    starts = list(range(0, last + 1, size - cfg['overlap']))
    if starts[-1] != last: starts.append(last)
    if len(starts) > cfg['max_chunks']:
        starts = [starts[i] for i in np.linspace(0, len(starts) - 1, cfg['max_chunks']).round().astype(int)]
    return starts


def max_case_chunks(cfg):
    return dict(head=1, tail=1, head_tail=2, hierarchical=cfg['max_chunks'])[cfg['representation']]


class CaseDataset(Dataset):
    def __init__(self, model_name, split, cfg):
        folder = TOKEN_FOLDERS[(model_name, split)]
        self.tokens = np.memmap(folder / 'tokens.bin', dtype='<u2', mode='r')
        self.rows = read_json(folder / 'index.json')
        self.records = CASE_INDEX[split]
        self.targets = label_matrix(self.records)
        self.cfg = cfg
        if [row['id'] for row in self.rows] != [row['id'] for row in self.records]:
            raise ValueError('Case/cache order mismatch.')
    def __len__(self): return len(self.rows)
    def __getitem__(self, index):
        row = self.rows[index]
        starts = window_starts(row['length'], self.cfg)
        windows = [np.array(self.tokens[row['start'] + start:row['start'] + min(row['length'], start + self.cfg['chunk_tokens'])], dtype=np.int64) for start in starts]
        positions = [start / max(1, row['length'] - 1) for start in starts]
        return windows, positions, self.targets[index], index


def collate_cases(samples):
    count = len(samples)
    chunks = max(len(sample[0]) for sample in samples)
    length = max(len(window) for sample in samples for window in sample[0])
    ids = torch.full((count, chunks, length), PAD, dtype=torch.long)
    mask = torch.zeros_like(ids, dtype=torch.bool)
    positions = torch.zeros(count, chunks, dtype=torch.float32)
    for i, (windows, places, _, _) in enumerate(samples):
        for j, (window, place) in enumerate(zip(windows, places)):
            ids[i, j, :len(window)] = torch.as_tensor(window)
            mask[i, j, :len(window)] = True
            positions[i, j] = place
    return dict(ids=ids, mask=mask, positions=positions,
                labels=torch.tensor(np.stack([sample[2] for sample in samples]), dtype=torch.float32),
                indices=torch.tensor([sample[3] for sample in samples]))


def coverage_report(model_name, cfg):
    report = {}
    for split in ('train', 'dev', 'test'):
        fractions, covered, total, counts = [], 0, 0, []
        for row in read_json(TOKEN_FOLDERS[(model_name, split)] / 'index.json'):
            end = amount = 0
            starts = window_starts(row['length'], cfg)
            for start in starts:
                stop = min(row['length'], start + cfg['chunk_tokens'])
                amount += max(0, stop - max(start, end))
                end = max(end, stop)
            covered += amount
            total += row['length']
            fractions.append(amount / row['length'])
            counts.append(len(starts))
        report[split] = dict(token_coverage=covered / total, mean_case_coverage=float(np.mean(fractions)),
                             mean_chunks=float(np.mean(counts)), max_chunks=max(counts))
    print(json.dumps(report, indent=2), flush=True)
    return report


In [9]:
def labelled_record(record, split):
    identifier = record.get('id')
    if not isinstance(identifier, str) or not identifier.strip():
        raise ValueError('Case IDs must be nonempty strings; preserve their original values.')
    text = text_value(record)
    labels = None
    if split != 'test':
        raw = record.get(LABEL_FIELD)
        if not isinstance(raw, list) or any(type(label) is not int or label < 0 for label in raw):
            raise ValueError(f'Case {identifier}: expected a list of nonnegative integer labels in {LABEL_FIELD!r}')
        labels = sorted(set(raw))
    return identifier, text, labels

def audit_corpus():
    index, profiles, exclusions = {}, {}, []
    blocked = set()
    dev_text = set()
    dev_ids = set()
    all_labels = set()
    training_variants = defaultdict(set)
    repeated_label_rows = 0
    for record in tqdm(iter_records(source_file('train')), desc='Check train annotations'):
        identifier, text, labels = labelled_record(record, 'train')
        normalized = hashlib.sha256(' '.join(text.split()).encode()).hexdigest()
        training_variants[normalized].add(tuple(labels))
        all_labels.update(labels)
        repeated_label_rows += len(record[LABEL_FIELD]) != len(labels)
    conflicting = {key for key, variants in training_variants.items() if len(variants) > 1}
    conflict_removed = 0
    write_json(ASSET_DIR / 'conflicting_train_text_groups.json', [
        dict(text_sha256=key, label_variants=[list(values) for values in sorted(training_variants[key])])
        for key in sorted(conflicting)
    ])
    print('Conflicting training text groups:', len(conflicting), flush=True)
    for split in ('dev', 'test', 'train'):
        rows, identities, duplicate_text = [], {}, {}
        words, cardinality = [], []
        duplicates = exposed = overlap_text = overlap_ids = 0
        label_counts = Counter()
        for record in tqdm(iter_records(source_file(split)), desc=f'Audit {split}'):
            identifier, text, labels = labelled_record(record, split)
            normalized = hashlib.sha256(' '.join(text.split()).encode()).hexdigest()
            if split == 'train' and normalized in conflicting:
                exclusions.append(dict(id=identifier, reason='conflicting_train_annotations',
                                       text_sha256=normalized, labels=labels))
                conflict_removed += 1
                continue
            identity = (hashlib.sha256(text.encode()).hexdigest(), None if labels is None else tuple(labels))
            if identifier in identities:
                if identities[identifier] != identity:
                    raise ValueError(f'Conflicting repeated ID in {split}: {identifier}')
                duplicates += 1
                continue
            identities[identifier] = identity
            if split == 'test':
                exposed += any(key in record for key in ('labels', 'label', 'gold', 'answer', 'answers'))
                overlap_text += normalized in dev_text
                overlap_ids += identifier in dev_ids
            if split == 'train':
                old = duplicate_text.get(normalized)
                if old is not None and old != tuple(labels):
                    raise ValueError(f'Conflicting labels for identical training text: {identifier}')
                repeated = normalized in duplicate_text
                duplicate_text[normalized] = tuple(labels)
                if normalized in blocked or repeated:
                    exclusions.append(dict(id=identifier, reason='heldout_text' if normalized in blocked else 'duplicate_train_text'))
                    continue
            else:
                blocked.add(normalized)
                if split == 'dev':
                    dev_text.add(normalized)
                    dev_ids.add(identifier)
            rows.append(dict(id=identifier, labels=labels, text_sha256=normalized, words=len(text.split())))
            words.append(len(text.split()))
            if labels is not None:
                all_labels.update(labels)
                label_counts.update(labels)
                cardinality.append(len(labels))
        if not rows:
            raise ValueError(f'Empty split after preprocessing: {split}')
        index[split] = rows
        profiles[split] = dict(examples=len(rows), identical_duplicate_ids=duplicates,
            word_percentiles=dict(zip(['50', '90', '95', '99', '100'], np.percentile(words, [50, 90, 95, 99, 100]).tolist())),
            label_counts={str(key): value for key, value in sorted(label_counts.items())},
            labels_per_case_mean=float(np.mean(cardinality)) if cardinality else None,
            labels_per_case_max=max(cardinality, default=None))
        if split == 'test':
            profiles[split].update(answer_fields_present=exposed,
                                   dev_text_matches=overlap_text, dev_id_matches=overlap_ids)
    if not all_labels:
        raise ValueError('No supervised statute labels found.')
    labels = sorted(all_labels)
    train_seen = set(label for row in index['train'] for label in row['labels'])
    dev_seen = set(label for row in index['dev'] for label in row['labels'])
    metadata = dict(ids=labels, dimension=len(labels), max_id=max(labels),
        evaluator_dimension=max(labels) + 1,
        dev_evaluator_dimension=max(dev_seen, default=-1) + 1,
        dev_labels_absent_from_training=sorted(dev_seen - train_seen),
        integer_gaps=[label for label in range(max(labels) + 1) if label not in all_labels])
    write_json(ASSET_DIR / 'case_index.json', index)
    write_json(ASSET_DIR / 'label_metadata.json', metadata)
    write_json(ASSET_DIR / 'corpus_audit.json', dict(splits=profiles, exclusions=len(exclusions),
        conflicting_train_text_groups=len(conflicting), conflicting_train_records_removed=conflict_removed,
        train_records_with_repeated_statute_ids=repeated_label_rows))
    write_json(ASSET_DIR / 'excluded_train.json', exclusions)
    write_json(ASSET_DIR / 'dev_truth.json', [dict(id=row['id'], gold=row['labels']) for row in index['dev']])
    counts = np.array([sum(label in row['labels'] for row in index['train']) for label in labels])
    with (ASSET_DIR / 'label_frequencies.csv').open('w', newline='', encoding='utf-8') as handle:
        writer = csv.writer(handle)
        writer.writerow(['statute_id', 'train_support', 'dev_support'])
        for label, count in zip(labels, counts):
            writer.writerow([label, int(count), sum(label in row['labels'] for row in index['dev'])])
    print(json.dumps(dict(splits=profiles, labels=metadata), indent=2), flush=True)
    print('Training records removed for conflicting annotations:', conflict_removed, flush=True)
    if profiles['test']['answer_fields_present'] or profiles['test']['dev_text_matches']:
        print('Data warning: exposed test fields or dev/test overlap detected. Test answer fields are ignored; consult the organizer about evaluation validity.', flush=True)
    return index, metadata


In [10]:
CASE_INDEX, LABEL_METADATA = audit_corpus()
LABEL_IDS = LABEL_METADATA['ids']
LABEL_TO_INDEX = {label: index for index, label in enumerate(LABEL_IDS)}
TRAIN_TARGETS = label_matrix(CASE_INDEX['train'])
DEV_TARGETS = label_matrix(CASE_INDEX['dev'])
print('Training cases:', len(TRAIN_TARGETS), '| Development cases:', len(DEV_TARGETS))
print('Observed statute IDs:', len(LABEL_IDS))
print(json.dumps(LABEL_METADATA, indent=2))


Check train annotations: 0it [00:00, ?it/s]

Conflicting training text groups: 80


Audit dev: 0it [00:00, ?it/s]

Audit test: 0it [00:00, ?it/s]

Audit train: 0it [00:00, ?it/s]

{
  "splits": {
    "dev": {
      "examples": 10181,
      "identical_duplicate_ids": 0,
      "word_percentiles": {
        "50": 543.0,
        "90": 2957.0,
        "95": 4230.0,
        "99": 8855.2,
        "100": 69230.0
      },
      "label_counts": {
        "0": 86,
        "1": 258,
        "2": 98,
        "3": 113,
        "4": 84,
        "5": 2936,
        "6": 174,
        "7": 260,
        "8": 92,
        "9": 206,
        "10": 1033,
        "11": 232,
        "12": 1086,
        "13": 1033,
        "14": 954,
        "15": 84,
        "16": 170,
        "17": 208,
        "18": 100,
        "19": 131,
        "20": 122,
        "21": 172,
        "22": 132,
        "23": 80,
        "24": 118,
        "25": 72,
        "26": 478,
        "27": 83,
        "28": 151,
        "29": 125,
        "30": 405,
        "31": 302,
        "32": 99,
        "33": 371,
        "34": 2596,
        "35": 790,
        "36": 142,
        "37": 318,
        "38": 425,
        "39"

## 4. Load the Phase 1 tokenizer and architectures

In [13]:
def split_bpe_pieces(text, max_chars=256):
    """Keep every character; separate words, punctuation and whitespace.

    An optional leading ASCII space can join a word or punctuation piece.
    Long pieces are split into bounded chunks to limit per-piece work.
    No normalization, strip(), split(), external tokenizer or regex package.
    """
    for match in re.finditer(' ?\\w+| ?[^\\w\\s]+|\\s+', text):
        piece = match.group(0)
        for start in range(0, len(piece), max_chars):
            yield piece[start:start + max_chars].encode('utf-8')


def apply_bpe_merges(piece, merge_ranks):
    """Apply learned merges by rank, breaking occurrence ties left-to-right.

    Token IDs 0..3 are reserved. Byte b always has ID b+4. A linked list
    and a heap avoid repeatedly scanning every adjacent pair in a piece.
    """
    ids = [byte + 4 for byte in piece]
    size = len(ids)
    if size < 2 or not merge_ranks:
        return tuple(ids)
    previous = [i - 1 for i in range(size)]
    following = [i + 1 for i in range(size)]
    following[-1] = -1
    active = [True] * size
    queue = []

    def offer(left):
        if left < 0 or not active[left]:
            return
        right = following[left]
        if right < 0:
            return
        pair = (ids[left], ids[right])
        rank = merge_ranks.get(pair)
        if rank is not None:
            heapq.heappush(queue, (rank, left, right, pair[0], pair[1]))
    for left in range(size - 1):
        offer(left)
    while queue:
        rank, left, right, a, b = heapq.heappop(queue)
        if not active[left] or not active[right] or following[left] != right:
            continue
        if ids[left] != a or ids[right] != b:
            continue
        ids[left] = 260 + rank
        active[right] = False
        following[left] = following[right]
        if following[right] >= 0:
            previous[following[right]] = left
        offer(previous[left])
        offer(left)
    result = []
    position = 0
    while position >= 0:
        result.append(ids[position])
        position = following[position]
    return tuple(result)


class ScratchByteBPE:
    """Our byte BPE vocabulary, encoder, decoder and JSON serialization.

    All base bytes and EVERY intermediate merge token are retained.
    Integer IDs identify symbols; concatenated decimal strings are never used.
    Literal '<s>' etc. in ordinary input are encoded as text, not control IDs.
    """
    FORMAT = 'handwritten-byte-bpe-v1'
    PATTERN = ' ?\\w+| ?[^\\w\\s]+|\\s+'

    def __init__(self, merges=None, max_piece_chars=256):
        self.specials = ['<pad>', '<unk>', '<s>', '</s>']
        self.max_piece_chars = max_piece_chars
        if not isinstance(max_piece_chars, int) or max_piece_chars < 1:
            raise ValueError('max_piece_chars must be a positive integer')
        self.merges = []
        self.merge_ranks = {}
        self.token_bytes = [None] * 4 + [bytes([b]) for b in range(256)]
        self._cache = {}
        for pair in merges or []:
            self.add_merge(tuple(pair))

    def add_merge(self, pair):
        if len(pair) != 2 or pair in self.merge_ranks:
            raise ValueError('Invalid or repeated merge pair')
        if any((type(token) is not int or not 4 <= token < len(self.token_bytes) for token in pair)):
            raise ValueError('A merge must reference earlier non-special token IDs')
        new_id = len(self.token_bytes)
        self.merge_ranks[pair] = len(self.merges)
        self.merges.append(pair)
        self.token_bytes.append(self.token_bytes[pair[0]] + self.token_bytes[pair[1]])
        self._cache.clear()
        return new_id

    def get_vocab_size(self):
        return len(self.token_bytes)

    def token_to_id(self, special):
        return self.specials.index(special) if special in self.specials else None

    def encode(self, text):
        if not isinstance(text, str):
            raise TypeError('encode expects a string')
        output = []
        for piece in split_bpe_pieces(text, self.max_piece_chars):
            encoded = self._cache.get(piece)
            if encoded is None:
                encoded = apply_bpe_merges(piece, self.merge_ranks)
                if len(self._cache) >= 65536:
                    self._cache.clear()
                self._cache[piece] = encoded
            output.extend(encoded)
        return SimpleNamespace(ids=output)

    def decode(self, ids, skip_special_tokens=True):
        chunks = []
        for value in ids:
            token = int(value)
            if not 0 <= token < len(self.token_bytes):
                raise ValueError(f'Unknown token ID: {token}')
            if token < 4:
                if not skip_special_tokens:
                    chunks.append(self.specials[token].encode('utf-8'))
            else:
                chunks.append(self.token_bytes[token])
        return b''.join(chunks).decode('utf-8', errors='replace')

    def save(self, path, training=None):
        payload = {'format': self.FORMAT, 'special_tokens': self.specials, 'byte_offset': 4, 'pattern': self.PATTERN, 'max_piece_chars': self.max_piece_chars, 'merges': self.merges, 'vocab_hex': [None if value is None else value.hex() for value in self.token_bytes]}
        if training is not None:
            payload['training'] = training
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        temporary = path.with_suffix(path.suffix + '.tmp')
        temporary.write_text(json.dumps(payload, ensure_ascii=False), encoding='utf-8')
        temporary.replace(path)

    @classmethod
    def from_file(cls, path):
        payload = json.loads(Path(path).read_text(encoding='utf-8'))
        if payload.get('format') != cls.FORMAT:
            raise ValueError('This is not our handwritten BPE format. Use a fresh data directory and retrain the tokenizer/token arrays.')
        if payload.get('special_tokens') != ['<pad>', '<unk>', '<s>', '</s>']:
            raise ValueError('Special-token IDs do not match')
        if payload.get('byte_offset') != 4 or payload.get('pattern') != cls.PATTERN:
            raise ValueError('Byte mapping or splitting rules do not match')
        tokenizer = cls(payload['merges'], payload['max_piece_chars'])
        expected = [None if value is None else value.hex() for value in tokenizer.token_bytes]
        if payload.get('vocab_hex') != expected:
            raise ValueError('Vocabulary is inconsistent with the merge history')
        return tokenizer


In [14]:
class DecoderAttention(nn.Module):

    def __init__(self, width, heads, dropout):
        super().__init__()
        assert width % heads == 0
        self.heads, self.dim, self.dropout = (heads, width // heads, dropout)
        self.qkv = nn.Linear(width, 3 * width)
        self.proj = nn.Linear(width, width)

    def forward(self, x, past=None, cache=False):
        b, t, c = x.shape
        q, k, v = self.qkv(x).view(b, t, 3, self.heads, self.dim).unbind(2)
        q, k, v = (a.transpose(1, 2) for a in (q, k, v))
        if past is not None:
            if t != 1:
                raise ValueError('Cached decoding accepts one new token at a time')
            k, v = (torch.cat((past[0], k), 2), torch.cat((past[1], v), 2))
        y = F.scaled_dot_product_attention(q, k, v, dropout_p=self.dropout if self.training else 0.0, is_causal=past is None)
        return (self.proj(y.transpose(1, 2).contiguous().view(b, t, c)), (k, v) if cache else None)


class Block(nn.Module):

    def __init__(self, width, heads, dropout):
        super().__init__()
        self.ln1, self.ln2 = (nn.LayerNorm(width), nn.LayerNorm(width))
        self.attn = DecoderAttention(width, heads, dropout)
        self.mlp = nn.Sequential(nn.Linear(width, 4 * width), nn.GELU(), nn.Linear(4 * width, width))
        self.drop = nn.Dropout(dropout)

    def forward(self, x, past=None, cache=False):
        a, kv = self.attn(self.ln1(x), past, cache)
        x = x + self.drop(a)
        return (x + self.drop(self.mlp(self.ln2(x))), kv)


class DecoderLM(nn.Module):

    def __init__(self, vocab_size=32768, width=704, layers=16, heads=11, context=2048, dropout=0.0):
        super().__init__()
        self.config = dict(vocab_size=vocab_size, width=width, layers=layers, heads=heads, context=context, dropout=dropout)
        self.token = nn.Embedding(vocab_size, width)
        self.position = nn.Embedding(context, width)
        self.blocks = nn.ModuleList([Block(width, heads, dropout) for _ in range(layers)])
        self.final_norm = nn.LayerNorm(width)
        self.drop = nn.Dropout(dropout)
        self.gradient_checkpointing = True
        self.apply(self._init)
        for block in self.blocks:
            nn.init.normal_(block.attn.proj.weight, std=0.02 / math.sqrt(2 * layers))
            nn.init.normal_(block.mlp[2].weight, std=0.02 / math.sqrt(2 * layers))

    @staticmethod
    def _init(module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, ids, targets=None, past=None, cache=False):
        offset = 0 if past is None else past[0][0].shape[2]
        if offset + ids.shape[1] > self.config['context']:
            raise ValueError('Context window exceeded')
        x = self.drop(self.token(ids) + self.position(torch.arange(offset, offset + ids.shape[1], device=ids.device)))
        kvs = []
        for i, block in enumerate(self.blocks):
            if self.training and self.gradient_checkpointing:
                x = checkpoint(lambda z, b=block: b(z)[0], x, use_reentrant=False)
            else:
                x, kv = block(x, None if past is None else past[i], cache)
                if cache:
                    kvs.append(kv)
        x = self.final_norm(x)
        if targets is not None:
            flat, labels = (x.reshape(-1, x.shape[-1]), targets.reshape(-1))

            def chunk_loss(h, y):
                return F.cross_entropy(F.linear(h, self.token.weight).float(), y, ignore_index=-100, reduction='sum')
            loss = x.new_zeros((), dtype=torch.float32)
            for start in range(0, len(labels), 256):
                h, y = (flat[start:start + 256], labels[start:start + 256])
                loss = loss + (checkpoint(chunk_loss, h, y, use_reentrant=False) if self.training else chunk_loss(h, y))
            return loss
        return (F.linear(x[:, -1], self.token.weight), kvs)


In [15]:
class EncDecAttention(nn.Module):
    def __init__(self,width,heads,dropout=0.0):
        super().__init__(); assert width%heads==0
        self.heads=heads; self.dim=width//heads; self.dropout=dropout
        self.q=nn.Linear(width,width); self.k=nn.Linear(width,width)
        self.v=nn.Linear(width,width); self.out=nn.Linear(width,width)

    def forward(self,x,memory=None,valid=None,causal=False,past=None,cache=False):
        b,t,w=x.shape
        def split(z): return z.view(b,-1,self.heads,self.dim).transpose(1,2)
        q=split(self.q(x))
        if memory is not None and past is not None:
            k,v=past
        else:
            source=x if memory is None else memory
            k,v=split(self.k(source)),split(self.v(source))
            if past is not None:
                k=torch.cat((past[0],k),dim=2); v=torch.cat((past[1],v),dim=2)
        mask=None
        if valid is not None: mask=valid[:,None,None,:]
        if causal:
            offset=k.shape[2]-t
            allowed=torch.arange(k.shape[2],device=x.device)[None,:] <= (torch.arange(t,device=x.device)+offset)[:,None]
            mask=allowed[None,None,:,:] if mask is None else mask & allowed[None,None,:,:]
        h=F.scaled_dot_product_attention(q,k,v,attn_mask=mask,
            dropout_p=self.dropout if self.training else 0.0)
        h=self.out(h.transpose(1,2).contiguous().view(b,t,w))
        return h, (k,v) if cache else None

class EncoderBlock(nn.Module):
    def __init__(self,w,h,p):
        super().__init__(); self.n1=nn.LayerNorm(w); self.n2=nn.LayerNorm(w)
        self.attn=EncDecAttention(w,h,p); self.drop=nn.Dropout(p)
        self.ff=nn.Sequential(nn.Linear(w,4*w),nn.GELU(),nn.Linear(4*w,w))
    def forward(self,x,valid):
        x=x+self.drop(self.attn(self.n1(x),valid=valid)[0])
        return x+self.drop(self.ff(self.n2(x)))

class DecoderBlock(nn.Module):
    def __init__(self,w,h,p):
        super().__init__(); self.n1=nn.LayerNorm(w); self.n2=nn.LayerNorm(w); self.n3=nn.LayerNorm(w)
        self.self_attn=EncDecAttention(w,h,p); self.cross=EncDecAttention(w,h,p); self.drop=nn.Dropout(p)
        self.ff=nn.Sequential(nn.Linear(w,4*w),nn.GELU(),nn.Linear(4*w,w))
    def forward(self,x,memory,source_valid,target_valid=None,past=None,cache=False):
        self_past,cross_past=(None,None) if past is None else past
        a,skv=self.self_attn(self.n1(x),valid=target_valid,causal=True,past=self_past,cache=cache)
        x=x+self.drop(a)
        a,ckv=self.cross(self.n2(x),memory=memory,valid=source_valid,past=cross_past,cache=cache)
        x=x+self.drop(a); x=x+self.drop(self.ff(self.n3(x)))
        return x, (skv,ckv) if cache else None


class EncoderDecoder(nn.Module):
    def __init__(self,vocab_size=32768,width=704,encoder_layers=6,decoder_layers=8,heads=11,dropout=0.0):
        super().__init__()
        self.config=dict(vocab_size=vocab_size,width=width,encoder_layers=encoder_layers,
                         decoder_layers=decoder_layers,heads=heads,dropout=dropout)
        self.token=nn.Embedding(vocab_size,width)
        self.encoder=nn.ModuleList([EncoderBlock(width,heads,dropout) for _ in range(encoder_layers)])
        self.decoder=nn.ModuleList([DecoderBlock(width,heads,dropout) for _ in range(decoder_layers)])
        self.enc_norm=nn.LayerNorm(width); self.dec_norm=nn.LayerNorm(width)
        self.drop=nn.Dropout(dropout); self.gradient_checkpointing=True
        self.apply(self.initialize)

    @staticmethod
    def initialize(m):
        if isinstance(m,(nn.Linear,nn.Embedding)):
            nn.init.normal_(m.weight,std=0.02)
            if isinstance(m,nn.Linear) and m.bias is not None: nn.init.zeros_(m.bias)

    def embed(self,ids,offset=0):
        w=self.config['width']; positions=torch.arange(offset,offset+ids.shape[1],device=ids.device).float()
        frequency=torch.exp(torch.arange(0,w,2,device=ids.device).float()*(-math.log(10000)/w))
        angle=positions[:,None]*frequency[None,:]
        pe=torch.stack((angle.sin(),angle.cos()),dim=-1).flatten(1)[:,:w]
        return self.drop(self.token(ids)+pe.to(self.token.weight.dtype))

    def encode(self,src):
        valid=src.ne(PAD); x=self.embed(src)
        for block in self.encoder:
            if self.training and self.gradient_checkpointing:
                x=checkpoint(block,x,valid,use_reentrant=False)
            else: x=block(x,valid)
        return self.enc_norm(x),valid

    def decode(self,ids,memory,source_valid,past=None,cache=False):
        offset=0 if past is None else past[0][0][0].shape[2]
        x=self.embed(ids,offset)
        valid=ids.ne(PAD) if past is None else None
        caches=[]
        for i,block in enumerate(self.decoder):
            if self.training and self.gradient_checkpointing:
                x=checkpoint(lambda a,b,c,d,layer=block: layer(a,b,c,d)[0],
                    x,memory,source_valid,valid,use_reentrant=False)
            else:
                x,kv=block(x,memory,source_valid,valid,None if past is None else past[i],cache)
                if cache: caches.append(kv)
        return self.dec_norm(x),caches

    def forward(self,src,decoder_ids,labels=None):
        memory,valid=self.encode(src); h,_=self.decode(decoder_ids,memory,valid)
        if labels is None: return F.linear(h,self.token.weight)
        flat=h.reshape(-1,h.shape[-1]); labels=labels.reshape(-1)
        def loss_chunk(a,b):
            return F.cross_entropy(F.linear(a,self.token.weight).float(),b,ignore_index=-100,reduction='sum')
        loss=h.new_zeros((),dtype=torch.float32)
        for start in range(0,len(labels),128):
            a,b=flat[start:start+128],labels[start:start+128]
            loss=loss+(checkpoint(loss_chunk,a,b,use_reentrant=False) if self.training else loss_chunk(a,b))
        return loss


In [16]:
def sliding_attention(q,k,v,window,chunk,dropout=0.0):
    b,h,t,d=q.shape; blocks=math.ceil(t/chunk); padded=blocks*chunk
    qpad=F.pad(q,(0,0,0,padded-t))
    query=qpad.reshape(b,h,blocks,chunk,d).permute(0,2,1,3,4).reshape(b*blocks,h,chunk,d)
    positions=torch.arange(blocks,device=q.device)[:,None]*chunk-window+1+torch.arange(window+chunk-1,device=q.device)[None,:]
    indices=positions.clamp(0,t-1)
    keys=k[:,:,indices,:].permute(0,2,1,3,4).reshape(b*blocks,h,window+chunk-1,d)
    values=v[:,:,indices,:].permute(0,2,1,3,4).reshape(b*blocks,h,window+chunk-1,d)
    qi=torch.arange(chunk,device=q.device)[:,None]
    kj=torch.arange(window+chunk-1,device=q.device)[None,:]
    relative=(kj>=qi)&(kj<=qi+window-1)
    valid=(positions>=0)&(positions<t)
    mask=(relative[None,:,:]&valid[:,None,:])[None,:,:,:].expand(b,-1,-1,-1).reshape(b*blocks,1,chunk,window+chunk-1)
    output=F.scaled_dot_product_attention(query,keys,values,attn_mask=mask,dropout_p=dropout)
    return output.reshape(b,blocks,h,chunk,d).permute(0,2,1,3,4).reshape(b,h,padded,d)[:,:,:t]

class MixedAttention(nn.Module):
    def __init__(self,width,heads,window,query_chunk,global_layer,dropout):
        super().__init__(); assert width%heads==0
        self.heads=heads; self.dim=width//heads; self.window=window
        self.chunk=query_chunk; self.global_layer=global_layer; self.dropout=dropout
        self.qkv=nn.Linear(width,3*width); self.proj=nn.Linear(width,width)
    def forward(self,x,past=None,cache=False):
        b,t,w=x.shape
        q,k,v=self.qkv(x).view(b,t,3,self.heads,self.dim).unbind(2)
        q,k,v=(a.transpose(1,2) for a in (q,k,v))
        if past is not None:
            if t!=1: raise ValueError('Incremental decoding expects one token')
            k=torch.cat((past[0],k),dim=2); v=torch.cat((past[1],v),dim=2)
        p=self.dropout if self.training else 0.0
        if self.global_layer or past is not None:
            output=F.scaled_dot_product_attention(q,k,v,dropout_p=p,is_causal=past is None)
        else: output=sliding_attention(q,k,v,self.window,self.chunk,p)
        kv=None
        if cache:
            kv=(k,v) if self.global_layer else (k[:,:,-(self.window-1):].contiguous(),v[:,:,-(self.window-1):].contiguous())
        return self.proj(output.transpose(1,2).contiguous().view(b,t,w)),kv

class MixedBlock(nn.Module):
    def __init__(self,width,heads,window,query_chunk,global_layer,dropout):
        super().__init__(); self.n1=nn.LayerNorm(width); self.n2=nn.LayerNorm(width)
        self.attn=MixedAttention(width,heads,window,query_chunk,global_layer,dropout)
        self.ff=nn.Sequential(nn.Linear(width,4*width),nn.GELU(),nn.Linear(4*width,width))
        self.drop=nn.Dropout(dropout)
    def forward(self,x,past=None,cache=False):
        a,kv=self.attn(self.n1(x),past,cache); x=x+self.drop(a)
        return x+self.drop(self.ff(self.n2(x))),kv


class LocalGlobalLM(nn.Module):
    def __init__(self,vocab_size=32768,width=704,layers=16,heads=11,context=2048,window=256,
                 global_every=4,query_chunk=128,dropout=0.0,activation_checkpointing=True,loss_chunk=256):
        super().__init__()
        if min(context,query_chunk,global_every,loss_chunk)<1 or window<2: raise ValueError('Invalid architecture limits')
        self.config=dict(vocab_size=vocab_size,width=width,layers=layers,heads=heads,context=context,
            window=window,global_every=global_every,query_chunk=query_chunk,dropout=dropout,
            activation_checkpointing=activation_checkpointing,loss_chunk=loss_chunk)
        self.token=nn.Embedding(vocab_size,width)
        self.blocks=nn.ModuleList([MixedBlock(width,heads,window,query_chunk,(i+1)%global_every==0,dropout) for i in range(layers)])
        self.norm=nn.LayerNorm(width); self.drop=nn.Dropout(dropout)
        self.apply(self.initialize)
        for block in self.blocks:
            nn.init.normal_(block.attn.proj.weight,std=0.02/math.sqrt(2*layers))
            nn.init.normal_(block.ff[2].weight,std=0.02/math.sqrt(2*layers))
    @staticmethod
    def initialize(m):
        if isinstance(m,(nn.Linear,nn.Embedding)):
            nn.init.normal_(m.weight,std=0.02)
            if isinstance(m,nn.Linear) and m.bias is not None: nn.init.zeros_(m.bias)
    def hidden(self,ids,past=None,position_offset=0,cache=False):
        t=ids.shape[1]; w=self.config['width']
        if position_offset+t>self.config['context']: raise ValueError('Context exceeded')
        pos=torch.arange(position_offset,position_offset+t,device=ids.device).float()
        freq=torch.exp(torch.arange(0,w,2,device=ids.device).float()*(-math.log(10000)/w))
        phase=pos[:,None]*freq[None,:]
        pe=torch.stack((phase.sin(),phase.cos()),dim=-1).flatten(1)[:,:w]
        # Match sinusoidal amplitude with scaled learned token embeddings.
        x=self.drop(self.token(ids)*math.sqrt(w)+pe.to(self.token.weight.dtype))
        caches=[]
        for i,block in enumerate(self.blocks):
            if self.training and self.config['activation_checkpointing']:
                x=checkpoint(lambda z,layer=block: layer(z)[0],x,use_reentrant=False)
            else:
                x,kv=block(x,None if past is None else past[i],cache)
                if cache: caches.append(kv)
        return self.norm(x),caches
    def forward(self,ids,targets=None,past=None,position_offset=0,cache=False):
        h,caches=self.hidden(ids,past,position_offset,cache)
        if targets is None: return F.linear(h[:,-1],self.token.weight),caches
        flat=h.reshape(-1,h.shape[-1]); labels=targets.reshape(-1)
        def chunk_loss(a,b): return F.cross_entropy(F.linear(a,self.token.weight).float(),b,ignore_index=-100,reduction='sum')
        loss=h.new_zeros((),dtype=torch.float32); size=self.config['loss_chunk']
        for start in range(0,len(labels),size):
            a,b=flat[start:start+size],labels[start:start+size]
            loss=loss+(checkpoint(chunk_loss,a,b,use_reentrant=False) if self.training and self.config['activation_checkpointing'] else chunk_loss(a,b))
        return loss


In [17]:
class CausalGatedConv(nn.Module):
    def __init__(self,width,kernel_size,dilation):
        super().__init__()
        if kernel_size<2 or dilation<1: raise ValueError('Invalid causal convolution settings')
        self.history=(kernel_size-1)*dilation
        self.conv=nn.Conv1d(width,2*width,kernel_size,dilation=dilation,padding=0)
        self.proj=nn.Linear(width,width)

    def forward(self,x,past=None,cache=False):
        values=x.transpose(1,2)
        if past is None:
            padded=F.pad(values,(self.history,0))
        else:
            if past.shape!=(x.shape[0],x.shape[2],self.history):
                raise ValueError('Convolution cache shape mismatch')
            padded=torch.cat((past,values),dim=2)
        features=self.conv(padded)
        a,b=features.chunk(2,dim=1)
        output=self.proj((a*torch.sigmoid(b)).transpose(1,2))
        # clone prevents a small cache view from retaining a full prompt buffer.
        state=padded[:,:,-self.history:].clone() if cache else None
        return output,state

class ConvBlock(nn.Module):
    def __init__(self,width,kernel_size,dilation,dropout):
        super().__init__()
        self.n1=nn.LayerNorm(width); self.n2=nn.LayerNorm(width)
        self.causal=CausalGatedConv(width,kernel_size,dilation)
        self.ff=nn.Sequential(nn.Linear(width,4*width),nn.GELU(),nn.Linear(4*width,width))
        self.drop=nn.Dropout(dropout)

    def forward(self,x,past=None,cache=False):
        h,state=self.causal(self.n1(x),past,cache)
        x=x+self.drop(h)
        return x+self.drop(self.ff(self.n2(x))),state


class GatedConvLM(nn.Module):
    def __init__(self,vocab_size=32768,width=640,layers=16,kernel_size=3,dilation_cycle=10,
                 context=2048,dropout=0.0,activation_checkpointing=True,loss_chunk=256):
        super().__init__()
        if min(width,layers,dilation_cycle,context,loss_chunk)<1 or kernel_size<2:
            raise ValueError('Invalid model configuration')
        self.config=dict(vocab_size=vocab_size,width=width,layers=layers,kernel_size=kernel_size,
            dilation_cycle=dilation_cycle,context=context,dropout=dropout,
            activation_checkpointing=activation_checkpointing,loss_chunk=loss_chunk)
        self.dilations=[2**(i%dilation_cycle) for i in range(layers)]
        self.receptive_field=1+(kernel_size-1)*sum(self.dilations)
        self.token=nn.Embedding(vocab_size,width)
        self.blocks=nn.ModuleList([ConvBlock(width,kernel_size,d,dropout) for d in self.dilations])
        self.norm=nn.LayerNorm(width); self.drop=nn.Dropout(dropout)
        self.apply(self.initialize)
        for block in self.blocks:
            nn.init.normal_(block.causal.proj.weight,std=0.02/math.sqrt(2*layers))
            nn.init.normal_(block.ff[2].weight,std=0.02/math.sqrt(2*layers))

    @staticmethod
    def initialize(module):
        if isinstance(module,(nn.Linear,nn.Conv1d,nn.Embedding)):
            nn.init.normal_(module.weight,std=0.02)
            if getattr(module,'bias',None) is not None: nn.init.zeros_(module.bias)

    def hidden(self,ids,past=None,cache=False):
        if past is None and ids.shape[1]>self.config['context']:
            raise ValueError('Initial prompt/training block exceeds configured context')
        if past is not None and (len(past)!=len(self.blocks) or ids.shape[1]!=1):
            raise ValueError('Cached generation accepts one token and one state per layer')
        if self.training and (cache or past is not None):
            raise ValueError('Caches are for eval-mode generation only')
        x=self.drop(self.token(ids)); states=[]
        for i,block in enumerate(self.blocks):
            if self.training and self.config['activation_checkpointing']:
                x=checkpoint(lambda z,layer=block: layer(z)[0],x,use_reentrant=False)
            else:
                x,state=block(x,None if past is None else past[i],cache)
                if cache: states.append(state)
        return self.norm(x),states

    def forward(self,ids,targets=None,past=None,cache=False):
        h,states=self.hidden(ids,past,cache)
        if targets is None: return F.linear(h[:,-1],self.token.weight),states
        flat=h.reshape(-1,h.shape[-1]); labels=targets.reshape(-1)
        def chunk_loss(a,b):
            return F.cross_entropy(F.linear(a,self.token.weight).float(),b,ignore_index=-100,reduction='sum')
        loss=h.new_zeros((),dtype=torch.float32); size=self.config['loss_chunk']
        for start in range(0,len(labels),size):
            a,b=flat[start:start+size],labels[start:start+size]
            loss=loss+(checkpoint(chunk_loss,a,b,use_reentrant=False) if self.training and self.config['activation_checkpointing'] else chunk_loss(a,b))
        return loss


In [18]:
MODEL_CLASSES = {
    "model1": DecoderLM,
    "model2": EncoderDecoder,
    "model3": LocalGlobalLM,
    "model4": GatedConvLM,
}

BASES = {}
for name in MODEL_NAMES:
    folder = MODEL_DIRS[name]
    checkpoint_path = folder / "best.pt"
    tokenizer_path = folder / "tokenizer.json"
    state = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    config = copy.deepcopy(state["model_config"])
    detected = classify_model_config(config)
    if detected != name:
        raise ValueError(f"{folder} is {detected}, expected {name}")
    tokenizer = ScratchByteBPE.from_file(tokenizer_path)
    if config["vocab_size"] != tokenizer.get_vocab_size():
        raise ValueError(f"Vocabulary mismatch for {name}")
    expected_hash = state.get("tokenizer_sha256")
    if expected_hash and digest_file(tokenizer_path) != expected_hash:
        raise ValueError(f"Tokenizer hash mismatch for {name}")
    BASES[name] = {
        "folder": folder,
        "checkpoint": checkpoint_path,
        "tokenizer_path": tokenizer_path,
        "tokenizer": tokenizer,
        "config": config,
        "state": state["model"],
        "checkpoint_sha256": digest_file(checkpoint_path),
        "tokenizer_sha256": digest_file(tokenizer_path),
    }

base_report = {}
for name in MODEL_NAMES:
    model = MODEL_CLASSES[name](**BASES[name]["config"])
    result = model.load_state_dict(BASES[name]["state"], strict=True)
    assert not result.missing_keys and not result.unexpected_keys
    count = sum(parameter.numel() for parameter in model.parameters())
    base_report[name] = {"parameters": count, "config": BASES[name]["config"]}
    print(name, f"{count:,}", BASES[name]["config"])
    del model
    gc.collect()
write_json(ASSET_DIR / "phase1_models.json", base_report)


model1 119,816,576 {'vocab_size': 32768, 'width': 704, 'layers': 16, 'heads': 11, 'context': 2048, 'dropout': 0.0}
model2 122,356,608 {'vocab_size': 32768, 'width': 704, 'encoder_layers': 6, 'decoder_layers': 8, 'heads': 11, 'dropout': 0.0}
model3 118,374,784 {'vocab_size': 32768, 'width': 704, 'layers': 16, 'heads': 11, 'context': 2048, 'window': 256, 'global_every': 4, 'query_chunk': 128, 'dropout': 0.0, 'activation_checkpointing': False, 'loss_chunk': 256}
model4 119,399,680 {'vocab_size': 32768, 'width': 640, 'layers': 16, 'kernel_size': 3, 'dilation_cycle': 10, 'context': 2048, 'dropout': 0.0, 'activation_checkpointing': False, 'loss_chunk': 256}


## 5. Prepare token caches

In [19]:
prepare_all_tokens()
coverage = {}
for name in MODEL_NAMES:
    coverage[name] = coverage_report(name, DEFAULT)
write_json(ASSET_DIR / 'default_coverage.json', coverage)
print(json.dumps(coverage, indent=2))


train [647.0, 4163.0, 5974.0, 12258.959999999992, 140017.0]
dev [800.0, 4473.0, 6469.0, 13519.400000000081, 116222.0]
test [794.0, 4256.600000000002, 6089.199999999997, 13120.039999999994, 217737.0]
{
  "train": {
    "token_coverage": 0.7504362669252369,
    "mean_case_coverage": 0.9586503546758783,
    "mean_chunks": 1.8397463303406014,
    "max_chunks": 4
  },
  "dev": {
    "token_coverage": 0.7406980588201497,
    "mean_case_coverage": 0.9520605533699282,
    "mean_chunks": 1.9556035752872998,
    "max_chunks": 4
  },
  "test": {
    "token_coverage": 0.7421527643035492,
    "mean_case_coverage": 0.9565465091074883,
    "mean_chunks": 1.924802212151471,
    "max_chunks": 4
  }
}
{
  "train": {
    "token_coverage": 0.7504362669252369,
    "mean_case_coverage": 0.9586503546758783,
    "mean_chunks": 1.8397463303406014,
    "max_chunks": 4
  },
  "dev": {
    "token_coverage": 0.7406980588201497,
    "mean_case_coverage": 0.9520605533699282,
    "mean_chunks": 1.9556035752872998,
  

## 6. Classification, adaptation, and loss functions


In [20]:
def set_activation_checkpointing(backbone, enabled):
    if hasattr(backbone, 'gradient_checkpointing'):
        backbone.gradient_checkpointing = enabled
    if 'activation_checkpointing' in backbone.config:
        backbone.config['activation_checkpointing'] = enabled


class LoRMAPlus(nn.Module):
    def __init__(self, base, rank, alpha):
        super().__init__()
        self.base = base
        self.base.requires_grad_(False)
        self.A = nn.Parameter(base.weight.new_empty(rank, base.out_features))
        self.B = nn.Parameter(base.weight.new_zeros(base.out_features, rank))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.scale = alpha / rank
    def forward(self, x):
        z = F.linear(x, self.base.weight)
        z = z + self.scale * F.linear(F.linear(z, self.A), self.B)
        return z if self.base.bias is None else z + self.base.bias


def install_lorma(module, rank, alpha, prefix=''):
    names = []
    for name, child in list(module.named_children()):
        path = f'{prefix}.{name}' if prefix else name
        if isinstance(child, nn.Linear):
            setattr(module, name, LoRMAPlus(child, rank, alpha))
            names.append(path)
        else:
            names.extend(install_lorma(child, rank, alpha, path))
    return names


class BottleneckAdapter(nn.Module):
    def __init__(self, width, bottleneck, dropout):
        super().__init__()
        self.norm = nn.LayerNorm(width)
        self.down = nn.Linear(width, bottleneck)
        self.up = nn.Linear(bottleneck, width)
        self.drop = nn.Dropout(dropout)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)
    def forward(self, x):
        return self.up(self.drop(F.gelu(self.down(self.norm(x)))))


class AdapterBlock(nn.Module):
    def __init__(self, base, width, bottleneck, dropout):
        super().__init__()
        self.base = base
        self.adapter = BottleneckAdapter(width, bottleneck, dropout)
    def forward(self, *args, **kwargs):
        output = self.base(*args, **kwargs)
        if isinstance(output, tuple):
            hidden = output[0]
            return (hidden + self.adapter(hidden),) + output[1:]
        return output + self.adapter(output)


def decoder_hidden(backbone, ids):
    if ids.shape[1] > backbone.config['context']:
        raise ValueError('Context exceeded.')
    pos = torch.arange(ids.shape[1], device=ids.device)
    hidden = backbone.drop(backbone.token(ids) + backbone.position(pos))
    for block in backbone.blocks:
        if backbone.training and backbone.gradient_checkpointing:
            hidden = checkpoint(lambda x, layer=block: layer(x)[0], hidden, use_reentrant=False)
        else:
            hidden = block(hidden)[0]
    return backbone.final_norm(hidden)


class ChunkEncoder(nn.Module):
    def __init__(self, model_name, cfg):
        super().__init__()
        base = BASES[model_name]
        self.model_name = model_name
        self.pool = cfg['model2_pool']
        self.backbone = MODEL_CLASSES[model_name](**copy.deepcopy(base['config']))
        self.backbone.load_state_dict(base['state'], strict=True)
        set_activation_checkpointing(self.backbone, cfg['activation_checkpointing'])
        limit = base['config'].get('context', 1408)
        if cfg['chunk_tokens'] > limit:
            raise ValueError(f'Chunk length exceeds pretrained input limit {limit}.')
        if cfg['student_keep'] is not None:
            keep = cfg['student_keep']
            if model_name != 'model4' or not keep or sorted(set(keep)) != keep:
                raise ValueError('Invalid reduced CNN block indices.')
            if min(keep) < 0 or max(keep) >= len(self.backbone.blocks):
                raise ValueError('Student block index out of range.')
            self.backbone.blocks = nn.ModuleList([self.backbone.blocks[i] for i in keep])
            self.backbone.dilations = [self.backbone.dilations[i] for i in keep]
            self.backbone.config['layers'] = len(keep)
        if model_name == 'model2' and self.pool == 'encoder_only':
            self.backbone.decoder.requires_grad_(False)
            self.backbone.dec_norm.requires_grad_(False)
    def forward(self, ids, mask):
        if self.model_name == 'model2':
            hidden, valid = self.backbone.encode(ids)
            if self.pool == 'encoder_query':
                query = torch.full((len(ids), 1), BOS, dtype=torch.long, device=ids.device)
                decoded, _ = self.backbone.decode(query, hidden, valid)
                summary = decoded[:, -1].float()
            elif self.pool == 'encoder_only':
                summary = (hidden.float() * mask.unsqueeze(-1)).sum(1) / mask.sum(1, keepdim=True).clamp_min(1)
            else:
                raise ValueError('Unknown Model 2 pooling.')
        else:
            hidden = decoder_hidden(self.backbone, ids) if self.model_name == 'model1' else self.backbone.hidden(ids)[0]
            last = mask.long().sum(1) - 1
            summary = hidden[torch.arange(len(ids), device=ids.device), last].float()
        mean = (hidden.float() * mask.unsqueeze(-1)).sum(1) / mask.sum(1, keepdim=True).clamp_min(1)
        return torch.cat([mean, summary], dim=-1)


class StatuteClassifier(nn.Module):
    def __init__(self, model_name, method, cfg, total_steps):
        super().__init__()
        self.model_name, self.method, self.cfg = model_name, method, copy.deepcopy(cfg)
        self.encoder = ChunkEncoder(model_name, cfg)
        self.adapted = []
        width = BASES[model_name]['config']['width']
        if method == 'lorma':
            self.encoder.requires_grad_(False)
            self.adapted = install_lorma(self.encoder.backbone.blocks, cfg['rank'], cfg['alpha'])
        elif method == 'adapters':
            if model_name != 'model2':
                raise ValueError('This experiment installs adapters in Model 2.')
            self.encoder.requires_grad_(False)
            for group in ('encoder', 'decoder'):
                if group == 'decoder' and cfg['model2_pool'] == 'encoder_only':
                    continue
                blocks = getattr(self.encoder.backbone, group)
                for index in range(len(blocks)):
                    blocks[index] = AdapterBlock(blocks[index], width, cfg['adapter_dim'], cfg['adapter_dropout'])
                    self.adapted.append(f'{group}.{index}')
        elif method not in ('full', 'kd', 'control'):
            raise ValueError(method)
        if cfg.get('freeze_backbone', False):
            self.encoder.requires_grad_(False)
        dim = cfg['head_dim']
        self.projection = nn.Sequential(nn.LayerNorm(2 * width), nn.Linear(2 * width, dim), nn.GELU(), nn.Dropout(cfg['dropout']))
        self.position = nn.Linear(1, dim, bias=False)
        self.norm = nn.LayerNorm(dim)
        self.queries = nn.Parameter(torch.empty(len(LABEL_IDS), dim))
        self.output_weight = nn.Parameter(torch.empty(len(LABEL_IDS), dim))
        self.output_bias = nn.Parameter(torch.zeros(len(LABEL_IDS)))
        nn.init.normal_(self.queries, std=0.02)
        nn.init.normal_(self.output_weight, std=0.02)
    def forward(self, ids, mask, positions, attention_mode=None):
        batch, chunks, length = ids.shape
        valid = mask.any(-1)
        flat_valid = valid.flatten()
        features = self.encoder(ids.reshape(batch * chunks, length)[flat_valid],
                                mask.reshape(batch * chunks, length)[flat_valid])
        features = self.projection(features)
        dense = features.new_zeros(batch * chunks, features.shape[-1])
        dense[flat_valid] = features
        dense = dense.reshape(batch, chunks, -1)
        dense = self.norm(dense + self.position(positions.unsqueeze(-1).to(dense.dtype)))
        mode = self.cfg['attention_mode'] if attention_mode is None else attention_mode
        if mode == 'label':
            scores = torch.einsum('bkd,cd->bck', dense.float(), self.queries.float()) / math.sqrt(dense.shape[-1])
            scores = scores.masked_fill(~valid[:, None, :], float('-inf'))
            weights = scores.softmax(-1).to(dense.dtype)
            pooled = torch.einsum('bck,bkd->bcd', weights, dense)
        elif mode == 'mean':
            mean = (dense * valid.unsqueeze(-1)).sum(1) / valid.sum(1, keepdim=True).clamp_min(1)
            pooled = mean[:, None, :].expand(-1, len(LABEL_IDS), -1)
        else:
            raise ValueError('Unknown attention mode.')
        return (pooled.float() * self.output_weight[None, :, :].float()).sum(-1) + self.output_bias.float()


def parameter_report(model):
    return dict(total=sum(p.numel() for p in model.parameters()),
                trainable=sum(p.numel() for p in model.parameters() if p.requires_grad),
                adapted_modules=len(model.adapted))


def make_model(model_name, method, cfg, total_steps, device):
    return StatuteClassifier(model_name, method, cfg, total_steps).to(device)


def supervised_loss(logits, targets, cfg, pos_weight=None):
    logits, targets = logits.float(), targets.float()
    if cfg['loss'] == 'bce':
        return F.binary_cross_entropy_with_logits(logits, targets, pos_weight=pos_weight, reduction='none').mean(-1)
    if cfg['loss'] == 'asymmetric':
        positive = logits.sigmoid()
        negative = (1 - positive + cfg['asl_clip']).clamp(max=1)
        loss = targets * F.logsigmoid(logits) + (1 - targets) * torch.log(negative.clamp_min(1e-8))
        probability = positive * targets + negative * (1 - targets)
        gamma = cfg['asl_gamma_pos'] * targets + cfg['asl_gamma_neg'] * (1 - targets)
        return -(loss * (1 - probability).pow(gamma)).mean(-1)
    raise ValueError('Unknown loss.')


def distillation_loss(logits, teacher_logits, temperature):
    target = (teacher_logits.float() / temperature).sigmoid()
    student = logits.float() / temperature
    cross_entropy = F.binary_cross_entropy_with_logits(student, target, reduction='none')
    entropy = -(target * torch.log(target.clamp_min(1e-8)) +
                (1 - target) * torch.log((1 - target).clamp_min(1e-8)))
    return (cross_entropy - entropy).mean(-1) * temperature**2


def training_positive_weights(cfg, device):
    if not cfg['positive_weighting'] or cfg['loss'] != 'bce':
        return None
    counts = TRAIN_TARGETS.sum(0)
    ratio = (len(TRAIN_TARGETS) - counts) / np.maximum(counts, 1)
    values = np.minimum(np.maximum(ratio, 1)**cfg['positive_weight_power'], cfg['positive_weight_cap'])
    values[counts == 0] = 1
    return torch.tensor(values, dtype=torch.float32, device=device)


def check_backbones():
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    for name in MODEL_NAMES:
        tok = BASES[name]['tokenizer']
        text = 'Section 42: भारत ₹100.'
        assert tok.decode(tok.encode(text).ids) == text
        cfg = dict(DEFAULT, chunk_tokens=32, representation='head', dropout=0.0)
        model = make_model(name, 'full', cfg, 100, device)
        tokens = tok.encode('The court considered evidence and the facts of the case.').ids[:28]
        sample = ([np.asarray(tokens, dtype=np.int64)], [0.0], np.zeros(len(LABEL_IDS)), 0)
        batch = collate_cases([sample])
        values = {key: batch[key].to(device) for key in ('ids', 'mask', 'positions')}
        model.eval()
        with torch.no_grad():
            first = model(**values)
            padded = dict(values, ids=F.pad(values['ids'], (0, 5), value=PAD), mask=F.pad(values['mask'], (0, 5), value=False))
            second = model(**padded)
        torch.testing.assert_close(first, second, rtol=2e-4, atol=2e-4)
        model.train()
        supervised_loss(model(**values), batch['labels'].to(device), cfg).mean().backward()
        assert all(torch.isfinite(p.grad).all() for p in model.parameters() if p.grad is not None)
        print(name, 'padding/gradient check passed', parameter_report(model), flush=True)
        del model, batch, values, first, second
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()


## 7. Metrics, capacity tests, and training

The loss is computed over independent statute labels. Distillation uses Bernoulli KL for every label, with temperature scaling; a softmax across statutes would be wrong for this task.

Capacity tests use disposable models and a full-length batch, backward pass, and optimizer step. AMP may lower its scale and retry a batch with non-finite gradients. Successful updates alone advance the schedule. Persistent consecutive failures stop with a saved checkpoint; occasional lifetime skips do not trigger the old 20-skip limit.

`best.pt` stores the highest development Macro-F1 checkpoint and its threshold. `latest.pt` stores model, optimizer, scheduler, AMP scale, random state, data position, and patience for continuation. Evaluation runs at epoch boundaries and session pauses. Training progress prints every 20 successful updates.


In [21]:
def decisions(probabilities, threshold):
    probabilities = np.asarray(probabilities, dtype=np.float64)
    threshold = np.asarray(threshold, dtype=np.float64)
    if probabilities.ndim != 2 or probabilities.shape[1] != len(LABEL_IDS):
        raise ValueError('Probability/label dimensions differ.')
    if not np.isfinite(probabilities).all() or np.any((probabilities < 0) | (probabilities > 1)):
        raise ValueError('Invalid probabilities.')
    if threshold.ndim > 1 or (threshold.ndim == 1 and threshold.shape != (len(LABEL_IDS),)):
        raise ValueError('Use one global threshold or one threshold per label.')
    if np.any((threshold <= 0) | (threshold >= 1)):
        raise ValueError('Thresholds must lie strictly between zero and one.')
    return probabilities >= threshold


def score_predictions(truth, prediction):
    truth, prediction = np.asarray(truth, dtype=bool), np.asarray(prediction, dtype=bool)
    if truth.shape != prediction.shape or truth.ndim != 2:
        raise ValueError('Truth/prediction dimensions differ.')
    true_positive = np.logical_and(truth, prediction).sum(0)
    false_positive = np.logical_and(~truth, prediction).sum(0)
    false_negative = np.logical_and(truth, ~prediction).sum(0)
    precision = true_positive / np.maximum(true_positive + false_positive, 1)
    recall = true_positive / np.maximum(true_positive + false_negative, 1)
    per_f1 = 2 * true_positive / np.maximum(2 * true_positive + false_positive + false_negative, 1)
    denominator = LABEL_METADATA['evaluator_dimension']
    micro_den = 2 * true_positive.sum() + false_positive.sum() + false_negative.sum()
    return dict(macro_f1=float(per_f1.sum() / denominator),
                micro_f1=float(2 * true_positive.sum() / max(micro_den, 1)),
                subset_accuracy=float(np.all(truth == prediction, axis=1).mean()),
                macro_precision=float(precision.sum() / denominator),
                macro_recall=float(recall.sum() / denominator),
                mean_predicted_labels=float(prediction.sum(1).mean()),
                mean_gold_labels=float(truth.sum(1).mean()),
                empty_predictions=int((prediction.sum(1) == 0).sum()),
                label_dimension=denominator,
                professor_dimension_matches=denominator == LABEL_METADATA['dev_evaluator_dimension'])


def metric_report(truth, probabilities, threshold=0.5):
    return score_predictions(truth, decisions(probabilities, threshold))


def select_global_threshold(truth, probabilities, grid):
    rows = [dict(threshold=float(t), **metric_report(truth, probabilities, float(t))) for t in grid]
    return max(rows, key=lambda row: (row['macro_f1'], -abs(row['threshold'] - 0.5))), rows


def predictions_json(records, probabilities, threshold):
    predicted = decisions(probabilities, threshold)
    return [dict(id=row['id'], pred=[LABEL_IDS[int(j)] for j in np.flatnonzero(predicted[i])])
            for i, row in enumerate(records)]


def validate_predictions(rows, expected):
    if not isinstance(rows, list) or len(rows) != len(expected):
        raise ValueError('Prediction count mismatch.')
    ids = []
    for row in rows:
        if set(row) != {'id', 'pred'} or not isinstance(row['id'], str) or not isinstance(row['pred'], list):
            raise ValueError('Invalid prediction record.')
        if any(type(label) is not int or label not in LABEL_TO_INDEX for label in row['pred']):
            raise ValueError('Invalid statute ID.')
        if len(set(row['pred'])) != len(row['pred']):
            raise ValueError('Repeated statute ID in a prediction.')
        ids.append(row['id'])
    if len(set(ids)) != len(ids) or set(ids) != {row['id'] for row in expected}:
        raise ValueError('Prediction IDs do not match the official cases.')


def run_professor_evaluator(prediction_path, out):
    rows = read_json(prediction_path)
    dev_dimension = LABEL_METADATA['dev_evaluator_dimension']
    if any(label >= dev_dimension for row in rows for label in row['pred']):
        message = ('The supplied evaluator derives its dimension from dev gold labels and cannot score '
                   'these valid training-label IDs. Full predictions were preserved; see label_metadata.json.')
        write_json(out / 'prof_evaluator_status.json', dict(scored=False, reason=message))
        print(message, flush=True)
        return
    command = [sys.executable, str(EVALUATOR_PATH), '--truth', str(ASSET_DIR / 'dev_truth.json'),
               '--prediction', str(prediction_path)]
    result = subprocess.run(command, capture_output=True, text=True)
    print(result.stdout, flush=True)
    if result.stderr: print(result.stderr, flush=True)
    (out / 'prof_evaluation.txt').write_text(result.stdout + '\n' + result.stderr, encoding='utf-8')
    write_json(out / 'evaluator_provenance.json', dict(
        script_sha256=digest_file(EVALUATOR_PATH), truth_sha256=digest_file(ASSET_DIR / 'dev_truth.json'),
        prediction_sha256=digest_file(prediction_path), returncode=result.returncode))
    result.check_returncode()


In [22]:
EVALUATOR_PATH = ASSET_DIR / "evaluate_phase5.py"
EVALUATOR_SOURCE = '#!/usr/bin/env python3\n"""Local evaluator for Phase 5.\n\nTruth JSON: a list of objects, [{"id":"...", "gold":[0,2,5,...]}, ...]\nPrediction JSON: [{"id":"...", "pred":[0,2,5,...]}, ...]\n"""\nimport argparse,json,sys\nimport numpy as np\nfrom sklearn.metrics import accuracy_score, f1_score\n\ndef read_truth(path):\n    try:\n        x=json.load(open(path,encoding="utf-8"))\n    except json.JSONDecodeError as e:\n        raise ValueError(f"Truth must be a JSON file containing a list of records: {e}")\n    if not isinstance(x,list) or not x: raise ValueError(\'Truth must be a non-empty JSON list.\')\n    out={}; max_label=-1\n    for n,r in enumerate(x):\n        if not isinstance(r,dict) or \'id\' not in r or \'gold\' not in r: raise ValueError(f\'Truth item {n} must contain id and gold.\')\n        if not isinstance(r[\'id\'],str) or not r[\'id\'].strip() or not isinstance(r[\'gold\'],list): raise ValueError(f\'Invalid truth item {n}.\')\n        if r[\'id\'] in out: raise ValueError(f\'Duplicate truth ID: {r["id"]}\')\n        seen=set()\n        for label in r[\'gold\']:\n            if isinstance(label,bool) or not isinstance(label,int) or label<0 or label in seen: raise ValueError(f\'Invalid/duplicate label in truth item {n}.\')\n            seen.add(label); max_label=max(max_label,label)\n        out[r[\'id\']]=r[\'gold\']\n    return out,max_label+1\n\ndef read_pred(path,nlab):\n    x=json.load(open(path,encoding=\'utf-8\'))\n    if not isinstance(x,list) or not x: raise ValueError(\'Prediction must be a non-empty JSON list.\')\n    out={}\n    for n,r in enumerate(x):\n        if not isinstance(r,dict) or \'id\' not in r or \'pred\' not in r: raise ValueError(f\'Prediction item {n} must contain id and pred.\')\n        if not isinstance(r[\'id\'],str) or not isinstance(r[\'pred\'],list) or r[\'id\'] in out: raise ValueError(f\'Invalid or duplicate prediction item {n}.\')\n        seen=set()\n        for x in r[\'pred\']:\n            if isinstance(x,bool) or not isinstance(x,int) or x<0 or x>=nlab or x in seen: raise ValueError(f\'Invalid/duplicate label in prediction item {n}.\')\n            seen.add(x)\n        out[r[\'id\']]=r[\'pred\']\n    return out\n\ndef main():\n    ap=argparse.ArgumentParser(description=\'Evaluate Phase 5 multilabel predictions on your own held-out split.\')\n    ap.add_argument(\'--truth\',required=True); ap.add_argument(\'--prediction\',required=True); a=ap.parse_args()\n    t,n=read_truth(a.truth); p=read_pred(a.prediction,n)\n    if set(t)!=set(p): raise ValueError(f"ID mismatch: missing={len(set(t)-set(p))}, extra={len(set(p)-set(t))}")\n    yt=np.zeros((len(t),n),dtype=np.uint8); yp=np.zeros((len(t),n),dtype=np.uint8)\n    for i,k in enumerate(t):\n        for x in t[k]: yt[i,x]=1\n        for x in p[k]: yp[i,x]=1\n    print(f\'Examples: {len(t)}\'); print(f\'Labels: {n}\')\n    print(f\'Macro-F1: {f1_score(yt,yp,average="macro",zero_division=0):.6f}\')\n    micro=f1_score(yt,yp,average=\'micro\',zero_division=0)\n    subset=accuracy_score(yt,yp)\n    print(f\'Micro-F1: {micro:.6f}\')\n    print(f\'Subset Accuracy: {subset:.6f}\')\nif __name__==\'__main__\':\n    try: main()\n    except Exception as e: print(f\'EVALUATION FAILED: {e}\',file=sys.stderr); sys.exit(1)\n'
EVALUATOR_PATH.write_text(EVALUATOR_SOURCE, encoding="utf-8")
print("Evaluator:", EVALUATOR_PATH)


Evaluator: /kaggle/working/phase5_lsi/assets/evaluate_phase5.py


In [23]:
def sync_cuda():
    if torch.cuda.is_available():
        for index in range(torch.cuda.device_count()): torch.cuda.synchronize(index)


def autocast_context(device):
    return torch.autocast('cuda', dtype=torch.float16) if str(device).startswith('cuda') else contextlib.nullcontext()


def parallel_model(model):
    return nn.DataParallel(model) if torch.cuda.device_count() > 1 and next(model.parameters()).is_cuda else model


def model_inputs(batch, device):
    return {key: batch[key].to(device) for key in ('ids', 'mask', 'positions')}


def make_optimizer(model, cfg):
    groups = []
    for backbone in (True, False):
        for decay in (True, False):
            params = [p for name, p in model.named_parameters() if p.requires_grad
                      and name.startswith('encoder.') == backbone and (p.ndim > 1) == decay]
            if params:
                lr = cfg['adapter_lr'] if backbone and model.method in ('lorma', 'adapters') else cfg['backbone_lr'] if backbone else cfg['head_lr']
                groups.append(dict(params=params, lr=lr, weight_decay=cfg['weight_decay'] if decay else 0.0))
    return torch.optim.AdamW(groups)


def make_scheduler(optimizer, cfg, total):
    warmup = max(1, int(total * cfg['warmup_fraction']))
    def factor(step):
        if step < warmup: return (step + 1) / warmup
        progress = min(1.0, (step - warmup) / max(1, total - warmup))
        return cfg['min_lr_ratio'] + (1 - cfg['min_lr_ratio']) * (1 + math.cos(math.pi * progress)) / 2
    return torch.optim.lr_scheduler.LambdaLR(optimizer, factor)


def save_torch(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    torch.save(payload, temporary)
    temporary.replace(path)


def rng_state():
    return dict(python=random.getstate(), numpy=np.random.get_state(), torch=torch.get_rng_state(),
                cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [])


def restore_rng(state):
    random.setstate(state['python'])
    np.random.set_state(state['numpy'])
    torch.set_rng_state(state['torch'])
    if torch.cuda.is_available() and state['cuda']:
        if len(state['cuda']) != torch.cuda.device_count():
            raise ValueError('Resume on the same GPU count.')
        torch.cuda.set_rng_state_all(state['cuda'])


@torch.no_grad()
def predict_logits(model, dataset, batch_size, device, description='Evaluate', attention_mode=None):
    model.eval()
    net = parallel_model(model)
    output = []
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_cases,
                        num_workers=0, generator=torch.Generator().manual_seed(0))
    for batch in tqdm(loader, desc=description, leave=False):
        with autocast_context(device):
            logits = net(**model_inputs(batch, device), attention_mode=attention_mode)
        if not torch.isfinite(logits).all().item():
            raise FloatingPointError('Non-finite inference logits.')
        output.append(logits.float().cpu())
    return torch.cat(output)


def register(name, model_name, method, **overrides):
    if not re.fullmatch(r'[a-z0-9_]+', name):
        raise ValueError('Use a plain unique run name.')
    cfg = dict(DEFAULT, **overrides)
    if not 0 <= cfg['min_epochs'] <= cfg['epochs'] or cfg['epochs'] < 1 or cfg['patience'] < 1:
        raise ValueError('Invalid epoch/patience budget.')
    if cfg['effective_batch'] < 1 or cfg['max_consecutive_amp_skips'] < 1:
        raise ValueError('Invalid batch or AMP retry limit.')
    EXPERIMENTS[name] = dict(name=name, model=model_name, method=method, config=cfg)
    return name


def run_folder(name): return OUTPUT_ROOT / 'runs' / name


def total_updates(cfg): return math.ceil(len(CASE_INDEX['train']) / cfg['effective_batch']) * cfg['epochs']


def hardware_signature(spec):
    return dict(version=VERSION, spec=spec, checkpoint=BASES[spec['model']]['checkpoint_sha256'],
                labels=LABEL_IDS, train_examples=len(CASE_INDEX['train']), torch=torch.__version__,
                devices=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])


def capacity_check(name):
    if not torch.cuda.is_available():
        raise RuntimeError('Enable the Kaggle GPU for capacity and training.')
    spec = EXPERIMENTS[name]
    cfg = spec['config']
    path = run_folder(name) / 'hardware.json'
    signature = hardware_signature(spec)
    if path.is_file():
        saved = read_json(path)
        if saved['signature'] != signature:
            raise ValueError('GPU/settings changed. Restore the original setup or use a new run name.')
        print('Saved capacity selection:', saved['selected'], flush=True)
        return saved['selected']
    results = []
    for checkpointing in (True, False):
        for micro in (1, 2, 4, 8):
            batch_size = micro * torch.cuda.device_count()
            if batch_size > cfg['effective_batch'] or cfg['effective_batch'] % batch_size:
                continue
            trial = dict(cfg, micro_batch_per_gpu=micro, activation_checkpointing=checkpointing)
            model = net = optimizer = scaler = x = y = loss = norm = None
            print(f'Testing {micro} cases/GPU; checkpointing={checkpointing}', flush=True)
            try:
                gc.collect()
                torch.cuda.empty_cache()
                seed_everything(cfg['seed'])
                model = make_model(spec['model'], spec['method'], trial, total_updates(cfg), 'cuda')
                model.train()
                net = parallel_model(model)
                optimizer = make_optimizer(model, trial)
                scaler = torch.amp.GradScaler('cuda', init_scale=cfg['amp_initial_scale'])
                chunks, length = max_case_chunks(cfg), cfg['chunk_tokens']
                x = dict(ids=torch.randint(4, BASES[spec['model']]['config']['vocab_size'],
                         (batch_size, chunks, length), device='cuda'),
                         mask=torch.ones(batch_size, chunks, length, dtype=torch.bool, device='cuda'),
                         positions=torch.linspace(0, 1, chunks, device='cuda')[None, :].expand(batch_size, -1))
                y = (torch.arange(batch_size * len(LABEL_IDS), device='cuda').reshape(batch_size, -1) % 3 == 0).float()
                for gpu in range(torch.cuda.device_count()): torch.cuda.reset_peak_memory_stats(gpu)
                accumulation = cfg['effective_batch'] // batch_size
                durations, successes, retries, consecutive = [], 0, 0, 0
                while successes < 4:
                    optimizer.zero_grad(set_to_none=True)
                    sync_cuda()
                    started = time.perf_counter()
                    for _ in range(accumulation):
                        with autocast_context('cuda'):
                            logits = net(**x)
                            losses = supervised_loss(logits, y, cfg)
                            if spec['method'] == 'kd':
                                losses = (1 - cfg['kd_alpha']) * losses + cfg['kd_alpha'] * distillation_loss(logits, torch.zeros_like(logits), cfg['kd_temperature'])
                            loss = losses.mean() / accumulation
                        if not torch.isfinite(loss).item(): raise FloatingPointError('Non-finite capacity forward loss.')
                        scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                    norm = nn.utils.clip_grad_norm_(model.parameters(), cfg['clip'])
                    if not torch.isfinite(norm).item():
                        retries += 1
                        consecutive += 1
                        scaler.update(new_scale=scaler.get_scale() / 2)
                        optimizer.zero_grad(set_to_none=True)
                        print('Capacity AMP retry:', consecutive, '| scale:', scaler.get_scale(), flush=True)
                        if consecutive >= cfg['max_consecutive_amp_skips']:
                            raise FloatingPointError('Repeated non-finite capacity gradients.')
                        continue
                    scaler.step(optimizer)
                    scaler.update()
                    sync_cuda()
                    durations.append(time.perf_counter() - started)
                    successes += 1
                    consecutive = 0
                memory = []
                for gpu in range(torch.cuda.device_count()):
                    free, total = torch.cuda.mem_get_info(gpu)
                    peak = torch.cuda.max_memory_reserved(gpu)
                    memory.append(dict(gpu=gpu, peak_reserved_gib=peak / 2**30,
                        peak_tensor_gib=torch.cuda.max_memory_allocated(gpu) / 2**30,
                        headroom_gib=min(free, total - peak) / 2**30))
                row = dict(micro_batch_per_gpu=micro, activation_checkpointing=checkpointing,
                    seconds_per_update=float(np.median(durations[1:])), amp_retries=retries, memory=memory,
                    passed=all(m['headroom_gib'] >= cfg['capacity_headroom_gib'] for m in memory))
            except (torch.cuda.OutOfMemoryError, FloatingPointError) as error:
                row = dict(micro_batch_per_gpu=micro, activation_checkpointing=checkpointing, passed=False, error=str(error))
            finally:
                model = net = optimizer = scaler = x = y = loss = norm = logits = losses = None
                gc.collect()
                torch.cuda.empty_cache()
            results.append(row)
            print(row, flush=True)
    passing = [row for row in results if row['passed']]
    if not passing: raise RuntimeError('No capacity candidate passed. Inspect the failures before training.')
    selected = min(passing, key=lambda row: row['seconds_per_update'])
    write_json(path, dict(signature=signature, selected=selected, results=results))
    print('Selected:', selected, flush=True)
    print('Training-only minutes/epoch:', selected['seconds_per_update'] * math.ceil(len(CASE_INDEX['train']) / cfg['effective_batch']) / 60, flush=True)
    return selected


def resolved_spec(name):
    spec = copy.deepcopy(EXPERIMENTS[name])
    hardware = read_json(run_folder(name) / 'hardware.json')
    if hardware['signature'] != hardware_signature(spec): raise ValueError('Capacity signature mismatch.')
    for key in ('micro_batch_per_gpu', 'activation_checkpointing'):
        spec['config'][key] = hardware['selected'][key]
    return spec


def training_signature(spec, teacher):
    return dict(version=VERSION, spec=spec, label_ids=LABEL_IDS,
        base_checkpoint=BASES[spec['model']]['checkpoint_sha256'], tokenizer=BASES[spec['model']]['tokenizer_sha256'],
        cases={split: object_hash(CASE_INDEX[split]) for split in ('train', 'dev')},
        tokens={split: digest_file(TOKEN_FOLDERS[(spec['model'], split)] / 'tokens.bin') for split in ('train', 'dev')},
        teacher=teacher, torch=torch.__version__)


def log_metric(folder, row):
    with (folder / 'metrics.jsonl').open('a', encoding='utf-8') as handle:
        handle.write(json.dumps(row) + '\n')
    print(row, flush=True)


def update_early_stopping(state, score, cfg, count_patience):
    improved = score > state['best_macro_f1'] + 1e-8
    if improved:
        state['best_macro_f1'], state['bad_epochs'] = score, 0
    elif count_patience:
        state['bad_epochs'] = 0 if state['epoch'] <= cfg['min_epochs'] else state['bad_epochs'] + 1
    return improved


In [24]:
def train_run(name, additional_steps=None, teacher_name=None):
    spec = resolved_spec(name)
    cfg = spec['config']
    folder = run_folder(name)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    teacher_reference = None
    teacher_logits = None
    if spec['method'] == 'kd':
        if teacher_name is None:
            raise ValueError('Select and cache the teacher before distillation.')
        teacher_logits, teacher_reference = cache_teacher(teacher_name)

    signature = training_signature(spec, teacher_reference)
    if (folder / 'signature.json').is_file() and read_json(folder / 'signature.json') != signature:
        raise ValueError('Run inputs or settings changed. Use the original settings to resume.')
    if additional_steps is not None and additional_steps < 1:
        raise ValueError('additional_steps must be positive')
    write_json(folder / 'signature.json', signature)
    write_json(folder / 'config.json', spec)
    seed_everything(cfg['seed'])
    train_data = CaseDataset(spec['model'], 'train', cfg)
    dev_data = CaseDataset(spec['model'], 'dev', cfg)
    total = total_updates(cfg)
    model = make_model(spec['model'], spec['method'], cfg, total, device)
    parameters = parameter_report(model)
    print('Parameters:', parameters, flush=True)
    if teacher_reference and parameters['total'] >= teacher_reference['parameters']:
        raise ValueError('The distilled student must be smaller than its teacher.')

    net = parallel_model(model)
    optimizer = make_optimizer(model, cfg)
    scheduler = make_scheduler(optimizer, cfg, total)
    scaler = torch.amp.GradScaler('cuda', enabled=device == 'cuda', init_scale=cfg['amp_initial_scale'])
    state = dict(epoch=0, position=0, step=0, best_macro_f1=-1.0,
                 bad_epochs=0, complete=False, skipped=0, consecutive_amp_skips=0,
                 epoch_loss_sum=0.0, epoch_examples=0)
    latest = folder / 'latest.pt'
    if latest.is_file():
        saved = torch.load(latest, map_location='cpu', weights_only=False)
        if saved['signature'] != signature:
            raise ValueError('Resume signature mismatch')
        model.load_state_dict(saved['model'], strict=True)
        optimizer.load_state_dict(saved['optimizer'])
        scheduler.load_state_dict(saved['scheduler'])
        scaler.load_state_dict(saved['scaler'])
        state = saved['progress']
        restore_rng(saved['rng'])
        del saved
        print('Resumed:', state, flush=True)

    start_step = state['step']
    started = time.monotonic()
    batch_size = cfg['micro_batch_per_gpu'] * max(1, torch.cuda.device_count())
    pos_weight = training_positive_weights(cfg, device)

    def save_latest():
        save_torch(latest, dict(
            signature=signature,
            model={k: v.detach().cpu() for k, v in model.state_dict().items()},
            optimizer=optimizer.state_dict(), scheduler=scheduler.state_dict(),
            scaler=scaler.state_dict(),
            progress=copy.deepcopy(state), rng=rng_state(),
        ))
        write_json(folder / 'progress.json', state)

    def validate(count_patience):
        logits = predict_logits(model, dev_data, batch_size, device)
        probabilities = logits.sigmoid().numpy()
        selected, _ = select_global_threshold(DEV_TARGETS, probabilities, cfg['threshold_grid'])
        threshold = selected['threshold']
        metrics = metric_report(DEV_TARGETS, probabilities, threshold)
        improved = update_early_stopping(state, metrics['macro_f1'], cfg, count_patience)
        row = dict(step=state['step'], epoch=state['epoch'],
                   validation_macro_f1=metrics['macro_f1'], validation_micro_f1=metrics['micro_f1'],
                   threshold=threshold, best_macro_f1=state['best_macro_f1'],
                   early_stopping_active=state['epoch'] > cfg['min_epochs'],
                   bad_epochs=state['bad_epochs'])
        if count_patience and state['epoch_examples']:
            row['epoch_train_loss'] = state['epoch_loss_sum'] / state['epoch_examples']
        log_metric(folder, row)
        if improved:
            save_torch(folder / 'best.pt', dict(
                signature=signature,
                model={k: v.detach().cpu() for k, v in model.state_dict().items()},
                step=state['step'], epoch=state['epoch'], metrics=metrics, threshold=threshold, parameters=parameters,
            ))
            write_json(folder / 'best_dev_probabilities.json', [
                dict(id=r['id'], gold=r['labels'], probabilities=p.tolist())
                for r, p in zip(CASE_INDEX['dev'], probabilities)
            ])
        model.train()

    try:
        if state['step'] == 0 and not (folder / 'best.pt').is_file():
            validate(False)
            save_latest()
        while not state['complete'] and state['epoch'] < cfg['epochs']:
            order = np.random.default_rng(cfg['seed'] + state['epoch']).permutation(len(train_data))
            model.train()
            while state['position'] < len(order):
                indices = order[state['position']:state['position'] + cfg['effective_batch']]
                optimizer.zero_grad(set_to_none=True)
                loss_sum = 0.0
                for begin in range(0, len(indices), batch_size):
                    selected = indices[begin:begin + batch_size]
                    batch = collate_cases([train_data[int(i)] for i in selected])
                    targets = batch['labels'].to(device)
                    with autocast_context(device):
                        logits = net(**model_inputs(batch, device))
                        ce = supervised_loss(logits, targets, cfg, pos_weight)
                        losses = ce
                        if teacher_logits is not None:
                            kl = distillation_loss(logits, teacher_logits[selected].to(device), cfg['kd_temperature'])
                            losses = (1 - cfg['kd_alpha']) * ce + cfg['kd_alpha'] * kl
                        loss = losses.sum() / len(indices)
                    if not torch.isfinite(loss).item():
                        raise FloatingPointError('Non-finite forward loss. Inspect the run before continuing.')
                    scaler.scale(loss).backward()
                    loss_sum += float(loss.detach())
                scaler.unscale_(optimizer)
                norm = nn.utils.clip_grad_norm_(model.parameters(), cfg['clip'])
                before = scaler.get_scale()
                if not torch.isfinite(norm).item():
                    if device != 'cuda':
                        raise FloatingPointError('Non-finite CPU gradients; optimizer was not stepped.')
                    scaler.update(new_scale=before / 2)
                    skipped = True
                else:
                    scaler.step(optimizer)
                    scaler.update()
                    skipped = scaler.get_scale() < before
                if skipped:
                    state['skipped'] += 1
                    state['consecutive_amp_skips'] += 1
                    log_metric(folder, dict(event='amp_retry', step=state['step'], epoch=state['epoch'],
                        total_skips=state['skipped'], consecutive_skips=state['consecutive_amp_skips'],
                        scale_before=before, scale_after=scaler.get_scale()))
                    if state['consecutive_amp_skips'] >= cfg['max_consecutive_amp_skips']:
                        save_latest()
                        raise FloatingPointError('Repeated AMP failures on the same batch. Checkpoint saved; inspect retry logs.')
                    continue
                state['consecutive_amp_skips'] = 0
                state['step'] += 1
                scheduler.step()
                state['position'] += len(indices)
                state['epoch_loss_sum'] += loss_sum * len(indices)
                state['epoch_examples'] += len(indices)
                if state['step'] == start_step + 1 or state['step'] % cfg['log_every'] == 0:
                    log_metric(folder, dict(step=state['step'], epoch=state['epoch'], loss=loss_sum,
                        lr=optimizer.param_groups[0]['lr'], gradient_norm=float(norm),
                        skipped=state['skipped'], elapsed_minutes=(time.monotonic() - started) / 60))
                if state['step'] % cfg['save_every'] == 0:
                    save_latest()
                pause = ((additional_steps is not None and state['step'] - start_step >= additional_steps)
                         or time.monotonic() - started >= cfg['max_hours'] * 3600)
                if pause and state['position'] < len(order):
                    validate(False)
                    save_latest()
                    report = dict(name=name, **state, session_seconds=time.monotonic() - started,
                                  session_paused=True)
                    log_metric(folder, dict(event='session_end', session_seconds=report['session_seconds'],
                                            start_step=start_step, end_step=state['step'], complete=False))
                    print(report, flush=True)
                    return report
            state['epoch'] += 1
            state['position'] = 0
            validate(True)
            state['epoch_loss_sum'] = 0.0
            state['epoch_examples'] = 0
            exhausted = state['epoch'] >= cfg['epochs']
            plateau = state['epoch'] > cfg['min_epochs'] and state['bad_epochs'] >= cfg['patience']
            state['complete'] = exhausted or plateau
            if state['complete']:
                state['stop_reason'] = 'epochs' if exhausted else 'early_stopping'
            save_latest()
            if additional_steps is not None and state['step'] - start_step >= additional_steps:
                break
            if time.monotonic() - started >= cfg['max_hours'] * 3600:
                break
        report = dict(name=name, **state, parameters=parameters,
                      session_seconds=time.monotonic() - started)
        write_json(folder / 'summary.json', report)
        log_metric(folder, dict(event='session_end', session_seconds=report['session_seconds'],
                                start_step=start_step, end_step=state['step'], complete=state['complete']))
        print(report, flush=True)
        return report
    finally:
        optimizer.zero_grad(set_to_none=True)
        del net, model, optimizer, scheduler, scaler
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


In [25]:
def load_trained(name, device):
    folder = run_folder(name)
    spec = read_json(folder / 'config.json')
    saved = torch.load(folder / 'best.pt', map_location='cpu', weights_only=False)
    if saved['signature'] != read_json(folder / 'signature.json'):
        raise ValueError('Saved checkpoint/signature mismatch.')
    if saved['signature']['base_checkpoint'] != BASES[spec['model']]['checkpoint_sha256']:
        raise ValueError('Different Phase 1 checkpoint attached.')
    if saved['signature']['label_ids'] != LABEL_IDS:
        raise ValueError('Statute label IDs changed.')
    if saved['signature']['cases']['dev'] != object_hash(CASE_INDEX['dev']):
        raise ValueError('Development cases changed.')
    model = make_model(spec['model'], spec['method'], spec['config'], total_updates(spec['config']), device)
    model.load_state_dict(saved['model'], strict=True)
    return model.eval(), spec, saved


def validate_run(name):
    folder = run_folder(name)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model, spec, saved = load_trained(name, device)
    cfg = spec['config']
    threshold = saved['threshold']
    parameters = parameter_report(model)
    if torch.cuda.is_available():
        for gpu in range(torch.cuda.device_count()): torch.cuda.reset_peak_memory_stats(gpu)
    sync_cuda()
    started = time.perf_counter()
    try:
        logits = predict_logits(model, CaseDataset(spec['model'], 'dev', cfg),
            cfg['micro_batch_per_gpu'] * max(1, torch.cuda.device_count()), device)
        sync_cuda()
        seconds = time.perf_counter() - started
        memory = [dict(gpu=i, peak_reserved_gib=torch.cuda.max_memory_reserved(i) / 2**30,
                       peak_tensor_gib=torch.cuda.max_memory_allocated(i) / 2**30)
                  for i in range(torch.cuda.device_count())] if torch.cuda.is_available() else []
    finally:
        del model
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    probabilities = logits.sigmoid().numpy()
    report = metric_report(DEV_TARGETS, probabilities, threshold)
    report.update(threshold=threshold, saved_best_macro_f1=saved['metrics']['macro_f1'],
        inference_seconds=seconds, documents_per_second=len(CASE_INDEX['dev']) / max(seconds, 1e-9),
        parameters=parameters, inference_memory=memory, device=device,
        checkpoint_bytes=(folder / 'best.pt').stat().st_size)
    if abs(report['macro_f1'] - saved['metrics']['macro_f1']) > 1e-5:
        print('Validation differs from the saved score. Check numerical/device/input differences.', flush=True)
    out = folder / 'validation'
    out.mkdir(parents=True, exist_ok=True)
    rows = predictions_json(CASE_INDEX['dev'], probabilities, threshold)
    validate_predictions(rows, CASE_INDEX['dev'])
    write_json(out / 'prediction.json', rows)
    write_json(out / 'ids.json', [row['id'] for row in CASE_INDEX['dev']])
    np.savez_compressed(out / 'probabilities.npz', probabilities=probabilities,
                        truth=DEV_TARGETS.astype(np.uint8), label_ids=np.asarray(LABEL_IDS, dtype=np.int64))
    write_json(out / 'metrics.json', report)
    write_json(out / 'provenance.json', dict(checkpoint=digest_file(folder / 'best.pt'),
        probabilities=digest_file(out / 'probabilities.npz'), cases=object_hash(CASE_INDEX['dev']),
        threshold=threshold, label_ids=LABEL_IDS))
    run_professor_evaluator(out / 'prediction.json', out)
    print(json.dumps(report, indent=2), flush=True)
    del saved
    return report


def export_test(name, threshold=None):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model, spec, saved = load_trained(name, device)
    cfg = spec['config']
    threshold = saved['threshold'] if threshold is None else threshold
    threshold_value = np.asarray(threshold).tolist()
    signature = dict(checkpoint=digest_file(run_folder(name) / 'best.pt'),
        cases=object_hash(CASE_INDEX['test']), label_ids=LABEL_IDS,
        tokens=digest_file(TOKEN_FOLDERS[(spec['model'], 'test')] / 'tokens.bin'), threshold=threshold_value)
    out = run_folder(name) / 'test' / object_hash(signature)[:12]
    out.mkdir(parents=True, exist_ok=True)
    try:
        logits = predict_logits(model, CaseDataset(spec['model'], 'test', cfg),
            cfg['micro_batch_per_gpu'] * max(1, torch.cuda.device_count()), device, 'Test cases')
    finally:
        del model, saved
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    probabilities = logits.sigmoid().numpy()
    rows = predictions_json(CASE_INDEX['test'], probabilities, threshold)
    validate_predictions(rows, CASE_INDEX['test'])
    write_json(out / 'prediction.json', rows)
    write_json(out / 'ids.json', [row['id'] for row in CASE_INDEX['test']])
    np.savez_compressed(out / 'probabilities.npz', probabilities=probabilities, label_ids=np.asarray(LABEL_IDS))
    write_json(out / 'generation.json', signature)
    archive = out / f'{name}_submission.zip'
    with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as handle:
        handle.write(out / 'prediction.json', 'prediction.json')
    with zipfile.ZipFile(archive) as handle:
        if handle.namelist() != ['prediction.json']: raise ValueError('Wrong submission ZIP layout.')
        validate_predictions(json.loads(handle.read('prediction.json')), CASE_INDEX['test'])
    payload = base64.b64encode(archive.read_bytes()).decode()
    display(HTML(f'<a download="{html.escape(archive.name)}" href="data:application/zip;base64,{payload}">Download {html.escape(archive.name)}</a>'))
    print('Submission:', archive, flush=True)
    return archive


def completed_runs():
    result = []
    for path in (OUTPUT_ROOT / 'runs').glob('*/progress.json'):
        if not (path.parent / 'best.pt').is_file(): continue
        spec, progress = read_json(path.parent / 'config.json'), read_json(path)
        result.append(dict(name=spec['name'], model=spec['model'], method=spec['method'],
            complete=progress['complete'], epoch=progress['epoch'], best_macro_f1=progress['best_macro_f1']))
    return sorted(result, key=lambda row: (-row['best_macro_f1'], row['name']))


def choose_teacher():
    rows = [row for row in completed_runs() if row['method'] in ('full', 'lorma', 'adapters')]
    if TEACHER_RUN_NAME is not None:
        rows = [row for row in rows if row['name'] == TEACHER_RUN_NAME]
    if not rows: raise ValueError('Train at least one Phase 5 supervised model before distillation.')
    print('Teacher selected by Phase 5 development Macro-F1:', rows[0], flush=True)
    return rows[0]['name']


def cache_teacher(name):
    folder = run_folder(name)
    spec = read_json(folder / 'config.json')
    signature = read_json(folder / 'signature.json')
    if signature['cases']['train'] != object_hash(CASE_INDEX['train']) or signature['label_ids'] != LABEL_IDS:
        raise ValueError('Teacher training cases or labels changed.')
    reference = dict(name=name, checkpoint=digest_file(folder / 'best.pt'),
                     cases=object_hash(CASE_INDEX['train']), label_ids=LABEL_IDS,
                     tokens=digest_file(TOKEN_FOLDERS[(spec['model'], 'train')] / 'tokens.bin'))
    cache = OUTPUT_ROOT / 'teacher_cache' / object_hash(reference)[:16]
    cache.mkdir(parents=True, exist_ok=True)
    if (cache / 'logits.pt').is_file():
        saved = torch.load(cache / 'logits.pt', map_location='cpu', weights_only=False)
        if saved['reference'] != reference or saved['logits'].shape != (len(CASE_INDEX['train']), len(LABEL_IDS)):
            raise ValueError('Teacher cache mismatch.')
        if not torch.isfinite(saved['logits']).all().item(): raise FloatingPointError('Non-finite teacher logits.')
        return saved['logits'], dict(reference, parameters=saved['parameters'])
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model, spec, saved = load_trained(name, device)
    try:
        cfg = spec['config']
        count = parameter_report(model)['total']
        logits = predict_logits(model, CaseDataset(spec['model'], 'train', cfg),
            cfg['micro_batch_per_gpu'] * max(1, torch.cuda.device_count()), device, 'Teacher training logits')
        save_torch(cache / 'logits.pt', dict(reference=reference, logits=logits, parameters=count))
    finally:
        del model, saved
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    return logits, dict(reference, parameters=count)


def small_sample_check(name, updates=80, examples=16):
    spec = resolved_spec(name)
    cfg = dict(spec['config'], dropout=0.0)
    seed_everything(cfg['seed'])
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    data = CaseDataset(spec['model'], 'train', cfg)
    selected = np.random.default_rng(cfg['seed']).choice(len(data), min(examples, len(data)), replace=False).tolist()
    model = make_model(spec['model'], spec['method'], cfg, updates, device)
    net = parallel_model(model)
    optimizer = make_optimizer(model, cfg)
    scaler = torch.amp.GradScaler('cuda', enabled=device == 'cuda', init_scale=cfg['amp_initial_scale'])
    batch_size = cfg['micro_batch_per_gpu'] * max(1, torch.cuda.device_count())
    report, completed, retries = [], 0, 0
    try:
        while completed < updates:
            model.train()
            optimizer.zero_grad(set_to_none=True)
            total = 0.0
            for start in range(0, len(selected), batch_size):
                indices = selected[start:start + batch_size]
                batch = collate_cases([data[i] for i in indices])
                with autocast_context(device):
                    logits = net(**model_inputs(batch, device))
                    loss = supervised_loss(logits, batch['labels'].to(device), cfg).sum() / len(selected)
                if not torch.isfinite(loss).item(): raise FloatingPointError('Non-finite tiny-sample loss.')
                scaler.scale(loss).backward()
                total += float(loss.detach())
            scaler.unscale_(optimizer)
            norm = nn.utils.clip_grad_norm_(model.parameters(), cfg['clip'])
            before = scaler.get_scale()
            if not torch.isfinite(norm).item():
                if device != 'cuda':
                    raise FloatingPointError('Non-finite tiny-sample gradients; optimizer was not stepped.')
                scaler.update(new_scale=before / 2)
            else:
                scaler.step(optimizer)
                scaler.update()
            if scaler.get_scale() < before:
                retries += 1
                if retries >= cfg['max_consecutive_amp_skips']: raise FloatingPointError('Repeated tiny-sample AMP failures.')
                continue
            if not torch.isfinite(norm).item(): raise FloatingPointError('Non-finite tiny-sample gradients.')
            retries = 0
            completed += 1
            if completed == 1 or completed % 20 == 0 or completed == updates:
                logits = predict_logits(model, torch.utils.data.Subset(data, selected), batch_size, device, 'Tiny-sample check')
                y = TRAIN_TARGETS[selected]
                best, _ = select_global_threshold(y, logits.sigmoid().numpy(), cfg['threshold_grid'])
                row = dict(update=completed, loss=total, **best)
                report.append(row)
                print(row, flush=True)
    finally:
        del model, net, optimizer, scaler
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    write_json(run_folder(name) / 'small_sample_check.json', dict(examples=len(selected), case_ids=[CASE_INDEX['train'][i]['id'] for i in selected], history=report))
    return report


## 8. Analysis helpers

Analysis reports per-statute scores, rare-label groups, common misses, performance by document length and label count, learning curves, and resource use. Optional representation analysis varies the inference windows while keeping the trained checkpoint and threshold fixed.

In [26]:
def read_validation_arrays(name):
    folder = run_folder(name) / 'validation'
    provenance = read_json(folder / 'provenance.json')
    if provenance['checkpoint'] != digest_file(run_folder(name) / 'best.pt'):
        raise ValueError('Validation belongs to an older best checkpoint. Rerun validate_run first.')
    if provenance['cases'] != object_hash(CASE_INDEX['dev']): raise ValueError('Development case order changed.')
    if provenance['probabilities'] != digest_file(folder / 'probabilities.npz'):
        raise ValueError('Saved validation probabilities changed.')
    with np.load(folder / 'probabilities.npz') as saved:
        probabilities, truth = saved['probabilities'].copy(), saved['truth'].copy()
        if saved['label_ids'].tolist() != LABEL_IDS: raise ValueError('Saved label IDs differ.')
    return truth, probabilities, read_json(folder / 'metrics.json')


def per_label_frame(truth, predicted):
    truth, predicted = truth.astype(bool), predicted.astype(bool)
    tp = (truth & predicted).sum(0)
    fp = (~truth & predicted).sum(0)
    fn = (truth & ~predicted).sum(0)
    counts = TRAIN_TARGETS.sum(0).astype(int)
    frame = pd.DataFrame(dict(statute_id=LABEL_IDS, train_support=counts,
        dev_support=truth.sum(0), predicted_support=predicted.sum(0),
        tp=tp, fp=fp, fn=fn, precision=tp / np.maximum(tp + fp, 1),
        recall=tp / np.maximum(tp + fn, 1), f1=2 * tp / np.maximum(2 * tp + fp + fn, 1)))
    frame['frequency_group'] = pd.cut(frame.train_support, bins=[-1, 0, 10, 100, np.inf],
                                     labels=['unseen', '1_to_10', '11_to_100', 'over_100'])
    return frame


def threshold_analysis(truth, probabilities, grid, folds=5):
    global_best, grid_rows = select_global_threshold(truth, probabilities, grid)
    def label_thresholds(y, p, global_threshold):
        thresholds = np.full(len(LABEL_IDS), global_threshold, dtype=float)
        for index in range(len(LABEL_IDS)):
            positives = int(y[:, index].sum())
            negatives = len(y) - positives
            if min(positives, negatives) < 5: continue
            candidates = []
            for threshold in grid:
                predicted = p[:, index] >= threshold
                tp = np.logical_and(y[:, index], predicted).sum()
                fp = np.logical_and(~y[:, index].astype(bool), predicted).sum()
                fn = np.logical_and(y[:, index], ~predicted).sum()
                candidates.append((2 * tp / max(2 * tp + fp + fn, 1), -abs(threshold - global_threshold), threshold))
            selected = max(candidates)[2]
            strength = min(positives, negatives) / (min(positives, negatives) + 20)
            thresholds[index] = strength * selected + (1 - strength) * global_threshold
        return thresholds
    thresholds = label_thresholds(truth, probabilities, global_best['threshold'])
    results = [dict(method='global_fit', **global_best),
               dict(method='per_label_fit', **metric_report(truth, probabilities, thresholds))]
    if len(truth) >= 4:
        global_oof = np.zeros_like(truth, dtype=bool)
        per_oof = np.zeros_like(truth, dtype=bool)
        fold_rows = []
        for fold, (fit, held) in enumerate(KFold(min(folds, len(truth)), shuffle=True, random_state=42).split(truth), 1):
            selected, _ = select_global_threshold(truth[fit], probabilities[fit], grid)
            local = label_thresholds(truth[fit], probabilities[fit], selected['threshold'])
            global_oof[held] = decisions(probabilities[held], selected['threshold'])
            per_oof[held] = decisions(probabilities[held], local)
            fold_rows.append(dict(fold=fold, global_threshold=selected['threshold'], held_cases=len(held)))
        results.extend([dict(method='global_threshold_oof', **score_predictions(truth, global_oof)),
                        dict(method='per_label_threshold_oof', **score_predictions(truth, per_oof))])
    else:
        fold_rows = []
    return dict(global_threshold=global_best['threshold'], per_label_thresholds=thresholds.tolist(),
                grid=grid_rows, comparison=results, folds=fold_rows,
                limitation='OOF separates threshold fitting only. Checkpoint selection already used development data; this is not an untouched model evaluation.')


def analyze_run(name):
    truth, probabilities, metrics = read_validation_arrays(name)
    threshold = metrics['threshold']
    predicted = decisions(probabilities, threshold)
    cfg = read_json(run_folder(name) / 'config.json')['config']
    out = run_folder(name) / 'analysis' / datetime.now().strftime('%Y%m%d_%H%M%S_%f')
    out.mkdir(parents=True, exist_ok=False)
    per_label = per_label_frame(truth, predicted)
    per_label.to_csv(out / 'per_label.csv', index=False)
    frequency = per_label.groupby('frequency_group', observed=True).agg(
        labels=('statute_id', 'count'), mean_f1=('f1', 'mean'), mean_precision=('precision', 'mean'),
        mean_recall=('recall', 'mean'), train_support=('train_support', 'sum'), dev_support=('dev_support', 'sum'))
    frequency.to_csv(out / 'frequency_groups.csv')
    thresholds = threshold_analysis(truth, probabilities, cfg['threshold_grid'])
    write_json(out / 'threshold_analysis.json', thresholds)
    pd.DataFrame(thresholds['grid']).to_csv(out / 'global_thresholds.csv', index=False)
    pd.DataFrame(thresholds['comparison']).to_csv(out / 'threshold_comparison.csv', index=False)
    pd.DataFrame(dict(statute_id=LABEL_IDS, threshold=thresholds['per_label_thresholds'])).to_csv(out / 'per_label_thresholds.csv', index=False)
    lengths = read_json(TOKEN_FOLDERS[(read_json(run_folder(name) / 'config.json')['model'], 'dev')] / 'index.json')
    case_rows = []
    for index, (record, length) in enumerate(zip(CASE_INDEX['dev'], lengths)):
        starts = window_starts(length['length'], cfg)
        end = covered = 0
        for start in starts:
            stop = min(length['length'], start + cfg['chunk_tokens'])
            covered += max(0, stop - max(start, end))
            end = max(end, stop)
        tp = int(np.logical_and(truth[index], predicted[index]).sum())
        fp = int(np.logical_and(~truth[index].astype(bool), predicted[index]).sum())
        fn = int(np.logical_and(truth[index], ~predicted[index]).sum())
        case_rows.append(dict(id=record['id'], tokens=length['length'], token_coverage=covered / length['length'],
            chunks=len(starts), gold_count=int(truth[index].sum()), predicted_count=int(predicted[index].sum()),
            tp=tp, fp=fp, fn=fn, exact_match=bool(np.array_equal(truth[index], predicted[index])),
            missed=[LABEL_IDS[j] for j in np.flatnonzero(truth[index] & ~predicted[index])],
            false_positives=[LABEL_IDS[j] for j in np.flatnonzero(~truth[index].astype(bool) & predicted[index])]))
    cases = pd.DataFrame(case_rows)
    cases.to_csv(out / 'case_errors.csv', index=False)
    group_rows = []
    masks = [(f'tokens_{low}_{high}', (cases.tokens > low) & (cases.tokens <= high))
             for low, high in [(0, 512), (512, 2048), (2048, 8192), (8192, np.inf)]]
    masks += [('one_gold_label', cases.gold_count == 1), ('multiple_gold_labels', cases.gold_count > 1),
              ('zero_gold_labels', cases.gold_count == 0)]
    for group, mask in masks:
        positions = np.flatnonzero(mask.to_numpy())
        if not len(positions): continue
        report = score_predictions(truth[positions], predicted[positions])
        supported = truth[positions].sum(0) > 0
        group_rows.append(dict(group=group, cases=len(positions), mean_coverage=float(cases.loc[mask, 'token_coverage'].mean()),
            **report, supported_label_mean_f1=float(per_label_frame(truth[positions], predicted[positions]).loc[supported, 'f1'].mean()) if supported.any() else None))
    pd.DataFrame(group_rows).to_csv(out / 'length_and_cardinality_groups.csv', index=False)
    counts = TRAIN_TARGETS.astype(np.int64).T @ TRAIN_TARGETS.astype(np.int64)
    np.save(out / 'training_label_cooccurrence.npy', counts)
    missed = truth.astype(bool) & ~predicted
    extra = ~truth.astype(bool) & predicted
    replacements = missed.astype(np.int64).T @ extra.astype(np.int64)
    pair_rows = [dict(missed_statute=LABEL_IDS[i], falsely_added_statute=LABEL_IDS[j], documents=int(replacements[i, j]))
                 for i, j in zip(*np.nonzero(replacements)) if i != j]
    pd.DataFrame(sorted(pair_rows, key=lambda row: -row['documents'])[:100]).to_csv(out / 'missed_with_false_positive_pairs.csv', index=False)
    history_path = run_folder(name) / 'metrics.jsonl'
    history = [json.loads(line) for line in history_path.read_text(encoding='utf-8').splitlines() if line.strip()]
    pd.DataFrame(history).to_csv(out / 'training_history.csv', index=False)
    validation = pd.DataFrame([row for row in history if 'validation_macro_f1' in row])
    if not validation.empty:
        figure, axes = plt.subplots(1, 2, figsize=(11, 4))
        axes[0].plot(validation['epoch'], validation['validation_macro_f1'], marker='o')
        axes[0].set(xlabel='Completed epoch', ylabel='Development Macro-F1')
        train_rows = [row for row in history if 'epoch_train_loss' in row]
        if train_rows:
            axes[1].plot([row['epoch'] for row in train_rows], [row['epoch_train_loss'] for row in train_rows])
        axes[1].set(xlabel='Completed epoch', ylabel='Mean training objective')
        figure.tight_layout()
        figure.savefig(out / 'training_curves.png', dpi=160)
        plt.close(figure)
    top = np.argsort(-TRAIN_TARGETS.sum(0))[:min(15, len(LABEL_IDS))]
    figure, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].bar(np.arange(len(per_label)), per_label.sort_values('train_support', ascending=False)['train_support'])
    axes[0].set(xlabel='Statutes ordered by training frequency', ylabel='Training support', yscale='symlog')
    image = axes[1].imshow(counts[np.ix_(top, top)], aspect='auto')
    axes[1].set_xticks(range(len(top)), [LABEL_IDS[i] for i in top], rotation=90)
    axes[1].set_yticks(range(len(top)), [LABEL_IDS[i] for i in top])
    axes[1].set_title('Training co-occurrence; diagonal is support')
    figure.colorbar(image, ax=axes[1])
    figure.tight_layout()
    figure.savefig(out / 'label_distribution_and_cooccurrence.png', dpi=160)
    plt.close(figure)
    hardware = read_json(run_folder(name) / 'hardware.json')
    write_json(out / 'efficiency.json', dict(validation=metrics, capacity=hardware['selected'],
        summed_training_call_seconds=sum(float(row.get('session_seconds', 0)) for row in history if row.get('event') == 'session_end')))
    write_json(out / 'provenance.json', dict(checkpoint=digest_file(run_folder(name) / 'best.pt'),
        validation_probabilities=digest_file(run_folder(name) / 'validation/probabilities.npz'),
        training_labels_sha256=object_hash(TRAIN_TARGETS.astype(int).tolist()), label_ids=LABEL_IDS,
        notes=['Groups describe performance; they do not isolate causal effects.', thresholds['limitation'],
               'Co-occurrence comes only from training labels. No graph model is trained by this analysis.']))
    print(per_label.sort_values('f1').head(15).to_string(index=False), flush=True)
    print('\nThreshold comparison:\n', pd.DataFrame(thresholds['comparison']).to_string(index=False), flush=True)
    print('Analysis saved:', out, flush=True)
    return out


def representation_analysis(name):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model, spec, saved = load_trained(name, device)
    cfg = spec['config']
    threshold = saved['threshold']
    rows = []
    try:
        for representation in ('head', 'tail', 'head_tail', 'hierarchical'):
            variant = dict(cfg, representation=representation)
            logits = predict_logits(model, CaseDataset(spec['model'], 'dev', variant),
                cfg['micro_batch_per_gpu'] * max(1, torch.cuda.device_count()), device, f'Representation: {representation}')
            rows.append(dict(representation=representation, attention='label',
                **metric_report(DEV_TARGETS, logits.sigmoid().numpy(), threshold)))
        logits = predict_logits(model, CaseDataset(spec['model'], 'dev', cfg),
            cfg['micro_batch_per_gpu'] * max(1, torch.cuda.device_count()), device, 'Uniform chunk attention', attention_mode='mean')
        rows.append(dict(representation=cfg['representation'], attention='mean',
            **metric_report(DEV_TARGETS, logits.sigmoid().numpy(), threshold)))
    finally:
        del model, saved
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    out = run_folder(name) / 'analysis' / ('representation_' + datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
    out.mkdir(parents=True)
    pd.DataFrame(rows).to_csv(out / 'comparison.csv', index=False)
    write_json(out / 'notes.json', dict(checkpoint=digest_file(run_folder(name) / 'best.pt'),
        threshold=threshold, interpretation='Inference-time evidence/attention ablation of one trained model. These variants were not independently trained; position and input-distribution changes can affect scores.'))
    print(pd.DataFrame(rows).to_string(index=False), flush=True)
    return out


def compare_distillation(student_name):
    signature = read_json(run_folder(student_name) / 'signature.json')
    teacher_name = signature['teacher']['name']
    teacher_truth, teacher_probability, teacher_metrics = read_validation_arrays(teacher_name)
    student_truth, student_probability, student_metrics = read_validation_arrays(student_name)
    if not np.array_equal(teacher_truth, student_truth): raise ValueError('Teacher/student cases differ.')
    if digest_file(run_folder(teacher_name) / 'best.pt') != signature['teacher']['checkpoint']:
        raise ValueError('The teacher checkpoint changed after student training.')
    teacher_per = per_label_frame(teacher_truth, decisions(teacher_probability, teacher_metrics['threshold']))
    student_per = per_label_frame(student_truth, decisions(student_probability, student_metrics['threshold']))
    frame = teacher_per[['statute_id', 'train_support', 'dev_support', 'f1']].rename(columns={'f1': 'teacher_f1'})
    frame['student_f1'] = student_per['f1']
    frame['change'] = frame['student_f1'] - frame['teacher_f1']
    out = run_folder(student_name) / 'distillation_comparison'
    out.mkdir(parents=True, exist_ok=True)
    frame.to_csv(out / 'per_label.csv', index=False)
    rows = [dict(role=role, name=name, macro_f1=metric['macro_f1'], micro_f1=metric['micro_f1'],
        subset_accuracy=metric['subset_accuracy'], parameters=metric['parameters']['total'],
        trainable_parameters=metric['parameters']['trainable'], inference_seconds=metric['inference_seconds'],
        checkpoint_bytes=metric['checkpoint_bytes']) for role, name, metric in (
            ('teacher', teacher_name, teacher_metrics), ('student', student_name, student_metrics))]
    pd.DataFrame(rows).to_csv(out / 'comparison.csv', index=False)
    write_json(out / 'notes.json', dict(
        interpretation='Teacher versus smaller distilled student. A no-distillation student control is needed to isolate the effect of the KD loss; architecture, initialization, and training also differ.'))
    print(pd.DataFrame(rows).to_string(index=False), flush=True)
    return out


## 9. Checks and saved-run restoration


In [27]:
restore_saved_runs()
check_backbones()
print('Setup complete.')


model1 padding/gradient check passed {'total': 120025828, 'trainable': 120025828, 'adapted_modules': 0}
model2 padding/gradient check passed {'total': 122565860, 'trainable': 122565860, 'adapted_modules': 0}
model3 padding/gradient check passed {'total': 118584036, 'trainable': 118584036, 'adapted_modules': 0}
model4 padding/gradient check passed {'total': 119592292, 'trainable': 119592292, 'adapted_modules': 0}
Setup complete.


## 10. Model 3 — full fine-tuning

Artifacts are saved under `phase5_lsi/runs/model3_full/`. Validation and test export use the best development checkpoint.

In [31]:
register('model3_full', 'model3', 'full')
capacity_check('model3_full')


Testing 1 cases/GPU; checkpointing=True
{'micro_batch_per_gpu': 1, 'activation_checkpointing': True, 'seconds_per_update': 1.476931752999917, 'amp_retries': 0, 'memory': [{'gpu': 0, 'peak_reserved_gib': 3.123046875, 'peak_tensor_gib': 2.696834087371826, 'headroom_gib': 11.21563720703125}, {'gpu': 1, 'peak_reserved_gib': 0.84375, 'peak_tensor_gib': 0.7276954650878906, 'headroom_gib': 13.50665283203125}], 'passed': True}
Testing 2 cases/GPU; checkpointing=True
{'micro_batch_per_gpu': 2, 'activation_checkpointing': True, 'seconds_per_update': 1.243984939000029, 'amp_retries': 0, 'memory': [{'gpu': 0, 'peak_reserved_gib': 3.095703125, 'peak_tensor_gib': 2.701875686645508, 'headroom_gib': 11.24102783203125}, {'gpu': 1, 'peak_reserved_gib': 1.1953125, 'peak_tensor_gib': 1.0375261306762695, 'headroom_gib': 13.15509033203125}], 'passed': True}
Testing 4 cases/GPU; checkpointing=True
{'micro_batch_per_gpu': 4, 'activation_checkpointing': True, 'seconds_per_update': 1.143410294999967, 'amp_retri

{'micro_batch_per_gpu': 4,
 'activation_checkpointing': False,
 'seconds_per_update': 0.9387136520001604,
 'amp_retries': 0,
 'memory': [{'gpu': 0,
   'peak_reserved_gib': 9.271484375,
   'peak_tensor_gib': 9.092005729675293,
   'headroom_gib': 5.06524658203125},
  {'gpu': 1,
   'peak_reserved_gib': 8.271484375,
   'peak_tensor_gib': 8.183613300323486,
   'headroom_gib': 6.07891845703125}],
 'passed': True}

In [32]:
model3_full_training = train_run('model3_full')


Parameters: {'total': 118584036, 'trainable': 118584036, 'adapted_modules': 0}


Evaluate:   0%|          | 0/1273 [00:00<?, ?it/s]

{'step': 0, 'epoch': 0, 'validation_macro_f1': 0.0714593819383141, 'validation_micro_f1': 0.07580680887947223, 'threshold': 0.4, 'best_macro_f1': 0.0714593819383141, 'early_stopping_active': False, 'bad_epochs': 0}
{'step': 1, 'epoch': 0, 'loss': 0.709544837474823, 'lr': 6.334125098970705e-09, 'gradient_norm': 0.5863776803016663, 'skipped': 0, 'elapsed_minutes': 4.640447354283337}
{'step': 20, 'epoch': 0, 'loss': 0.7057003974914551, 'lr': 6.65083135391924e-08, 'gradient_norm': 0.673926055431366, 'skipped': 0, 'elapsed_minutes': 4.827509843116669}
{'step': 40, 'epoch': 0, 'loss': 0.6955200433731079, 'lr': 1.2984956452889946e-07, 'gradient_norm': 0.7135727405548096, 'skipped': 0, 'elapsed_minutes': 5.032433708000004}
{'step': 60, 'epoch': 0, 'loss': 0.6942836046218872, 'lr': 1.931908155186065e-07, 'gradient_norm': 0.7138877511024475, 'skipped': 0, 'elapsed_minutes': 5.249480276300002}
{'step': 80, 'epoch': 0, 'loss': 0.6981431245803833, 'lr': 2.565320665083136e-07, 'gradient_norm': 0.535

Evaluate:   0%|          | 0/1273 [00:00<?, ?it/s]

{'step': 5263, 'epoch': 1, 'validation_macro_f1': 0.18691508415071542, 'validation_micro_f1': 0.3320820307313481, 'threshold': 0.05, 'best_macro_f1': 0.18691508415071542, 'early_stopping_active': True, 'bad_epochs': 0, 'epoch_train_loss': 0.2016970566847076}
{'step': 5280, 'epoch': 1, 'loss': 0.1389399766921997, 'lr': 1.6725257323832147e-05, 'gradient_norm': 0.214734748005867, 'skipped': 0, 'elapsed_minutes': 64.54788028368334}
{'step': 5300, 'epoch': 1, 'loss': 0.15468420088291168, 'lr': 1.6788598574821856e-05, 'gradient_norm': 0.17127501964569092, 'skipped': 0, 'elapsed_minutes': 64.76749250221668}
{'step': 5320, 'epoch': 1, 'loss': 0.09885700047016144, 'lr': 1.685193982581156e-05, 'gradient_norm': 0.18239295482635498, 'skipped': 0, 'elapsed_minutes': 64.99620068743334}
{'step': 5340, 'epoch': 1, 'loss': 0.13425764441490173, 'lr': 1.691528107680127e-05, 'gradient_norm': 0.17436330020427704, 'skipped': 0, 'elapsed_minutes': 65.19627487575}
{'step': 5360, 'epoch': 1, 'loss': 0.10323771

Evaluate:   0%|          | 0/1273 [00:00<?, ?it/s]

... (15 outputs left out)

Evaluate:   0%|          | 0/1273 [00:00<?, ?it/s]

{'step': 52630, 'epoch': 10, 'validation_macro_f1': 0.37525707818287635, 'validation_micro_f1': 0.4950311769290725, 'threshold': 0.2, 'best_macro_f1': 0.37525707818287635, 'early_stopping_active': True, 'bad_epochs': 0, 'epoch_train_loss': 0.0322982805089266}
{'step': 52640, 'epoch': 10, 'loss': 0.01983865350484848, 'lr': 1.189792720537989e-05, 'gradient_norm': 0.36167964339256287, 'skipped': 15, 'elapsed_minutes': 612.9433153528}
{'step': 52660, 'epoch': 10, 'loss': 0.028922738507390022, 'lr': 1.1892240378457957e-05, 'gradient_norm': 0.3677559792995453, 'skipped': 15, 'elapsed_minutes': 613.1554277163667}
{'step': 52680, 'epoch': 10, 'loss': 0.02706972137093544, 'lr': 1.1886553191742049e-05, 'gradient_norm': 0.57722008228302, 'skipped': 15, 'elapsed_minutes': 613.3616769405332}
{'step': 52700, 'epoch': 10, 'loss': 0.04195920377969742, 'lr': 1.188086564752551e-05, 'gradient_norm': 0.38550496101379395, 'skipped': 15, 'elapsed_minutes': 613.5670481531834}
{'step': 52720, 'epoch': 10, 'lo

Evaluate:   0%|          | 0/1273 [00:00<?, ?it/s]

{'step': 54231, 'epoch': 10, 'validation_macro_f1': 0.37709790916281327, 'validation_micro_f1': 0.4944354277764897, 'threshold': 0.2, 'best_macro_f1': 0.37709790916281327, 'early_stopping_active': True, 'bad_epochs': 0}
{'event': 'session_end', 'session_seconds': 38080.697873306, 'start_step': 0, 'end_step': 54231, 'complete': False}
{'name': 'model3_full', 'epoch': 10, 'position': 12808, 'step': 54231, 'best_macro_f1': 0.37709790916281327, 'bad_epochs': 0, 'complete': False, 'skipped': 16, 'consecutive_amp_skips': 0, 'epoch_loss_sum': 352.34326722845435, 'epoch_examples': 12808, 'session_seconds': 38080.697873306, 'session_paused': True}


In [33]:
model3_full_validation = validate_run('model3_full')


Evaluate:   0%|          | 0/1273 [00:00<?, ?it/s]

Examples: 10181
Labels: 100
Macro-F1: 0.377098
Micro-F1: 0.494435
Subset Accuracy: 0.053728

{
  "macro_f1": 0.37709790916281327,
  "micro_f1": 0.4944354277764897,
  "subset_accuracy": 0.05372753167665259,
  "macro_precision": 0.37811580012978857,
  "macro_recall": 0.39577523303995904,
  "mean_predicted_labels": 4.55691975248011,
  "mean_gold_labels": 3.9157253707887243,
  "empty_predictions": 280,
  "label_dimension": 100,
  "professor_dimension_matches": true,
  "threshold": 0.2,
  "saved_best_macro_f1": 0.37709790916281327,
  "inference_seconds": 276.6311876480031,
  "documents_per_second": 36.80351476838802,
  "parameters": {
    "total": 118584036,
    "trainable": 118584036,
    "adapted_modules": 0
  },
  "inference_memory": [
    {
      "gpu": 0,
      "peak_reserved_gib": 1.541015625,
      "peak_tensor_gib": 0.9986076354980469
    },
    {
      "gpu": 1,
      "peak_reserved_gib": 1.333984375,
      "peak_tensor_gib": 0.8496513366699219
    }
  ],
  "device": "cuda",
  "che

In [34]:
model3_full_analysis = analyze_run('model3_full')


 statute_id  train_support  dev_support  predicted_support  tp  fp  fn  precision   recall       f1 frequency_group
          0            341           86                 22   1  21  85   0.045455 0.011628 0.018519        over_100
          2            367           98                 37   4  33  94   0.108108 0.040816 0.059259        over_100
          8            346           92                 46   5  41  87   0.108696 0.054348 0.072464        over_100
         65            344          105                 30   6  24  99   0.200000 0.057143 0.088889        over_100
         25            286           72                 87   9  78  63   0.103448 0.125000 0.113208        over_100
         27            329           83                 44   8  36  75   0.181818 0.096386 0.125984        over_100
          1           1020          258                132  25 107 233   0.189394 0.096899 0.128205        over_100
         41            816          205                277  31 246 174  

In [35]:
if RUN_REPRESENTATION_ANALYSIS:
    model3_full_representations = representation_analysis('model3_full')


In [36]:
model3_full_submission = export_test('model3_full')


Test cases:   0%|          | 0/1628 [00:00<?, ?it/s]

Submission: /kaggle/working/phase5_lsi/runs/model3_full/test/92a4eaf4f862/model3_full_submission.zip


## 11. Model 1 — LoRMA

Artifacts are saved under `phase5_lsi/runs/model1_lorma/`. The memory check tests the runtime settings used for this run.

In [28]:
register('model1_lorma', 'model1', 'lorma')
capacity_check('model1_lorma')


Saved capacity selection: {'micro_batch_per_gpu': 4, 'activation_checkpointing': False, 'seconds_per_update': 1.1051716750007472, 'amp_retries': 0, 'memory': [{'gpu': 0, 'peak_reserved_gib': 9.90625, 'peak_tensor_gib': 9.615711212158203, 'headroom_gib': 4.43048095703125}, {'gpu': 1, 'peak_reserved_gib': 9.875, 'peak_tensor_gib': 9.587199211120605, 'headroom_gib': 4.47540283203125}], 'passed': True}


{'micro_batch_per_gpu': 4,
 'activation_checkpointing': False,
 'seconds_per_update': 1.1051716750007472,
 'amp_retries': 0,
 'memory': [{'gpu': 0,
   'peak_reserved_gib': 9.90625,
   'peak_tensor_gib': 9.615711212158203,
   'headroom_gib': 4.43048095703125},
  {'gpu': 1,
   'peak_reserved_gib': 9.875,
   'peak_tensor_gib': 9.587199211120605,
   'headroom_gib': 4.47540283203125}],
 'passed': True}

In [41]:
def train_run(name, additional_steps=None, teacher_name=None,
              micro_batch_per_gpu=None, activation_checkpointing=None):
    spec = resolved_spec(name)
    cfg = spec['config']
    execution_cfg = copy.deepcopy(cfg)
    if micro_batch_per_gpu is not None:
        if type(micro_batch_per_gpu) is not int or micro_batch_per_gpu < 1:
            raise ValueError('micro_batch_per_gpu must be a positive integer.')
        execution_cfg['micro_batch_per_gpu'] = micro_batch_per_gpu
    if activation_checkpointing is not None:
        if type(activation_checkpointing) is not bool:
            raise ValueError('activation_checkpointing must be True or False.')
        execution_cfg['activation_checkpointing'] = activation_checkpointing
    execution = dict(micro_batch_per_gpu=execution_cfg['micro_batch_per_gpu'],
                     activation_checkpointing=execution_cfg['activation_checkpointing'],
                     effective_batch=cfg['effective_batch'], gpus=torch.cuda.device_count())
    folder = run_folder(name)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    teacher_reference = None
    teacher_logits = None
    if spec['method'] == 'kd':
        if teacher_name is None:
            raise ValueError('Select and cache the teacher before distillation.')
        teacher_logits, teacher_reference = cache_teacher(teacher_name)

    signature = training_signature(spec, teacher_reference)
    if (folder / 'signature.json').is_file() and read_json(folder / 'signature.json') != signature:
        raise ValueError('Run inputs or settings changed. Use the original settings to resume.')
    if additional_steps is not None and additional_steps < 1:
        raise ValueError('additional_steps must be positive')
    write_json(folder / 'signature.json', signature)
    write_json(folder / 'config.json', spec)
    seed_everything(cfg['seed'])
    train_data = CaseDataset(spec['model'], 'train', cfg)
    dev_data = CaseDataset(spec['model'], 'dev', cfg)
    total = total_updates(cfg)
    model = make_model(spec['model'], spec['method'], execution_cfg, total, device)
    parameters = parameter_report(model)
    print('Parameters:', parameters, flush=True)
    if teacher_reference and parameters['total'] >= teacher_reference['parameters']:
        raise ValueError('The distilled student must be smaller than its teacher.')

    net = parallel_model(model)
    optimizer = make_optimizer(model, cfg)
    scheduler = make_scheduler(optimizer, cfg, total)
    scaler = torch.amp.GradScaler('cuda', enabled=device == 'cuda', init_scale=cfg['amp_initial_scale'])
    state = dict(epoch=0, position=0, step=0, best_macro_f1=-1.0,
                 bad_epochs=0, complete=False, skipped=0, consecutive_amp_skips=0,
                 epoch_loss_sum=0.0, epoch_examples=0)
    latest = folder / 'latest.pt'
    if latest.is_file():
        saved = torch.load(latest, map_location='cpu', weights_only=False)
        if saved['signature'] != signature:
            raise ValueError('Resume signature mismatch')
        model.load_state_dict(saved['model'], strict=True)
        optimizer.load_state_dict(saved['optimizer'])
        scheduler.load_state_dict(saved['scheduler'])
        scaler.load_state_dict(saved['scaler'])
        state = saved['progress']
        restore_rng(saved['rng'])
        del saved
        print('Resumed:', state, flush=True)

    start_step = state['step']
    started = time.monotonic()
    batch_size = execution_cfg['micro_batch_per_gpu'] * max(1, torch.cuda.device_count())
    write_json(folder / 'execution.json', execution)
    log_metric(folder, dict(event='execution_settings', step=state['step'], **execution))
    pos_weight = training_positive_weights(cfg, device)

    def save_latest():
        save_torch(latest, dict(
            signature=signature,
            model={k: v.detach().cpu() for k, v in model.state_dict().items()},
            optimizer=optimizer.state_dict(), scheduler=scheduler.state_dict(),
            scaler=scaler.state_dict(),
            progress=copy.deepcopy(state), rng=rng_state(), execution=execution,
        ))
        write_json(folder / 'progress.json', state)

    def validate(count_patience):
        logits = predict_logits(model, dev_data, batch_size, device)
        probabilities = logits.sigmoid().numpy()
        selected, _ = select_global_threshold(DEV_TARGETS, probabilities, cfg['threshold_grid'])
        threshold = selected['threshold']
        metrics = metric_report(DEV_TARGETS, probabilities, threshold)
        improved = update_early_stopping(state, metrics['macro_f1'], cfg, count_patience)
        row = dict(step=state['step'], epoch=state['epoch'],
                   validation_macro_f1=metrics['macro_f1'], validation_micro_f1=metrics['micro_f1'],
                   threshold=threshold, best_macro_f1=state['best_macro_f1'],
                   early_stopping_active=state['epoch'] > cfg['min_epochs'],
                   bad_epochs=state['bad_epochs'])
        if count_patience and state['epoch_examples']:
            row['epoch_train_loss'] = state['epoch_loss_sum'] / state['epoch_examples']
        log_metric(folder, row)
        if improved:
            save_torch(folder / 'best.pt', dict(
                signature=signature,
                model={k: v.detach().cpu() for k, v in model.state_dict().items()},
                step=state['step'], epoch=state['epoch'], metrics=metrics, threshold=threshold, parameters=parameters,
                execution=execution,
            ))
            write_json(folder / 'best_dev_probabilities.json', [
                dict(id=r['id'], gold=r['labels'], probabilities=p.tolist())
                for r, p in zip(CASE_INDEX['dev'], probabilities)
            ])
        model.train()

    try:
        if state['step'] == 0 and not (folder / 'best.pt').is_file():
            validate(False)
            save_latest()
        while not state['complete'] and state['epoch'] < cfg['epochs']:
            order = np.random.default_rng(cfg['seed'] + state['epoch']).permutation(len(train_data))
            model.train()
            while state['position'] < len(order):
                indices = order[state['position']:state['position'] + cfg['effective_batch']]
                optimizer.zero_grad(set_to_none=True)
                loss_sum = 0.0
                for begin in range(0, len(indices), batch_size):
                    selected = indices[begin:begin + batch_size]
                    batch = collate_cases([train_data[int(i)] for i in selected])
                    targets = batch['labels'].to(device)
                    with autocast_context(device):
                        logits = net(**model_inputs(batch, device))
                        ce = supervised_loss(logits, targets, cfg, pos_weight)
                        losses = ce
                        if teacher_logits is not None:
                            kl = distillation_loss(logits, teacher_logits[selected].to(device), cfg['kd_temperature'])
                            losses = (1 - cfg['kd_alpha']) * ce + cfg['kd_alpha'] * kl
                        loss = losses.sum() / len(indices)
                    if not torch.isfinite(loss).item():
                        raise FloatingPointError('Non-finite forward loss. Inspect the run before continuing.')
                    scaler.scale(loss).backward()
                    loss_sum += float(loss.detach())
                    del batch, targets, logits, ce, losses, loss
                    if teacher_logits is not None:
                        del kl
                scaler.unscale_(optimizer)
                norm = nn.utils.clip_grad_norm_(model.parameters(), cfg['clip'])
                before = scaler.get_scale()
                if not torch.isfinite(norm).item():
                    if device != 'cuda':
                        raise FloatingPointError('Non-finite CPU gradients; optimizer was not stepped.')
                    scaler.update(new_scale=before / 2)
                    skipped = True
                else:
                    scaler.step(optimizer)
                    scaler.update()
                    skipped = scaler.get_scale() < before
                if skipped:
                    state['skipped'] += 1
                    state['consecutive_amp_skips'] += 1
                    log_metric(folder, dict(event='amp_retry', step=state['step'], epoch=state['epoch'],
                        total_skips=state['skipped'], consecutive_skips=state['consecutive_amp_skips'],
                        scale_before=before, scale_after=scaler.get_scale()))
                    if state['consecutive_amp_skips'] >= cfg['max_consecutive_amp_skips']:
                        save_latest()
                        raise FloatingPointError('Repeated AMP failures on the same batch. Checkpoint saved; inspect retry logs.')
                    continue
                state['consecutive_amp_skips'] = 0
                state['step'] += 1
                scheduler.step()
                state['position'] += len(indices)
                state['epoch_loss_sum'] += loss_sum * len(indices)
                state['epoch_examples'] += len(indices)
                if state['step'] == start_step + 1 or state['step'] % cfg['log_every'] == 0:
                    log_metric(folder, dict(step=state['step'], epoch=state['epoch'], loss=loss_sum,
                        lr=optimizer.param_groups[0]['lr'], gradient_norm=float(norm),
                        skipped=state['skipped'], elapsed_minutes=(time.monotonic() - started) / 60))
                if state['step'] % cfg['save_every'] == 0:
                    save_latest()
                pause = ((additional_steps is not None and state['step'] - start_step >= additional_steps)
                         or time.monotonic() - started >= cfg['max_hours'] * 3600)
                if pause and state['position'] < len(order):
                    validate(False)
                    save_latest()
                    report = dict(name=name, **state, session_seconds=time.monotonic() - started,
                                  session_paused=True, execution=execution)
                    log_metric(folder, dict(event='session_end', session_seconds=report['session_seconds'],
                                            start_step=start_step, end_step=state['step'], complete=False))
                    print(report, flush=True)
                    return report
            state['epoch'] += 1
            state['position'] = 0
            validate(True)
            state['epoch_loss_sum'] = 0.0
            state['epoch_examples'] = 0
            exhausted = state['epoch'] >= cfg['epochs']
            plateau = state['epoch'] > cfg['min_epochs'] and state['bad_epochs'] >= cfg['patience']
            state['complete'] = exhausted or plateau
            if state['complete']:
                state['stop_reason'] = 'epochs' if exhausted else 'early_stopping'
            save_latest()
            if additional_steps is not None and state['step'] - start_step >= additional_steps:
                break
            if time.monotonic() - started >= cfg['max_hours'] * 3600:
                break
        report = dict(name=name, **state, parameters=parameters,
                      session_seconds=time.monotonic() - started, execution=execution)
        write_json(folder / 'summary.json', report)
        log_metric(folder, dict(event='session_end', session_seconds=report['session_seconds'],
                                start_step=start_step, end_step=state['step'], complete=state['complete']))
        print(report, flush=True)
        return report
    finally:
        optimizer.zero_grad(set_to_none=True)
        del net, model, optimizer, scheduler, scaler
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


def clear_error_memory():
    import sys
    import traceback
    errors = [getattr(sys, name, None) for name in ('last_exc', 'last_value')]
    try:
        shell = get_ipython()
    except NameError:
        shell = None
    result = getattr(shell, 'last_execution_result', None)
    if result is not None:
        errors.extend([result.error_in_exec, result.error_before_exec])
        result.error_in_exec = result.error_before_exec = None
    visited = set()
    while errors:
        error = errors.pop()
        if not isinstance(error, BaseException) or id(error) in visited:
            continue
        visited.add(id(error))
        if error.__traceback__ is not None:
            traceback.clear_frames(error.__traceback__)
        errors.extend([error.__cause__, error.__context__])
    old_traceback = getattr(sys, 'last_traceback', None)
    if old_traceback is not None:
        traceback.clear_frames(old_traceback)
    for name in ('last_exc', 'last_value', 'last_traceback', 'last_type'):
        if hasattr(sys, name): delattr(sys, name)
    errors = error = old_traceback = result = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        for gpu in range(torch.cuda.device_count()):
            free, total = torch.cuda.mem_get_info(gpu)
            print(f'GPU {gpu}: {free / 2**30:.2f}/{total / 2**30:.2f} GiB free', flush=True)


def check_resume_memory(name, successful_updates=2):
    if not torch.cuda.is_available(): raise RuntimeError('A CUDA GPU is required.')
    if type(successful_updates) is not int or successful_updates < 1:
        raise ValueError('successful_updates must be a positive integer.')
    clear_error_memory()
    spec = resolved_spec(name)
    if spec['method'] == 'kd': raise ValueError('This resume check is for supervised runs.')
    cfg = dict(spec['config'], micro_batch_per_gpu=1, activation_checkpointing=True)
    path = run_folder(name) / 'latest.pt'
    if not path.is_file(): raise FileNotFoundError(path)
    model = net = optimizer = scaler = saved = x = y = logits = loss = norm = None
    completed = failures = 0
    try:
        saved = torch.load(path, map_location='cpu', weights_only=False)
        if saved['signature'] != training_signature(spec, None):
            raise ValueError('Checkpoint/data/settings signature mismatch.')
        model = make_model(spec['model'], spec['method'], cfg, total_updates(cfg), 'cuda')
        model.load_state_dict(saved['model'], strict=True)
        model.train()
        net = parallel_model(model)
        optimizer = make_optimizer(model, cfg)
        optimizer.load_state_dict(saved['optimizer'])
        scaler = torch.amp.GradScaler('cuda', init_scale=cfg['amp_initial_scale'])
        scaler.load_state_dict(saved['scaler'])
        saved = None
        batch_size = min(torch.cuda.device_count(), cfg['effective_batch'])
        chunks, length = max_case_chunks(cfg), cfg['chunk_tokens']
        x = dict(ids=torch.randint(4, BASES[spec['model']]['config']['vocab_size'],
                     (batch_size, chunks, length), device='cuda'),
                 mask=torch.ones(batch_size, chunks, length, dtype=torch.bool, device='cuda'),
                 positions=torch.linspace(0, 1, chunks, device='cuda')[None].expand(batch_size, -1))
        y = (torch.arange(batch_size * len(LABEL_IDS), device='cuda').reshape(batch_size, -1) % 3 == 0).float()
        for gpu in range(torch.cuda.device_count()): torch.cuda.reset_peak_memory_stats(gpu)
        while completed < successful_updates:
            optimizer.zero_grad(set_to_none=True)
            for offset in range(0, cfg['effective_batch'], batch_size):
                count = min(batch_size, cfg['effective_batch'] - offset)
                with autocast_context('cuda'):
                    logits = net(**{key: value[:count] for key, value in x.items()})
                    loss = supervised_loss(logits, y[:count], cfg).sum() / cfg['effective_batch']
                if not torch.isfinite(loss).item(): raise FloatingPointError('Non-finite stress-test loss.')
                scaler.scale(loss).backward()
                del logits, loss
                logits = loss = None
            scaler.unscale_(optimizer)
            norm = nn.utils.clip_grad_norm_(model.parameters(), cfg['clip'])
            if not torch.isfinite(norm).item():
                failures += 1
                scaler.update(new_scale=scaler.get_scale() / 2)
                if failures >= cfg['max_consecutive_amp_skips']:
                    raise FloatingPointError('Repeated non-finite stress-test gradients.')
                continue
            scaler.step(optimizer)
            scaler.update()
            failures = 0
            completed += 1
            sync_cuda()
            print(f'Maximum-shape update {completed}/{successful_updates} passed.', flush=True)
        memory = []
        for gpu in range(torch.cuda.device_count()):
            free, total = torch.cuda.mem_get_info(gpu)
            peak = torch.cuda.max_memory_reserved(gpu)
            row = dict(gpu=gpu, peak_reserved_gib=peak / 2**30,
                       headroom_gib=min(free, total - peak) / 2**30)
            memory.append(row)
            print(row, flush=True)
        if any(row['headroom_gib'] < cfg['capacity_headroom_gib'] for row in memory):
            raise RuntimeError('The smaller batch still has insufficient memory headroom. Restart the kernel and rerun setup before retrying.')
        write_json(run_folder(name) / 'memory_resume_check.json', dict(
            checkpoint_sha256=digest_file(path), micro_batch_per_gpu=1,
            activation_checkpointing=True, effective_batch=cfg['effective_batch'],
            successful_updates=completed, memory=memory))
        print('Memory check passed. Saved model/optimizer checkpoints were not modified.', flush=True)
    finally:
        if optimizer is not None: optimizer.zero_grad(set_to_none=True)
        model = net = optimizer = scaler = saved = x = y = logits = loss = norm = None
        gc.collect()
        torch.cuda.empty_cache()


In [42]:
check_resume_memory("model1_lorma")

GPU 0: 14.20/14.56 GiB free
GPU 1: 14.31/14.56 GiB free
Maximum-shape update 1/2 passed.
Maximum-shape update 2/2 passed.
{'gpu': 0, 'peak_reserved_gib': 2.06640625, 'headroom_gib': 12.27032470703125}
{'gpu': 1, 'peak_reserved_gib': 0.875, 'headroom_gib': 13.47540283203125}
Memory check passed. Saved model/optimizer checkpoints were not modified.


In [43]:
model1_lorma_training = train_run(
    "model1_lorma",
    micro_batch_per_gpu=1,
    activation_checkpointing=True,
)

Parameters: {'total': 121647844, 'trainable': 1831268, 'adapted_modules': 64}
Resumed: {'epoch': 0, 'position': 41600, 'step': 5200, 'best_macro_f1': 0.07144196882913578, 'bad_epochs': 0, 'complete': False, 'skipped': 0, 'consecutive_amp_skips': 0, 'epoch_loss_sum': 8436.454857051373, 'epoch_examples': 41600}
{'event': 'execution_settings', 'step': 5200, 'micro_batch_per_gpu': 1, 'activation_checkpointing': True, 'effective_batch': 8, 'gpus': 2}
{'step': 5201, 'epoch': 0, 'loss': 0.1061587780714035, 'lr': 8.237529691211402e-05, 'gradient_norm': 0.13484816253185272, 'skipped': 0, 'elapsed_minutes': 0.023699541083332557}
{'step': 5220, 'epoch': 0, 'loss': 0.11770264804363251, 'lr': 8.267616785431513e-05, 'gradient_norm': 0.17652885615825653, 'skipped': 0, 'elapsed_minutes': 0.5301618638333329}
{'step': 5240, 'epoch': 0, 'loss': 0.0864377710968256, 'lr': 8.299287410926366e-05, 'gradient_norm': 0.14429982006549835, 'skipped': 0, 'elapsed_minutes': 1.0282863895166618}
{'step': 5260, 'epoch'

Evaluate:   0%|          | 0/5091 [00:00<?, ?it/s]

{'step': 5263, 'epoch': 1, 'validation_macro_f1': 0.16108529178302916, 'validation_micro_f1': 0.3147573524378836, 'threshold': 0.05, 'best_macro_f1': 0.16108529178302916, 'early_stopping_active': True, 'bad_epochs': 0, 'epoch_train_loss': 0.20178381652776053}
{'step': 5280, 'epoch': 1, 'loss': 0.1332082748413086, 'lr': 8.362628661916073e-05, 'gradient_norm': 0.15761078894138336, 'skipped': 0, 'elapsed_minutes': 13.06993756241667}
{'step': 5300, 'epoch': 1, 'loss': 0.1703578308224678, 'lr': 8.394299287410927e-05, 'gradient_norm': 0.14739352464675903, 'skipped': 0, 'elapsed_minutes': 13.581975320599998}
{'step': 5320, 'epoch': 1, 'loss': 0.09970990382134914, 'lr': 8.42596991290578e-05, 'gradient_norm': 0.12633535265922546, 'skipped': 0, 'elapsed_minutes': 14.078699977000001}
{'step': 5340, 'epoch': 1, 'loss': 0.13553068321198225, 'lr': 8.457640538400634e-05, 'gradient_norm': 0.12902610003948212, 'skipped': 0, 'elapsed_minutes': 14.582805961599998}
{'step': 5360, 'epoch': 1, 'loss': 0.104

Evaluate:   0%|          | 0/5091 [00:00<?, ?it/s]

{'step': 10526, 'epoch': 2, 'validation_macro_f1': 0.27255957689722776, 'validation_micro_f1': 0.424154760998469, 'threshold': 0.1, 'best_macro_f1': 0.27255957689722776, 'early_stopping_active': True, 'bad_epochs': 0, 'epoch_train_loss': 0.10447048594603815}
{'step': 10540, 'epoch': 2, 'loss': 0.11125395819544792, 'lr': 9.959570664293356e-05, 'gradient_norm': 0.17305828630924225, 'skipped': 0, 'elapsed_minutes': 157.60607515555}
{'step': 10560, 'epoch': 2, 'loss': 0.12903212755918503, 'lr': 9.959187576396546e-05, 'gradient_norm': 0.1990971714258194, 'skipped': 0, 'elapsed_minutes': 158.10326041766666}
{'step': 10580, 'epoch': 2, 'loss': 0.11984672397375107, 'lr': 9.958802690342115e-05, 'gradient_norm': 0.20504514873027802, 'skipped': 0, 'elapsed_minutes': 158.62281911388334}
{'step': 10600, 'epoch': 2, 'loss': 0.10578492842614651, 'lr': 9.958416006285269e-05, 'gradient_norm': 0.21525296568870544, 'skipped': 0, 'elapsed_minutes': 159.10941792648333}
{'step': 10620, 'epoch': 2, 'loss': 0

Evaluate:   0%|          | 0/5091 [00:00<?, ?it/s]

... (3 outputs left out)

Evaluate:   0%|          | 0/5091 [00:00<?, ?it/s]

{'step': 26315, 'epoch': 5, 'validation_macro_f1': 0.3336104398079705, 'validation_micro_f1': 0.4590934567611058, 'threshold': 0.1, 'best_macro_f1': 0.3336104398079705, 'early_stopping_active': True, 'bad_epochs': 0, 'epoch_train_loss': 0.08815118811402915}
{'step': 26320, 'epoch': 5, 'loss': 0.09994460828602314, 'lr': 9.12235084626786e-05, 'gradient_norm': 0.45532190799713135, 'skipped': 4, 'elapsed_minutes': 589.9987142217501}
{'step': 26340, 'epoch': 5, 'loss': 0.05333046615123749, 'lr': 9.120654656774821e-05, 'gradient_norm': 0.15667398273944855, 'skipped': 4, 'elapsed_minutes': 590.4830654698834}
{'step': 26360, 'epoch': 5, 'loss': 0.06307191774249077, 'lr': 9.118957007260723e-05, 'gradient_norm': 0.19450734555721283, 'skipped': 4, 'elapsed_minutes': 590.9981798688001}
{'step': 26380, 'epoch': 5, 'loss': 0.08734499383717775, 'lr': 9.117257898410143e-05, 'gradient_norm': 0.2729014754295349, 'skipped': 4, 'elapsed_minutes': 591.4950722731334}
{'step': 26400, 'epoch': 5, 'loss': 0.07

Evaluate:   0%|          | 0/5091 [00:00<?, ?it/s]

{'step': 27910, 'epoch': 5, 'validation_macro_f1': 0.3334856105119935, 'validation_micro_f1': 0.45020906220667295, 'threshold': 0.1, 'best_macro_f1': 0.3336104398079705, 'early_stopping_active': True, 'bad_epochs': 0}
{'event': 'session_end', 'session_seconds': 38446.572697775, 'start_step': 5200, 'end_step': 27910, 'complete': False}
{'name': 'model1_lorma', 'epoch': 5, 'position': 12760, 'step': 27910, 'best_macro_f1': 0.3336104398079705, 'bad_epochs': 0, 'complete': False, 'skipped': 4, 'consecutive_amp_skips': 0, 'epoch_loss_sum': 1066.2945904545486, 'epoch_examples': 12760, 'session_seconds': 38446.572697775, 'session_paused': True, 'execution': {'micro_batch_per_gpu': 1, 'activation_checkpointing': True, 'effective_batch': 8, 'gpus': 2}}


In [44]:
model1_lorma_validation = validate_run('model1_lorma')


Evaluate:   0%|          | 0/1273 [00:00<?, ?it/s]

Validation differs from the saved score. Check numerical/device/input differences.
Examples: 10181
Labels: 100
Macro-F1: 0.333523
Micro-F1: 0.459023
Subset Accuracy: 0.017975

{
  "macro_f1": 0.3335231812490585,
  "micro_f1": 0.45902261756251017,
  "subset_accuracy": 0.017974658677929477,
  "macro_precision": 0.2912233151699791,
  "macro_recall": 0.4409067664772497,
  "mean_predicted_labels": 6.949808466751793,
  "mean_gold_labels": 3.9157253707887243,
  "empty_predictions": 98,
  "label_dimension": 100,
  "professor_dimension_matches": true,
  "threshold": 0.1,
  "saved_best_macro_f1": 0.3336104398079705,
  "inference_seconds": 410.8628804079999,
  "documents_per_second": 24.779556600221326,
  "parameters": {
    "total": 121647844,
    "trainable": 1831268,
    "adapted_modules": 64
  },
  "inference_memory": [
    {
      "gpu": 0,
      "peak_reserved_gib": 2.26171875,
      "peak_tensor_gib": 1.0001888275146484
    },
    {
      "gpu": 1,
      "peak_reserved_gib": 1.712890625,
 

In [45]:
model1_lorma_analysis = analyze_run('model1_lorma')


 statute_id  train_support  dev_support  predicted_support  tp  fp  fn  precision   recall       f1 frequency_group
          0            341           86                  4   0   4  86   0.000000 0.000000 0.000000        over_100
          2            367           98                  0   0   0  98   0.000000 0.000000 0.000000        over_100
          8            346           92                  2   0   2  92   0.000000 0.000000 0.000000        over_100
         65            344          105                  1   0   1 105   0.000000 0.000000 0.000000        over_100
         40            531          152                 21   1  20 151   0.047619 0.006579 0.011561        over_100
         84            266           68                  4   2   2  66   0.500000 0.029412 0.055556        over_100
         52           1068          272                 54  11  43 261   0.203704 0.040441 0.067485        over_100
          1           1020          258                 83  12  71 246  

In [46]:
if RUN_REPRESENTATION_ANALYSIS:
    model1_lorma_representations = representation_analysis('model1_lorma')


In [47]:
model1_lorma_submission = export_test('model1_lorma')


Test cases:   0%|          | 0/1628 [00:00<?, ?it/s]

Submission: /kaggle/working/phase5_lsi/runs/model1_lorma/test/c2877478c680/model1_lorma_submission.zip


## 12. Model 4 · Smaller CNN with distillation

The student keeps alternate convolution blocks. Attach both saved teacher candidates to start this section from a fresh session.

In [28]:
def train_run(name, additional_steps=None, teacher_name=None,
              micro_batch_per_gpu=None, activation_checkpointing=None):
    spec = resolved_spec(name)
    cfg = spec['config']
    execution_cfg = copy.deepcopy(cfg)
    if micro_batch_per_gpu is not None:
        if type(micro_batch_per_gpu) is not int or micro_batch_per_gpu < 1:
            raise ValueError('micro_batch_per_gpu must be a positive integer.')
        execution_cfg['micro_batch_per_gpu'] = micro_batch_per_gpu
    if activation_checkpointing is not None:
        if type(activation_checkpointing) is not bool:
            raise ValueError('activation_checkpointing must be True or False.')
        execution_cfg['activation_checkpointing'] = activation_checkpointing
    execution = dict(micro_batch_per_gpu=execution_cfg['micro_batch_per_gpu'],
                     activation_checkpointing=execution_cfg['activation_checkpointing'],
                     effective_batch=cfg['effective_batch'], gpus=torch.cuda.device_count())
    folder = run_folder(name)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    teacher_reference = None
    teacher_logits = None
    if spec['method'] == 'kd':
        if teacher_name is None:
            raise ValueError('Select and cache the teacher before distillation.')
        teacher_logits, teacher_reference = cache_teacher(teacher_name)

    signature = training_signature(spec, teacher_reference)
    if (folder / 'signature.json').is_file() and read_json(folder / 'signature.json') != signature:
        raise ValueError('Run inputs or settings changed. Use the original settings to resume.')
    if additional_steps is not None and additional_steps < 1:
        raise ValueError('additional_steps must be positive')
    write_json(folder / 'signature.json', signature)
    write_json(folder / 'config.json', spec)
    seed_everything(cfg['seed'])
    train_data = CaseDataset(spec['model'], 'train', cfg)
    dev_data = CaseDataset(spec['model'], 'dev', cfg)
    total = total_updates(cfg)
    model = make_model(spec['model'], spec['method'], execution_cfg, total, device)
    parameters = parameter_report(model)
    print('Parameters:', parameters, flush=True)
    if teacher_reference and parameters['total'] >= teacher_reference['parameters']:
        raise ValueError('The distilled student must be smaller than its teacher.')

    net = parallel_model(model)
    optimizer = make_optimizer(model, cfg)
    scheduler = make_scheduler(optimizer, cfg, total)
    scaler = torch.amp.GradScaler('cuda', enabled=device == 'cuda', init_scale=cfg['amp_initial_scale'])
    state = dict(epoch=0, position=0, step=0, best_macro_f1=-1.0,
                 bad_epochs=0, complete=False, skipped=0, consecutive_amp_skips=0,
                 epoch_loss_sum=0.0, epoch_examples=0)
    latest = folder / 'latest.pt'
    if latest.is_file():
        saved = torch.load(latest, map_location='cpu', weights_only=False)
        if saved['signature'] != signature:
            raise ValueError('Resume signature mismatch')
        model.load_state_dict(saved['model'], strict=True)
        optimizer.load_state_dict(saved['optimizer'])
        scheduler.load_state_dict(saved['scheduler'])
        scaler.load_state_dict(saved['scaler'])
        state = saved['progress']
        restore_rng(saved['rng'])
        del saved
        print('Resumed:', state, flush=True)

    start_step = state['step']
    started = time.monotonic()
    batch_size = execution_cfg['micro_batch_per_gpu'] * max(1, torch.cuda.device_count())
    write_json(folder / 'execution.json', execution)
    log_metric(folder, dict(event='execution_settings', step=state['step'], **execution))
    pos_weight = training_positive_weights(cfg, device)

    def save_latest():
        save_torch(latest, dict(
            signature=signature,
            model={k: v.detach().cpu() for k, v in model.state_dict().items()},
            optimizer=optimizer.state_dict(), scheduler=scheduler.state_dict(),
            scaler=scaler.state_dict(),
            progress=copy.deepcopy(state), rng=rng_state(), execution=execution,
        ))
        write_json(folder / 'progress.json', state)

    def validate(count_patience):
        logits = predict_logits(model, dev_data, batch_size, device)
        probabilities = logits.sigmoid().numpy()
        selected, _ = select_global_threshold(DEV_TARGETS, probabilities, cfg['threshold_grid'])
        threshold = selected['threshold']
        metrics = metric_report(DEV_TARGETS, probabilities, threshold)
        improved = update_early_stopping(state, metrics['macro_f1'], cfg, count_patience)
        row = dict(step=state['step'], epoch=state['epoch'],
                   validation_macro_f1=metrics['macro_f1'], validation_micro_f1=metrics['micro_f1'],
                   threshold=threshold, best_macro_f1=state['best_macro_f1'],
                   early_stopping_active=state['epoch'] > cfg['min_epochs'],
                   bad_epochs=state['bad_epochs'])
        if count_patience and state['epoch_examples']:
            row['epoch_train_loss'] = state['epoch_loss_sum'] / state['epoch_examples']
        log_metric(folder, row)
        if improved:
            save_torch(folder / 'best.pt', dict(
                signature=signature,
                model={k: v.detach().cpu() for k, v in model.state_dict().items()},
                step=state['step'], epoch=state['epoch'], metrics=metrics, threshold=threshold, parameters=parameters,
                execution=execution,
            ))
            write_json(folder / 'best_dev_probabilities.json', [
                dict(id=r['id'], gold=r['labels'], probabilities=p.tolist())
                for r, p in zip(CASE_INDEX['dev'], probabilities)
            ])
        model.train()

    try:
        if state['step'] == 0 and not (folder / 'best.pt').is_file():
            validate(False)
            save_latest()
        while not state['complete'] and state['epoch'] < cfg['epochs']:
            order = np.random.default_rng(cfg['seed'] + state['epoch']).permutation(len(train_data))
            model.train()
            while state['position'] < len(order):
                indices = order[state['position']:state['position'] + cfg['effective_batch']]
                optimizer.zero_grad(set_to_none=True)
                loss_sum = 0.0
                for begin in range(0, len(indices), batch_size):
                    selected = indices[begin:begin + batch_size]
                    batch = collate_cases([train_data[int(i)] for i in selected])
                    targets = batch['labels'].to(device)
                    with autocast_context(device):
                        logits = net(**model_inputs(batch, device))
                        ce = supervised_loss(logits, targets, cfg, pos_weight)
                        losses = ce
                        if teacher_logits is not None:
                            kl = distillation_loss(logits, teacher_logits[selected].to(device), cfg['kd_temperature'])
                            losses = (1 - cfg['kd_alpha']) * ce + cfg['kd_alpha'] * kl
                        loss = losses.sum() / len(indices)
                    if not torch.isfinite(loss).item():
                        raise FloatingPointError('Non-finite forward loss. Inspect the run before continuing.')
                    scaler.scale(loss).backward()
                    loss_sum += float(loss.detach())
                    del batch, targets, logits, ce, losses, loss
                    if teacher_logits is not None:
                        del kl
                scaler.unscale_(optimizer)
                norm = nn.utils.clip_grad_norm_(model.parameters(), cfg['clip'])
                before = scaler.get_scale()
                if not torch.isfinite(norm).item():
                    if device != 'cuda':
                        raise FloatingPointError('Non-finite CPU gradients; optimizer was not stepped.')
                    scaler.update(new_scale=before / 2)
                    skipped = True
                else:
                    scaler.step(optimizer)
                    scaler.update()
                    skipped = scaler.get_scale() < before
                if skipped:
                    state['skipped'] += 1
                    state['consecutive_amp_skips'] += 1
                    log_metric(folder, dict(event='amp_retry', step=state['step'], epoch=state['epoch'],
                        total_skips=state['skipped'], consecutive_skips=state['consecutive_amp_skips'],
                        scale_before=before, scale_after=scaler.get_scale()))
                    if state['consecutive_amp_skips'] >= cfg['max_consecutive_amp_skips']:
                        save_latest()
                        raise FloatingPointError('Repeated AMP failures on the same batch. Checkpoint saved; inspect retry logs.')
                    continue
                state['consecutive_amp_skips'] = 0
                state['step'] += 1
                scheduler.step()
                state['position'] += len(indices)
                state['epoch_loss_sum'] += loss_sum * len(indices)
                state['epoch_examples'] += len(indices)
                if state['step'] == start_step + 1 or state['step'] % cfg['log_every'] == 0:
                    log_metric(folder, dict(step=state['step'], epoch=state['epoch'], loss=loss_sum,
                        lr=optimizer.param_groups[0]['lr'], gradient_norm=float(norm),
                        skipped=state['skipped'], elapsed_minutes=(time.monotonic() - started) / 60))
                if state['step'] % cfg['save_every'] == 0:
                    save_latest()
                pause = ((additional_steps is not None and state['step'] - start_step >= additional_steps)
                         or time.monotonic() - started >= cfg['max_hours'] * 3600)
                if pause and state['position'] < len(order):
                    validate(False)
                    save_latest()
                    report = dict(name=name, **state, session_seconds=time.monotonic() - started,
                                  session_paused=True, execution=execution)
                    log_metric(folder, dict(event='session_end', session_seconds=report['session_seconds'],
                                            start_step=start_step, end_step=state['step'], complete=False))
                    print(report, flush=True)
                    return report
            state['epoch'] += 1
            state['position'] = 0
            validate(True)
            state['epoch_loss_sum'] = 0.0
            state['epoch_examples'] = 0
            exhausted = state['epoch'] >= cfg['epochs']
            plateau = state['epoch'] > cfg['min_epochs'] and state['bad_epochs'] >= cfg['patience']
            state['complete'] = exhausted or plateau
            if state['complete']:
                state['stop_reason'] = 'epochs' if exhausted else 'early_stopping'
            save_latest()
            if additional_steps is not None and state['step'] - start_step >= additional_steps:
                break
            if time.monotonic() - started >= cfg['max_hours'] * 3600:
                break
        report = dict(name=name, **state, parameters=parameters,
                      session_seconds=time.monotonic() - started, execution=execution)
        write_json(folder / 'summary.json', report)
        log_metric(folder, dict(event='session_end', session_seconds=report['session_seconds'],
                                start_step=start_step, end_step=state['step'], complete=state['complete']))
        print(report, flush=True)
        return report
    finally:
        optimizer.zero_grad(set_to_none=True)
        del net, model, optimizer, scheduler, scaler
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


def clear_error_memory():
    import sys
    import traceback
    errors = [getattr(sys, name, None) for name in ('last_exc', 'last_value')]
    try:
        shell = get_ipython()
    except NameError:
        shell = None
    result = getattr(shell, 'last_execution_result', None)
    if result is not None:
        errors.extend([result.error_in_exec, result.error_before_exec])
        result.error_in_exec = result.error_before_exec = None
    visited = set()
    while errors:
        error = errors.pop()
        if not isinstance(error, BaseException) or id(error) in visited:
            continue
        visited.add(id(error))
        if error.__traceback__ is not None:
            traceback.clear_frames(error.__traceback__)
        errors.extend([error.__cause__, error.__context__])
    old_traceback = getattr(sys, 'last_traceback', None)
    if old_traceback is not None:
        traceback.clear_frames(old_traceback)
    for name in ('last_exc', 'last_value', 'last_traceback', 'last_type'):
        if hasattr(sys, name): delattr(sys, name)
    errors = error = old_traceback = result = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        for gpu in range(torch.cuda.device_count()):
            free, total = torch.cuda.mem_get_info(gpu)
            print(f'GPU {gpu}: {free / 2**30:.2f}/{total / 2**30:.2f} GiB free', flush=True)


def check_resume_memory(name, successful_updates=2):
    if not torch.cuda.is_available(): raise RuntimeError('A CUDA GPU is required.')
    if type(successful_updates) is not int or successful_updates < 1:
        raise ValueError('successful_updates must be a positive integer.')
    clear_error_memory()
    spec = resolved_spec(name)
    if spec['method'] == 'kd': raise ValueError('This resume check is for supervised runs.')
    cfg = dict(spec['config'], micro_batch_per_gpu=1, activation_checkpointing=True)
    path = run_folder(name) / 'latest.pt'
    if not path.is_file(): raise FileNotFoundError(path)
    model = net = optimizer = scaler = saved = x = y = logits = loss = norm = None
    completed = failures = 0
    try:
        saved = torch.load(path, map_location='cpu', weights_only=False)
        if saved['signature'] != training_signature(spec, None):
            raise ValueError('Checkpoint/data/settings signature mismatch.')
        model = make_model(spec['model'], spec['method'], cfg, total_updates(cfg), 'cuda')
        model.load_state_dict(saved['model'], strict=True)
        model.train()
        net = parallel_model(model)
        optimizer = make_optimizer(model, cfg)
        optimizer.load_state_dict(saved['optimizer'])
        scaler = torch.amp.GradScaler('cuda', init_scale=cfg['amp_initial_scale'])
        scaler.load_state_dict(saved['scaler'])
        saved = None
        batch_size = min(torch.cuda.device_count(), cfg['effective_batch'])
        chunks, length = max_case_chunks(cfg), cfg['chunk_tokens']
        x = dict(ids=torch.randint(4, BASES[spec['model']]['config']['vocab_size'],
                     (batch_size, chunks, length), device='cuda'),
                 mask=torch.ones(batch_size, chunks, length, dtype=torch.bool, device='cuda'),
                 positions=torch.linspace(0, 1, chunks, device='cuda')[None].expand(batch_size, -1))
        y = (torch.arange(batch_size * len(LABEL_IDS), device='cuda').reshape(batch_size, -1) % 3 == 0).float()
        for gpu in range(torch.cuda.device_count()): torch.cuda.reset_peak_memory_stats(gpu)
        while completed < successful_updates:
            optimizer.zero_grad(set_to_none=True)
            for offset in range(0, cfg['effective_batch'], batch_size):
                count = min(batch_size, cfg['effective_batch'] - offset)
                with autocast_context('cuda'):
                    logits = net(**{key: value[:count] for key, value in x.items()})
                    loss = supervised_loss(logits, y[:count], cfg).sum() / cfg['effective_batch']
                if not torch.isfinite(loss).item(): raise FloatingPointError('Non-finite stress-test loss.')
                scaler.scale(loss).backward()
                del logits, loss
                logits = loss = None
            scaler.unscale_(optimizer)
            norm = nn.utils.clip_grad_norm_(model.parameters(), cfg['clip'])
            if not torch.isfinite(norm).item():
                failures += 1
                scaler.update(new_scale=scaler.get_scale() / 2)
                if failures >= cfg['max_consecutive_amp_skips']:
                    raise FloatingPointError('Repeated non-finite stress-test gradients.')
                continue
            scaler.step(optimizer)
            scaler.update()
            failures = 0
            completed += 1
            sync_cuda()
            print(f'Maximum-shape update {completed}/{successful_updates} passed.', flush=True)
        memory = []
        for gpu in range(torch.cuda.device_count()):
            free, total = torch.cuda.mem_get_info(gpu)
            peak = torch.cuda.max_memory_reserved(gpu)
            row = dict(gpu=gpu, peak_reserved_gib=peak / 2**30,
                       headroom_gib=min(free, total - peak) / 2**30)
            memory.append(row)
            print(row, flush=True)
        if any(row['headroom_gib'] < cfg['capacity_headroom_gib'] for row in memory):
            raise RuntimeError('The smaller batch still has insufficient memory headroom. Restart the kernel and rerun setup before retrying.')
        write_json(run_folder(name) / 'memory_resume_check.json', dict(
            checkpoint_sha256=digest_file(path), micro_batch_per_gpu=1,
            activation_checkpointing=True, effective_batch=cfg['effective_batch'],
            successful_updates=completed, memory=memory))
        print('Memory check passed. Saved model/optimizer checkpoints were not modified.', flush=True)
    finally:
        if optimizer is not None: optimizer.zero_grad(set_to_none=True)
        model = net = optimizer = scaler = saved = x = y = logits = loss = norm = None
        gc.collect()
        torch.cuda.empty_cache()


required_setup = (
    'BASES', 'MODEL_CLASSES', 'CASE_INDEX', 'LABEL_IDS', 'TRAIN_TARGETS',
    'DEV_TARGETS', 'TOKEN_FOLDERS', 'DEFAULT', 'register', 'capacity_check',
    'resolved_spec', 'make_model', 'training_signature', 'validate_run',
    'cache_teacher', 'analyze_run', 'export_test', 'parallel_model',
    'training_positive_weights', 'predict_logits', 'save_torch', 'rng_state',
    'restore_rng', 'log_metric', 'update_early_stopping', 'total_updates',
    'model_inputs', 'supervised_loss', 'distillation_loss', 'autocast_context',
    'make_optimizer', 'make_scheduler', 'CaseDataset', 'collate_cases',
)
missing_setup = [name for name in required_setup if name not in globals()]
if missing_setup:
    raise RuntimeError(
        'Run setup and definition cells through section 9 first; skip the earlier training cells. '
        f'Missing: {missing_setup}'
    )
for model_name in ('model1', 'model3', 'model4'):
    if model_name not in BASES or (model_name, 'train') not in TOKEN_FOLDERS:
        raise RuntimeError(f'Missing loaded assets or prepared tokens for {model_name}. Rerun setup.')
for split in ('train', 'dev', 'test'):
    for model_name in ('model1', 'model3', 'model4'):
        folder = TOKEN_FOLDERS.get((model_name, split))
        if folder is None or not (Path(folder) / 'tokens.bin').is_file():
            raise FileNotFoundError(f'Missing prepared {split} tokens for {model_name}.')
for teacher_run in ('model3_full', 'model1_lorma'):
    for filename in ('best.pt', 'config.json', 'signature.json'):
        file = run_folder(teacher_run) / filename
        if not file.is_file():
            raise FileNotFoundError(f'Restore the saved teacher run before proceeding: {file}')
RUN_NAMES = ['model3_full', 'model1_lorma', 'model4_distilled']
clear_error_memory()
register('model4_distilled', 'model4', 'kd', student_keep=list(range(0, 16, 2)))
model4_capacity = capacity_check('model4_distilled')


GPU 0: 14.38/14.56 GiB free
GPU 1: 14.46/14.56 GiB free
Saved capacity selection: {'micro_batch_per_gpu': 4, 'activation_checkpointing': False, 'seconds_per_update': 0.49902103900240036, 'amp_retries': 0, 'memory': [{'gpu': 0, 'peak_reserved_gib': 4.205078125, 'peak_tensor_gib': 3.901907444000244, 'headroom_gib': 9.75079345703125}, {'gpu': 1, 'peak_reserved_gib': 3.515625, 'peak_tensor_gib': 3.33944034576416, 'headroom_gib': 10.82891845703125}], 'passed': True}


### Select the teacher

Select the better saved Model 3 full or Model 1 LoRMA checkpoint by development macro-F1. The student caches its training-case logits and keeps the same teacher when resuming.

In [29]:
def choose_teacher():
    allowed = {'model3_full': ('model3', 'full'), 'model1_lorma': ('model1', 'lorma')}
    candidates = []
    expected_cases = {split: object_hash(CASE_INDEX[split]) for split in ('train', 'dev')}
    for name, (model_name, method) in allowed.items():
        folder = run_folder(name)
        for filename in ('best.pt', 'config.json', 'signature.json'):
            if not (folder / filename).is_file():
                raise FileNotFoundError(f'Finish or pause both teacher runs first. Missing: {folder / filename}')
        spec = read_json(folder / 'config.json')
        if (spec['model'], spec['method']) != (model_name, method):
            raise ValueError(f'Unexpected model/method in {name}.')
        before = digest_file(folder / 'best.pt')
        saved = torch.load(folder / 'best.pt', map_location='cpu', weights_only=False)
        try:
            signature = read_json(folder / 'signature.json')
            if saved['signature'] != signature:
                raise ValueError(f'Checkpoint signature mismatch: {name}')
            if signature['cases'] != expected_cases or signature['label_ids'] != LABEL_IDS:
                raise ValueError(f'Teacher {name} used different cases or label IDs.')
            if signature['base_checkpoint'] != BASES[model_name]['checkpoint_sha256']:
                raise ValueError(f'Teacher {name} used a different Phase 1 checkpoint.')
            if signature['tokenizer'] != BASES[model_name]['tokenizer_sha256']:
                raise ValueError(f'Teacher {name} used a different tokenizer.')
            score = float(saved['metrics']['macro_f1'])
            if not math.isfinite(score): raise ValueError(f'Invalid best score in {name}.')
            progress = read_json(folder / 'progress.json') if (folder / 'progress.json').is_file() else {}
            candidates.append(dict(name=name, model=model_name, method=method, macro_f1=score,
                best_step=int(saved['step']), best_epoch=int(saved['epoch']),
                training_complete=bool(progress.get('complete', False)), checkpoint_sha256=before))
        finally:
            del saved
            gc.collect()
        if digest_file(folder / 'best.pt') != before:
            raise RuntimeError('A teacher checkpoint changed during selection. Finish or pause teacher training first.')
    candidates.sort(key=lambda row: (-row['macro_f1'], row['name']))
    selected = candidates[0]
    display(pd.DataFrame(candidates)[['name', 'macro_f1', 'best_epoch', 'best_step', 'training_complete']])
    write_json(OUTPUT_ROOT / 'teacher_selection_restricted.json', dict(
        allowed_names=list(allowed), candidates=candidates, selected=selected))
    print('Selected teacher:', selected['name'], '| best development Macro-F1:', selected['macro_f1'], flush=True)
    return selected['name']


student_signature = run_folder('model4_distilled') / 'signature.json'
if student_signature.is_file():
    recorded_teacher = read_json(student_signature).get('teacher')
    if not recorded_teacher or recorded_teacher['name'] not in ('model3_full', 'model1_lorma'):
        raise ValueError('This student has a different teacher. Use a new student run name rather than changing a resumed experiment.')
    teacher_name = recorded_teacher['name']
    expected_teacher_hash = recorded_teacher['checkpoint']
    print('Resuming with the recorded teacher:', teacher_name, flush=True)
else:
    teacher_name = choose_teacher()
    expected_teacher_hash = read_json(OUTPUT_ROOT / 'teacher_selection_restricted.json')['selected']['checkpoint_sha256']

if digest_file(run_folder(teacher_name) / 'best.pt') != expected_teacher_hash:
    raise ValueError('The selected teacher checkpoint changed. Preserve the original teacher for this student.')

teacher_validation = validate_run(teacher_name)
teacher_logits, teacher_reference = cache_teacher(teacher_name)
if teacher_reference['checkpoint'] != expected_teacher_hash:
    del teacher_logits
    raise ValueError('Teacher changed while preparing its cache. Pause teacher training and retry.')
print('Teacher:', teacher_reference, flush=True)
del teacher_logits


Resuming with the recorded teacher: model3_full


Evaluate:   0%|          | 0/1273 [00:00<?, ?it/s]

Examples: 10181
Labels: 100
Macro-F1: 0.377098
Micro-F1: 0.494435
Subset Accuracy: 0.053728

{
  "macro_f1": 0.37709790916281327,
  "micro_f1": 0.4944354277764897,
  "subset_accuracy": 0.05372753167665259,
  "macro_precision": 0.37811580012978857,
  "macro_recall": 0.39577523303995904,
  "mean_predicted_labels": 4.55691975248011,
  "mean_gold_labels": 3.9157253707887243,
  "empty_predictions": 280,
  "label_dimension": 100,
  "professor_dimension_matches": true,
  "threshold": 0.2,
  "saved_best_macro_f1": 0.37709790916281327,
  "inference_seconds": 291.50239805,
  "documents_per_second": 34.92595624634862,
  "parameters": {
    "total": 118584036,
    "trainable": 118584036,
    "adapted_modules": 0
  },
  "inference_memory": [
    {
      "gpu": 0,
      "peak_reserved_gib": 1.5546875,
      "peak_tensor_gib": 0.9976997375488281
    },
    {
      "gpu": 1,
      "peak_reserved_gib": 1.32421875,
      "peak_tensor_gib": 0.8407402038574219
    }
  ],
  "device": "cuda",
  "checkpoint_

In [ ]:
model4_distilled_training = train_run(
    'model4_distilled', teacher_name=teacher_name
)
print('Training complete:', model4_distilled_training['complete'], flush=True)
if not model4_distilled_training['complete']:
    print('The run paused. Rerun this training cell to continue; later cells use its saved best checkpoint.', flush=True)


Parameters: {'total': 70378852, 'trainable': 70378852, 'adapted_modules': 0}
{'event': 'execution_settings', 'step': 0, 'micro_batch_per_gpu': 4, 'activation_checkpointing': False, 'effective_batch': 8, 'gpus': 2}


Evaluate:   0%|          | 0/1273 [00:00<?, ?it/s]

{'step': 0, 'epoch': 0, 'validation_macro_f1': 0.07144000786937695, 'validation_micro_f1': 0.07536511891472414, 'threshold': 0.3, 'best_macro_f1': 0.07144000786937695, 'early_stopping_active': False, 'bad_epochs': 0}
{'step': 1, 'epoch': 0, 'loss': 1.4989368915557861, 'lr': 6.334125098970705e-09, 'gradient_norm': 0.7942777872085571, 'skipped': 0, 'elapsed_minutes': 2.2149066913500066}
{'step': 20, 'epoch': 0, 'loss': 1.6046273708343506, 'lr': 6.65083135391924e-08, 'gradient_norm': 1.0523961782455444, 'skipped': 0, 'elapsed_minutes': 2.308621527566667}
{'step': 40, 'epoch': 0, 'loss': 1.5471012592315674, 'lr': 1.2984956452889946e-07, 'gradient_norm': 1.0591052770614624, 'skipped': 0, 'elapsed_minutes': 2.4095315356500047}
{'step': 60, 'epoch': 0, 'loss': 1.5450944900512695, 'lr': 1.931908155186065e-07, 'gradient_norm': 1.0429675579071045, 'skipped': 0, 'elapsed_minutes': 2.513568223716667}
{'step': 80, 'epoch': 0, 'loss': 1.5371439456939697, 'lr': 2.565320665083136e-07, 'gradient_norm':

Evaluate:   0%|          | 0/1273 [00:00<?, ?it/s]

{'step': 5263, 'epoch': 1, 'validation_macro_f1': 0.1065768814702009, 'validation_micro_f1': 0.3618814418313308, 'threshold': 0.05, 'best_macro_f1': 0.1065768814702009, 'early_stopping_active': True, 'bad_epochs': 0, 'epoch_train_loss': 0.4736700634521224}
{'step': 5280, 'epoch': 1, 'loss': 0.2835214138031006, 'lr': 1.6725257323832147e-05, 'gradient_norm': 0.6519091725349426, 'skipped': 0, 'elapsed_minutes': 32.07088699255}
{'step': 5300, 'epoch': 1, 'loss': 0.30975502729415894, 'lr': 1.6788598574821856e-05, 'gradient_norm': 0.36794546246528625, 'skipped': 0, 'elapsed_minutes': 32.18019536246667}
{'step': 5320, 'epoch': 1, 'loss': 0.21455107629299164, 'lr': 1.685193982581156e-05, 'gradient_norm': 0.38140416145324707, 'skipped': 0, 'elapsed_minutes': 32.296319588116674}
{'step': 5340, 'epoch': 1, 'loss': 0.26940637826919556, 'lr': 1.691528107680127e-05, 'gradient_norm': 0.30378618836402893, 'skipped': 0, 'elapsed_minutes': 32.39792614346667}
{'step': 5360, 'epoch': 1, 'loss': 0.20894341

Evaluate:   0%|          | 0/1273 [00:00<?, ?it/s]

... (9 outputs left out)

Evaluate:   0%|          | 0/1273 [00:00<?, ?it/s]

{'step': 36841, 'epoch': 7, 'validation_macro_f1': 0.3146832548689996, 'validation_micro_f1': 0.42403260124779113, 'threshold': 0.05, 'best_macro_f1': 0.3146832548689996, 'early_stopping_active': True, 'bad_epochs': 0, 'epoch_train_loss': 0.09593473600827908}
{'step': 36860, 'epoch': 7, 'loss': 0.07308200001716614, 'lr': 1.6088948910348307e-05, 'gradient_norm': 0.7639464735984802, 'skipped': 9, 'elapsed_minutes': 214.81750659705}
{'step': 36880, 'epoch': 7, 'loss': 0.08385811746120453, 'lr': 1.6084234074786813e-05, 'gradient_norm': 0.8825846910476685, 'skipped': 9, 'elapsed_minutes': 214.91611918515}
{'step': 36900, 'epoch': 7, 'loss': 0.05629535764455795, 'lr': 1.6079517189019072e-05, 'gradient_norm': 0.9332281947135925, 'skipped': 9, 'elapsed_minutes': 215.01800488245001}
{'step': 36920, 'epoch': 7, 'loss': 0.10538605600595474, 'lr': 1.607479825494715e-05, 'gradient_norm': 1.2982454299926758, 'skipped': 9, 'elapsed_minutes': 215.14063473391667}
{'step': 36940, 'epoch': 7, 'loss': 0.0

Evaluate:   0%|          | 0/1273 [00:00<?, ?it/s]

{'step': 42104, 'epoch': 8, 'validation_macro_f1': 0.31279101678194526, 'validation_micro_f1': 0.44163094615786724, 'threshold': 0.1, 'best_macro_f1': 0.3146832548689996, 'early_stopping_active': True, 'bad_epochs': 1, 'epoch_train_loss': 0.0819785535844492}
{'step': 42120, 'epoch': 8, 'loss': 0.08446544408798218, 'lr': 1.4784165741122166e-05, 'gradient_norm': 1.0266722440719604, 'skipped': 11, 'elapsed_minutes': 245.33812147755}
{'step': 42140, 'epoch': 8, 'loss': 0.06514961272478104, 'lr': 1.4778979556526897e-05, 'gradient_norm': 0.894051194190979, 'skipped': 11, 'elapsed_minutes': 245.4448008458667}
{'step': 42160, 'epoch': 8, 'loss': 0.04922785609960556, 'lr': 1.4773791848066405e-05, 'gradient_norm': 0.8171512484550476, 'skipped': 11, 'elapsed_minutes': 245.55151093184998}
{'step': 42180, 'epoch': 8, 'loss': 0.055728115141391754, 'lr': 1.4768602617832616e-05, 'gradient_norm': 0.5180888772010803, 'skipped': 11, 'elapsed_minutes': 245.66566883413333}
{'step': 42200, 'epoch': 8, 'loss

In [30]:
model4_distilled_validation = validate_run('model4_distilled')


Evaluate:   0%|          | 0/1273 [00:00<?, ?it/s]

Examples: 10181
Labels: 100
Macro-F1: 0.319917
Micro-F1: 0.445790
Subset Accuracy: 0.035851

{
  "macro_f1": 0.31991700180140287,
  "micro_f1": 0.44578977524567304,
  "subset_accuracy": 0.035851095177291036,
  "macro_precision": 0.3036622010091248,
  "macro_recall": 0.36224935657456575,
  "mean_predicted_labels": 5.169924368922502,
  "mean_gold_labels": 3.9157253707887243,
  "empty_predictions": 331,
  "label_dimension": 100,
  "professor_dimension_matches": true,
  "threshold": 0.1,
  "saved_best_macro_f1": 0.31991700180140287,
  "inference_seconds": 130.3837027080001,
  "documents_per_second": 78.0849123666996,
  "parameters": {
    "total": 70378852,
    "trainable": 70378852,
    "adapted_modules": 0
  },
  "inference_memory": [
    {
      "gpu": 0,
      "peak_reserved_gib": 0.947265625,
      "peak_tensor_gib": 0.6741299629211426
    },
    {
      "gpu": 1,
      "peak_reserved_gib": 0.9765625,
      "peak_tensor_gib": 0.5643372535705566
    }
  ],
  "device": "cuda",
  "checkp

In [31]:
model4_distilled_analysis = analyze_run('model4_distilled')
model4_distilled_comparison = compare_distillation('model4_distilled')


 statute_id  train_support  dev_support  predicted_support  tp  fp  fn  precision   recall       f1 frequency_group
          8            346           92                 36   0  36  92   0.000000 0.000000 0.000000        over_100
         84            266           68                 20   1  19  67   0.050000 0.014706 0.022727        over_100
          0            341           86                 46   3  43  83   0.065217 0.034884 0.045455        over_100
          2            367           98                 17   3  14  95   0.176471 0.030612 0.052174        over_100
         99            655          176                200  10 190 166   0.050000 0.056818 0.053191        over_100
         52           1068          272                 96  11  85 261   0.114583 0.040441 0.059783        over_100
          3            449          113                 55   6  49 107   0.109091 0.053097 0.071429        over_100
         41            816          205                225  16 209 189  

In [32]:
if RUN_REPRESENTATION_ANALYSIS:
    model4_distilled_representations = representation_analysis('model4_distilled')


In [33]:
model4_distilled_submission = export_test('model4_distilled')


Test cases:   0%|          | 0/1628 [00:00<?, ?it/s]

Submission: /kaggle/working/phase5_lsi/runs/model4_distilled/test/b92c01152475/model4_distilled_submission.zip


## 13. Compare completed runs

In [34]:
comparison = []
for name in ('model3_full', 'model1_lorma', 'model4_distilled'):
    folder = run_folder(name)
    path = folder / 'validation/metrics.json'
    if not path.is_file(): continue
    report = read_json(path)
    spec = read_json(folder / 'config.json')
    progress_path = folder / 'progress.json'
    progress = read_json(progress_path) if progress_path.is_file() else {}
    comparison.append(dict(training_complete=progress.get('complete'), name=name, model=spec['model'], method=spec['method'],
        macro_f1=report['macro_f1'], micro_f1=report['micro_f1'],
        subset_accuracy=report['subset_accuracy'], threshold=report['threshold'],
        total_parameters=report['parameters']['total'], trainable_parameters=report['parameters']['trainable'],
        inference_seconds=report['inference_seconds'], checkpoint_bytes=report['checkpoint_bytes']))
comparison = pd.DataFrame(comparison)
comparison.to_csv(OUTPUT_ROOT / 'experiment_comparison.csv', index=False)
display(comparison)


,training_complete,name,model,method,macro_f1,micro_f1,subset_accuracy,threshold,total_parameters,trainable_parameters,inference_seconds,checkpoint_bytes
0,False,model3_full,model3,full,0.377098,0.494435,0.053728,0.2,118584036,118584036,291.502398,474403045
1,False,model1_lorma,model1,lorma,0.333523,0.459023,0.017975,0.1,121647844,1831268,410.862880,486699555
2,False,model4_distilled,model4,kd,0.319917,0.445790,0.035851,0.1,70378852,70378852,130.383703,281552613


## 14. Optional controls

Use a separate run name for weighted BCE, asymmetric loss, a frozen backbone, or a no-distillation control.

In [ ]:
def run_optional_control(reference_name, control_name, **changes):
    source = read_json(run_folder(reference_name) / 'config.json')
    cfg = dict(source['config'], **changes)
    method = source['method']
    if method == 'kd': method = 'control'
    register(control_name, source['model'], method, **cfg)
    capacity_check(control_name)
    report = train_run(control_name)
    validate_run(control_name)
    analyze_run(control_name)
    return report
